<a href="https://colab.research.google.com/github/rcmp-code/S05-IHM/blob/main/Diversidade_Revis%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Universidade de São Paulo

Escola de Artes, Ciências e Humanidades


# **Fomento à inovação por colaborações interdisciplinares em ciência e tecnologia: Análise de redes em projetos do programa Pesquisa Inovativa em Pequenas Empresas da FAPESP.**

Raphael Cardoso Mota Pereira.

Área de Concentração: Sistemas Complexos.

Orientadora: Profa. Dra. Flavia Mori Sarti.

São Paulo, 2024.

[DOI 10.11606/D.100.2024.tde-03042024-140601](https://doi.org/10.11606/D.100.2024.tde-03042024-140601)

<br>

# Resumo
O presente estudo constitui análise quantitativa de dados obtidos a partir de consulta à Fundação de Amparo à Pesquisa do Estado de São Paulo (FAPESP) quanto à execução de projetos aprovados no contexto do programa Pesquisa Inovativa em Pequenas Empresas (PIPE) no período de 1997 a 2022. O estudo teve como objetivo analisar a configuração e as métricas das redes de colaboração estabelecidas para empreendedorismo tecnológico do programa PIPE-FAPESP, a partir das características dos projetos aprovados complementadas por informações obtidas a partir dos currículos Lattes dos pesquisadores envolvidos nos projetos. Foram construídas redes de colaborações de pesquisadores nos projetos aprovados do programa PIPE-FAPESP segundo áreas de conhecimento, avaliando-se a evolução das colaborações em termos de diversidade de especialidades (interdisciplinaridade) e o estabelecimento de coopetição (cooperação e competição) entre empresas e Instituições de Ciência e Tecnologia (ICTs) nos projetos de inovação tecnológica aprovados no programa. Os resultados obtidos apontam para aumento da diversidade de áreas de conhecimento e da qualidade das colaborações entre pesquisadores nos projetos, crescimento do potencial de difusão do conhecimento na rede por meio de pesquisadores atuantes em múltiplos projetos de áreas de conhecimento diferentes e crescimento da interdisciplinaridade em setores de atividade estratégicos.

**Palavras-chave:** análise de redes; inovação; ciência e tecnologia; sistemas complexos; cooperação; competição.
<br><br>

---

# **Tratamento e Análise dos Dados**

Este caderno, desenvolvido em linguagem Python e fundamentado em bibliotecas especializadas na ciência de dados, visa conferir maior rigor e clareza ao tratamento, processamento e análise exploratória dos dados.

## Sumário
1. Setup e Configurações Iniciais
2. Ingestão de Dados
3. Limpeza e Padronização<br>
3.1. `df_projetos_pipe`<br>
3.2. `df_pesquisadores_projetos`<br>
3.3. `df_pesquisadores_unicos`<br>
3.4. `df_lattes`<br>
4. Engenharia de Variáveis<br>
4.1. `df_contagem_conhecimentos_pesquisadores`<br>
4.2. `df_contagem_conhecimentos_projetos`<br>
4.3. `df_diversidade_projetos`<br>
4.4. `df_coesao_projetos`<br>
4.5. `df_redes`<br>
5. Unificação da Base de Dados
6. Análise Exploratória e Visualização
7. Modelagem e Clusterização
8. Exportação dos Resultados

# 1. Setup e Configurações Iniciais
> ❗Falta incluir a instalação das bibliotecas de rede.

In [ ]:
# @title
# ==============================================================================
# 1. INSTALAÇÕES NECESSÁRIAS
# ==============================================================================

!pip install fuzzywuzzy
!pip install python-Levenshtein
!pip install scikit-bio

# ==============================================================================
# 2. IMPORTAÇÃO DE BIBLIOTECAS
# ==============================================================================

# 2.1. Ambiente e Sistema ---
from google.colab import drive

# 2.2. Manipulação de Texto e Limpeza ---
import unicodedata
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

# 2.3. Manipulação de Dados ---
import numpy as np
import pandas as pd

# 2.4. Matemática, Diversidade e Machine Learning ---
import scipy as sp
from skbio.diversity import alpha_diversity
from sklearn.cluster import KMeans

# 2.5. Visualização de Dados ---
import matplotlib.pyplot as plt
import seaborn as sns

# 2.6. Exportação e Formatação de Arquivos ---
#from openpyxl.styles import Font, PatternFill
#from openpyxl.utils import get_column_letter

# 2. Ingestão de Dados

> ❗Incluir da dissertação a origem dos dados e o link para os dados no git.


**Biblioteca Virtual FAPESP**
1. Download dos dados sobre os Projetos PIPE. \
[PROJETOS_PIPE.csv](https://drive.google.com/file/d/1BYCpyQF6v8_Lw4o2Ah1n4Oz9nya0X1Ks/view?usp=sharing)
2. Web Scrapping dos CNPJs e links dos curriculos Lattes. \
[CNAES.csv](https://drive.google.com/file/d/1BZe0BQ_EZjMsVIizMaZUe66uGAnLrUGu/view?usp=sharing) /
[LATTES_1.csv](https://drive.google.com/file/d/1BZcx7YSZzZvMpsgebL3jDKvcZICRDTZY/view?usp=sharing) /
[LATTES_2.csv](https://drive.google.com/file/d/1BYxkGPSLm6Zy4mNXBUF7xRO6ceGeqTKC/view?usp=sharing) \
3. Solicitação de dados ao Sistema de Informação ao Cidadão (SIC). \
[AUXILIOS_CONCEDIDOS_COM_NIVEL.xlsx](https://docs.google.com/spreadsheets/d/1nGM8IqyeobdpWIedlODzhd-XuWfDeskY/edit?usp=sharing&ouid=101274464048368031766&rtpof=true&sd=true) /
[AUXILIOS_CONCEDIDOS.xlsx](https://docs.google.com/spreadsheets/d/1nGnaBtsaWlz53R64-6IDVX6ztbbYI-Rt/edit?usp=sharing&ouid=101274464048368031766&rtpof=true&sd=true) /
[BOLSAS_CONCEDIDAS.xlsx](https://docs.google.com/spreadsheets/d/1nQ4NOgPZh8vnc98ukyd_U5h2rSsC-f00/edit?usp=sharing&ouid=101274464048368031766&rtpof=true&sd=true) /
[PIPE_DENEGADOS.xlsx](https://docs.google.com/spreadsheets/d/1nZCvdTLubtMBHzV4z3dyuXiql0XYMg09/edit?usp=sharing&ouid=101274464048368031766&rtpof=true&sd=true)

**Plataforma Lattes CNPq**
1. Download dos curriculos em XML. \
[CURRICULOS_XML](https://drive.google.com/drive/folders/1o-338s9HYSpTekQTCA4uID8XIL64XYix?usp=sharing)
2. Extração dos dados do XML para um CSV. \
[XSD Lattes](https://rcmp-collab.github.io/lattes/) /
[DADOS_LATTES_PESQUISADORES.csv](https://drive.google.com/file/d/1B_k8JshmZhBK0p9jgGjlQYd6ZvXEmhSI/view?usp=sharing)

In [ ]:
# @title
# ==============================================================================
# 1. MONTAR GOOGLE DRIVE
# ==============================================================================

drive.mount('/content/drive')
spreadsheet_id = '1TZXDGB9VMB4OUVawJSrtgRfnNXOZcbkH93R3QC_7cCs'
projetos_pipe_gid = '1244809562'
bolsas_concedidas_gid = '2055929609'
conhecimentos_cnpq_gid = '1760819603'
lattes_gid = '1187992543'

# ==============================================================================
# 2. CARREGAR TABELAS
# ==============================================================================

# Construir URLs para exportar as abas como CSV
url_projetos_pipe = f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?format=csv&gid={projetos_pipe_gid}'
url_bolsas_concedidas = f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?format=csv&gid={bolsas_concedidas_gid}'
url_conhecimentos_cnpq = f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?format=csv&gid={conhecimentos_cnpq_gid}'
url_lattes = f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?format=csv&gid={lattes_gid}'

# Carregar 'PROJETOS PIPE'
df_projetos_pipe = pd.read_csv(url_projetos_pipe)
print(f"Tabela 'df_projetos_pipe' - Total de linhas: {len(df_projetos_pipe)}")
print(df_projetos_pipe.dtypes)
print("-" * 60)
#print(df_projetos_pipe.head())

# Carregar 'BOLSAS CONCEDIDAS'
df_bolsas_concedidas = pd.read_csv(url_bolsas_concedidas)
print(f"Tabela 'df_bolsas_concedidas' - Total de linhas: {len(df_bolsas_concedidas)}")
print(df_bolsas_concedidas.dtypes)
print("-" * 60)
#print(df_bolsas_concedidas.head())

# Carregar 'CONHECIMENTOS CNPq'
df_conhecimentos_cnpq = pd.read_csv(url_conhecimentos_cnpq)
print(f"Tabela 'df_conhecimentos_cnpq' - Total de linhas: {len(df_conhecimentos_cnpq)}")
print(df_conhecimentos_cnpq.dtypes)
print("-" * 60)
#print(df_lattes.head())

# Carregar 'CURRICULOS LATTES'
df_lattes = pd.read_csv(url_lattes)
print(f"Tabela 'df_lattes' - Total de linhas: {len(df_lattes)}")
print(df_lattes.dtypes)
print("-" * 60)
#print(df_lattes.head())



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tabela 'df_projetos_pipe' - Total de linhas: 4689
N. Processo                                          object
Título (Português)                                   object
Título (Inglês)                                      object
Beneficiário                                         object
Instituição                                         float64
Cidade Instituição                                  float64
Instituição Parceira                                 object
Empresa                                              object
Município                                            object
Pesquisador Responsável                              object
Pesquisadores Principais                             object
Pesquisadores Associados                             object
Supervisor                                          float64
Local de Pesquisa                        

## 2.1. Web Scraping

Alguns dados apresentados no HTML não são incluidos no arquivo CSV, e para complementar os dados dos projetos foi realizado um *web scraping* no HTML das páginas de resultado da BV FAPESP, com o objetivo de coletar os CNAEs das empresas e os links dos curriculos lattes de todos os pesquisadores alocados nos projetos. Para isso, foi utilizado a biblioteca [Scrapy](https://scrapy.org/), e a seguir são apresentados os três 'spiders' utilizadas no framework:


### 2.1.1. Extração dos CNPJs


In [ ]:
# @title
# # @title
# import scrapy

# class cnaeSpider(scrapy.Spider):
#     name = 'cnae'
#     start_urls = [
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=1&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=2&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=3&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=4&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=5&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=6&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=7&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=8&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=9&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=10&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=11&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=12&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=13&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=14&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=15&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=16&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=17&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=18&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=19&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=20&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=21&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=22&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=23&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=24&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=25&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=26&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=27&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=28&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=29&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=39&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=31&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=32&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=33&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=34&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=35&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=36&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=37&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=38&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=39&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=40&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=41&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=42&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=43&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=44&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=45&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=46&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=47&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=48&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=49&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=50&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=51&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=52&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=53&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=54&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=55&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=56&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=57&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=58&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=59&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=60&count=50'
#     ]

# # Código para o PROCESSO COM CNAEs das empresas:

#     def parse(self, response):
#         for projeto in response.xpath('*//div[@class="table_details"]'):
#             yield{
#                 'processo': projeto.xpath('*//td[contains(text(), "Processo")]/following-sibling::td/text()').get(),
#                 'cnae 1': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[1]').get(),
#                 'cnae 2': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[2]').get(),
#                 'cnae 3': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[3]').get(),
#                 'cnae 4': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[4]').get(),
#                 'cnae 5': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[5]').get(),
#                 'cnae 6': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[6]').get(),
#                 'cnae 7': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[7]').get(),
#                 'cnae 8': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[8]').get(),
#                 'cnae 9': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[9]').get(),
#                 'cnae 10': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[10]').get(),
#                 'cnae 11': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[11]').get(),
#                 'cnae 12': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[12]').get(),
#                 'cnae 13': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[13]').get(),
#                 'cnae 14': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[14]').get(),
#                 'cnae 15': projeto.xpath('*//td[contains(., "CNAE")]/following-sibling::td/text()[15]').get()
#             }

### 2.1.2. Coleta dos links de curriculos Lattes I

Este levantamento não incluiu o link dos Pesquisadores em Treinamento Técnico. \
[LATTES_1.csv](https://drive.google.com/file/d/1BZcx7YSZzZvMpsgebL3jDKvcZICRDTZY/view?usp=sharing)

In [ ]:
# @title
# # @title
# import scrapy


# class lattesSpider(scrapy.Spider):
#     name = 'lattes'
#     start_urls = [
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=1&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=2&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=3&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=4&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=5&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=6&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=7&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=8&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=9&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=10&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=11&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=12&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=13&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=14&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=15&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=16&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=17&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=18&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=19&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=20&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=21&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=22&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=23&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=24&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=25&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=26&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=27&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=28&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=29&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=39&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=31&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=32&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=33&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=34&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=35&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=36&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=37&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=38&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=39&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=40&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=41&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=42&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=43&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=44&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=45&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=46&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=47&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=48&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=49&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=50&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=51&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=52&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=53&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=54&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=55&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=56&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=57&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=58&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=59&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Conclu%C3%ADdos%22)&page=60&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Em%20andamento%22)&page=1&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Em%20andamento%22)&page=2&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Em%20andamento%22)&page=3&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Em%20andamento%22)&page=4&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Em%20andamento%22)&page=5&count=50',
#         'https://bv.fapesp.br/pt/pesquisa/buscador/?q2=(PIPE)%20AND%20(auxilio%3A*%20AND%20situacao%3A%22Em%20andamento%22)&page=6&count=50'
#     ]

# # Código para o link do lattes dos pesquisadores:

#     def parse(self, response):
#         for projeto in response.xpath('*//div[@class="table_details"]'):
#             yield{
#                 'processo': projeto.xpath('*//td[contains(text(), "Processo")]/following-sibling::td/text()').get(),
#                 'PR_NOME': projeto.xpath('*//td[contains(., "Pesquisador responsável")]/following-sibling::td/a[1]/text()').get(),
#                 'PR': projeto.xpath('*//td[contains(., "Pesquisador responsável")]/following-sibling::td/a[2]/@onclick').get(),
#                 'PP 1_NOME': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[1]/a/text()').get(),
#                 'PP 1': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[1]/span/a/@onclick').get(),
#                 'PP 2_NOME': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[2]/a/text()').get(),
#                 'PP 2': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[2]/span/a/@onclick').get(),
#                 'PP 3_NOME': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[3]/a/text()').get(),
#                 'PP 3': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[3]/span/a/@onclick').get(),
#                 'PP 4_NOME': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[4]/a/text()').get(),
#                 'PP 4': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[4]/span/a/@onclick').get(),
#                 'PP 5_NOME': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[5]/a/text()').get(),
#                 'PP 5': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[5]/span/a/@onclick').get(),
#                 'PP 6_NOME': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[6]/a/text()').get(),
#                 'PP 6': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[6]/span/a/@onclick').get(),
#                 'PP 7_NOME': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[7]/a/text()').get(),
#                 'PP 7': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[7]/span/a/@onclick').get(),
#                 'PP 8_NOME': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[8]/a/text()').get(),
#                 'PP 8': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[8]/span/a/@onclick').get(),
#                 'PP 9_NOME': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[9]/a/text()').get(),
#                 'PP 9': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[9]/span/a/@onclick').get(),
#                 'PP 10_NOME': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[10]/a/text()').get(),
#                 'PP 10': projeto.xpath('*//td[contains(., "Pesquisadores principais")]/following-sibling::td/h2/span[10]/span/a/@onclick').get(),
#                 'PA 1_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[1]/a/text()').get(),
#                 'PA 1': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[1]/span/a/@onclick').get(),
#                 'PA 2_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[2]/a/text()').get(),
#                 'PA 2': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[2]/span/a/@onclick').get(),
#                 'PA 3_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[3]/a/text()').get(),
#                 'PA 3': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[3]/span/a/@onclick').get(),
#                 'PA 4_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[4]/a/text()').get(),
#                 'PA 4': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[4]/span/a/@onclick').get(),
#                 'PA 5_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[5]/a/text()').get(),
#                 'PA 5': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[5]/span/a/@onclick').get(),
#                 'PA 6_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[6]/a/text()').get(),
#                 'PA 6': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[6]/span/a/@onclick').get(),
#                 'PA 7_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[7]/a/text()').get(),
#                 'PA 7': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[7]/span/a/@onclick').get(),
#                 'PA 8_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[8]/a/text()').get(),
#                 'PA 8': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[8]/span/a/@onclick').get(),
#                 'PA 9_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[9]/a/text()').get(),
#                 'PA 9': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[9]/span/a/@onclick').get(),
#                 'PA 10_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[10]/a/text()').get(),
#                 'PA 10': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[10]/span/a/@onclick').get(),
#                 'PA 11_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[11]/a/text()').get(),
#                 'PA 11': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[11]/span/a/@onclick').get(),
#                 'PA 12_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[12]/a/text()').get(),
#                 'PA 12': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[12]/span/a/@onclick').get(),
#                 'PA 13_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[13]/a/text()').get(),
#                 'PA 13': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[13]/span/a/@onclick').get(),
#                 'PA 14_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[14]/a/text()').get(),
#                 'PA 14': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[14]/span/a/@onclick').get(),
#                 'PA 15_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[15]/a/text()').get(),
#                 'PA 15': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[15]/span/a/@onclick').get(),
#                 'PA 16_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[16]/a/text()').get(),
#                 'PA 16': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[16]/span/a/@onclick').get(),
#                 'PA 17_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[17]/a/text()').get(),
#                 'PA 17': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[17]/span/a/@onclick').get(),
#                 'PA 18_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[18]/a/text()').get(),
#                 'PA 18': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[18]/span/a/@onclick').get(),
#                 'PA 19_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[19]/a/text()').get(),
#                 'PA 19': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[19]/span/a/@onclick').get(),
#                 'PA 20_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[20]/a/text()').get(),
#                 'PA 20': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[20]/span/a/@onclick').get(),
#                 'PA 21_NOME': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[21]/a/text()').get(),
#                 'PA 21': projeto.xpath('*//td[contains(., "Pesq. associados")]/following-sibling::td/h2/span[21]/span/a/@onclick').get()
#             }


### 2.1.3. Coleta dos links de curriculo Lattes II
Esse levantamento foi baseado na base de dados Bolsas Concedidas obtida através do SIC. Nesse levantamento todos os Pesquisadores em Treinamento Técnico estão inclusos.


In [ ]:
# @title
# # @title
# import scrapy

# class pesqSpider(scrapy.Spider):
#     name = 'pesq_2'
#     start_urls = [
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADELITA%20CAROLINA%20SANTIAGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADEMIR%20AZEVEDO%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANA%20BARRINHA%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANA%20PAVINATTO%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANO%20CASIMIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AGUINALDO%20PEREIRA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AILTON%20DE%20ASSIS%20QUEIROGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AILTON%20YOSHINORI%20TANAKA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALBERTO%20JOSE%20SCHMIELIAUSKAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALBERTO%20O%20FARRILL%20VANNINI%20PESSINA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEKSANDRA%20ALVES%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRA%20GALLO%20PETRAROLI%20TATEYAMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRA%20VENANCIO%20DINIZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEX%20CARVALHO%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20CAMILO%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20CAPTIAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20MINAMI%20FIOROTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20MORON%20BERNARDONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20NUNES%20DA%20TRINDADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20PICCHI%20NEVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALFREDO%20MARCIAL%20MONTES%20NINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALFREDO%20YAHN%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20BERNARDES%20MUNIZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20GOMES%20MARCELINO%20PEREZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CAROLINA%20CAMPOS%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CLAUDIA%20OLIVEIRA%20CARREIRA%20NISHIYAMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20MARISA%20CHUDZINSKI-TAVASSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20FELIPE%20PIRES%20SONNENBURG',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20GUSTAVO%20CAVALCANTI%20DE%20MELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20DE%20ABREU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20GONCALVES%20LOUREIRO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20ROCHA%20D.%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20MENEZES%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREA%20ARRUDA%20MARTINS%20SHIMOJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREA%20STÖCKL%20GEROSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANGELICA%20FERNANDEZ%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANNA%20LUCIA%20CASAÑAS%20HAASIS%20VILLAVICENCIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20ADEMIR%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20CARLOS%20OLIVEIRA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20CLAUDIO%20REIS%20DE%20PAIVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20ROBERTO%20DE%20GODOI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20SERGIO%20ASSUNCAO%20TAVARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARCHIAS%20ALVES%20DE%20ALMEIDA%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARMANDO%20ANTONIO%20MARIA%20LAGANA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTHUR%20PRUDENCIO%20DE%20ARAUJO%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTHUR%20ROZA%20AUGUSTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARY%20BIAZOTTO%20CORTE%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AURELINDO%20LEME%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AYRTON%20SALVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BENEDITO%20CARLOS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BENTO%20DA%20COSTA%20CARVALHO%20JR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BORIS%20ROTTER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRETT%20MYLO%20DRURY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNA%20THAISE%20RODRIGUES%20RHEIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20JENSEN%20VIRGINIO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20LUIZ%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20SQUIZATO%20FAICAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20ALBERTO%20FERRAGINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20ANTONIO%20TAUBE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20EDUARDO%20FONTENELLE%20CARNEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20EDUARDO%20JUNQUEIRA%20FONSECA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20LEONARDO%20HERRERA%20MUNOZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINA%20COLOMBELLI%20PACCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINA%20SIEQUEROLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINE%20CRISTIANO%20REAL%20GREGORIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLYNE%20BITENCOURT%20FARIA%20TORRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAURE%20BARBOSA%20PORTUGAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CELSO%20BARBIERI%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CEM%20MUSA%20ALBUKREK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CESAR%20AUGUSTO%20DUARTE%20RODRIGUEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHIU%20CHIH%20MING',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTIANE%20BACCI%20REGHINE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTIANE%20DE%20ARRUDA%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTOPH%20TREUDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIA%20FILONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDINEI%20ANTONIO%20SALATA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIO%20DE%20LIMA%20MIGUEL%20MARTINEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIO%20VINICIUS%20BUONAMICI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEBER%20JOSE%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEMENCIA%20NORIEGA%20SÖNDAHL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=COSIMO%20GUARINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CYNTHIA%20CRISTINA%20MARTINS%20JUNQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DALTON%20YOSHIMI%20KINA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20GIL%20MONTEIRO%20DE%20FARIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20MOUTIN%20SEGORIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20ROBERTO%20CALLEJON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIELLE%20BRUNA%20LEAL%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIELLE%20GOBBI%20BRANTS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIELLE%20GREGORIO%20GOMES%20CALDAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANILO%20RODRIGUES%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVI%20FERREIRA%20DE%20CASTRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DEBORA%20MARCONDES%20BASTOS%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENILSON%20SHIKAKO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENISE%20CRISPIM%20TAVARES%20BARBOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENISE%20TEREZINHA%20BARNABE%20ABACKERLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20SILVA%20SIQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20VAZ%20PONTES%20CAMBRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIETER%20LUBECK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DILSON%20SADANOBU%20TSURUMAKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIOGO%20BRANQUINHO%20RAMOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIOGO%20GONCALVES%20BIAGI%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DURVAL%20MARCOS%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDELCIO%20LEME%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDJAR%20MARTINS%20TELLES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDMAR%20JOSE%20KIEHL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDSON%20BORGES%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDSON%20RODRIGUES%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20CANNIZZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20HENRIQUE%20TOZETTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20VETTORI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELAINE%20CRISTINA%20DE%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELEONORA%20SELIGMANN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELI%20SIDNEY%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELIRIA%20MARIA%20DE%20JESUS%20AGNOLON%20PALLONE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELMER%20ALBERTO%20CCOPA%20RIVERA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ENNER%20HERENIO%20DE%20ALCANTARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERICA%20ENGELBERG%20COOK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERNA%20ELISABETH%20BACH',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EURIPEDES%20GUILHERME%20DE%20OLIVEIRA%20NOBREGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVERTON%20SERGIO%20ESTRACANHOLLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANA%20HENRIQUES%20MACHADO%20DE%20MELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANO%20SIMAO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20FERNANDO%20ALVES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20POMES%20SALLES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABRICIO%20PUENTE%20MANSILLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20AUGUSTO%20FIGUEIREDO%20FRAGOSO%20PIRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20MARCELO%20PEREIRA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20DA%20CRUZ%20LANDIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20MACITELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20SAYURI%20YOSHINO%20WATANABE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20TOLEDO%20BASTOS%20GUANDALINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20DE%20QUEIROZ%20CUNHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20FERNANDES%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FILIPE%20FIGUEREDO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FILIPPO%20GHIGLIENO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIA%20MAUAD%20LEVY%20ABRAHAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIA%20PINHEIRO%20ZANOTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20FARIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20GONCALVES%20BOSKOVITZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20SALSONI%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20CLEBER%20SOUSA%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20MANUEL%20BARRALES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20DE%20OLIVEIRA%20AMANCIO%20BALAZINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GEISON%20VOGA%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GERALDO%20PEDROSO%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GILBERTO%20PETRACONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GILBERTO%20RIGOBELLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GILBERTO%20VICENTE%20CONCILIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANNI%20AUGUSTO%20DE%20SANTANA%20SCHIMIDT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLAUBER%20EDURADO%20SAPIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLAUCO%20JOSE%20RIZZANTI%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GRACE%20MENDONCA%20DIAS%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20AUGUSTO%20DUARTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20DE%20ALMEIDA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20DE%20CASTRO%20HISSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20HOLLOWAY%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20JEAN%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20PERON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HABIB%20GUY%20MARIE%20NAHAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HAROLDO%20THOMAZ%20KERRY%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELIARA%20DALVA%20LOPES%20DO%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELIO%20AVELINO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELIO%20KOITI%20KUGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELIO%20SHIGUEKI%20OZAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELTON%20RICARDO%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HERNAN%20CORTES%20GOMEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HERNANDE%20CARLOS%20BUENO%20PREVIATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HILDA%20ALICIA%20GOMEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HOMERO%20FERRACINI%20GUMERATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HUGO%20VASCONCELOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HUMBERTO%20PONTES%20CARDOSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IL%20YOUNG%20AHN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ILKA%20TIEMY%20KATO%20PRATES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=INGRID%20MÜLLER%20LEDRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IRON%20CALIL%20DAHER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABEL%20MENEZES%20DE%20BULHOES%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ITALO%20SALUSSOLIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IVAN%20DA%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IVAN%20DE%20GODOY%20MAIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JAKSAN%20MOREIRA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JAYME%20ANTONIO%20ABOIN%20SERTIE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEAN%20CLAUDE%20LAMARCHE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEAN%20PAULO%20AGOSTINHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEFFERSON%20GARCIA%20FRANCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JESSICA%20ANDRADE%20VILAS%20BOAS%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JESUS%20RAINDO%20GOMEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20ANTONIO%20MATTEI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20BATISTA%20ALMEIDA%20MEDEIROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20BATISTA%20SGORBISSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20BRAGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20CONTART%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20FELIPE%20MANFRINATO%20MARIANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20FELIPE%20MISEROCHI%20DE%20OLIVEIRA%20LINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20TAKASHI%20OHASHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOEL%20AIRES%20PRATES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOHANNA%20BAJONERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOHNSON%20DELIBERO%20ANGELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JONI%20JULIANO%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JORGE%20EDUARDO%20VETTORAZZO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JORGE%20ENRIQUE%20RODRIGUEZ%20CHANFRAU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JORGE%20HIDEMI%20OHASHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ALBERTO%20QUINTANILHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ALVARO%20OGANDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ANTONIO%20NEVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ARMANDO%20FURLANI%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20CARLOS%20DE%20JESUS%20BERTACINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20CARLOS%20FARIAS%20ALVES%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20CARLOS%20PENA%20BARBOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20CARLOS%20VERTEMATTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20DIAS%20CARVALHO%20MELLO%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20LEONI%20TREMESCHIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20MAIA%20DANTAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20RICARDO%20PORTILLO%20NAVAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ROBERTO%20MELO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20SIDNEI%20COLOMBO%20MARTINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20SOARES%20GUERRERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANO%20BUZZINI%20PULICCI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIE%20GIOVANNA%20CHACON%20OROZCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIETA%20ADRIANA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIO%20AUGUSTO%20LEITAO%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIO%20CESAR%20BASTOS%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIO%20EZEQUIEL%20PALACIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KAMILLA%20SWIECH%20ANTONIETTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KAORU%20BABA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KAREN%20ELIZABETH%20ADARME%20GALVAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KAREN%20JULIE%20SANTOS%20GRANCIANINOV%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KARINA%20BARRETO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KARISA%20KARLA%20MANHANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KARLA%20CRISTIANE%20BASTOS%20MELLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KEIJI%20ROBERTO%20NAKASHIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KEILA%20KEIKO%20MATSUMURA%20KAYATT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KLAUS%20GARGITTER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAURA%20STERIAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAURO%20BENASSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20AGOSTINI%20DO%20AMARAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20FARIAS%20NOGUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20VICTOR%20FIDELIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEILA%20KEIKO%20CANEGUSUCO%20JANSEN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20FRANCO%20DE%20GODOI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LIBORIO%20JOSE%20FARIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LINCOLN%20MIZIARA%20BARBOZA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUANA%20MARTINS%20DE%20ANDRADE%20DA%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20FELIX%20LIMA%20BARBOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIA%20HELENA%20SASSERON%20ROBERTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANA%20GODINHO%20DEGRECCI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20AUGUSTO%20LUPATO%20CONRADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20FERNANDO%20DE%20OLIVEIRA%20FURTADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20ARTHUR%20CURY%20E%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20CARLOS%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20FERNANDO%20ROMANHOLO%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20GERALDO%20MIALHE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20MANOEL%20DIAS%20HENRIQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20PAULO%20LAVOIE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAMEDE%20AUGUSTO%20MACHADO%20DA%20SILVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20ROMANO%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20SIMIONI%20PONTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20TADEU%20BERTANHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIA%20MARIZA%20GOMES%20JUSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIO%20HENRIQUE%20NIGRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIO%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCO%20ANTONIO%20DE%20OLIVEIRA%20ALVES%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCO%20ANTONIO%20STEPHANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCO%20AURELIO%20GEROSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCO-AURELIO%20DE%20PAOLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20DE%20CASTRO%20REINACH',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20MAKOTO%20IKEGAME',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20VINICIUS%20RAYOL%20SOBREIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20ANGELICA%20DE%20CAMARGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20BEATRIZ%20CALDERAN%20RODRIGUES%20BONASSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20CAROLINA%20QUECINE%20VERDI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20DA%20GRACA%20CAMPOS%20PIMENTEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20DE%20LURDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20ISABEL%20BERTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20LUISA%20LOPES%20DE%20FARIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20RENATA%20VALENTE%20BRANDAO%20FREIRE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20HELENA%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARINO%20ARPINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIO%20ALEXANDRE%20GAZZIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIO%20LOBO%20DE%20SOUZA%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARLENE%20GUIDI%20BRAGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20CASTRO%20CARDOSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURICIO%20RIBEIRO%20HIRDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURICIO%20RUMENOS%20GUIDETTI%20ZAGATTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURO%20HIRDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAYLA%20WILLIK%20VALENTI%20ROESE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MICHEL%20LEVY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MIGUEL%20LUIS%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MILTON%20FELICISSIMO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MILTON%20HIROKAZU%20SHIMABUKURO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MIQUEAS%20DE%20OLIVEIRA%20BRAGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MOACYR%20MARTUCCI%20JR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MURILO%20VIEIRA%20GERALDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NAIANE%20SANGALETTI%20GERHARD',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NARIANE%20MARSELHE%20RIBEIRO%20BERNARDO%20DO%20CARMO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20MARCHESAN%20BEXIGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20VALIAS%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NEDINALVA%20DE%20ARAUJO%20SELLIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NELSON%20AKIRA%20ASSATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NELSON%20ANTONIO%20FELIX%20BEIRAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NILCE%20ORTIZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NILTON%20DIAS%20BORREGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NILTON%20NOBUHIRO%20IMAI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NILTON%20PEREIRA%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NINIVE%20AGUIAR%20COLONELLO%20FRATTINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NUNCIO%20LOBELLO%20CARDINALI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ODILON%20AFONSO%20VIEIRA%20CENAMO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSCAR%20EDUARDO%20SOLARTE%20MONTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSCAR%20NISHIMURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSEAS%20VALENTE%20DE%20AVILEZ%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSNI%20FLAVIO%20PASSOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSWALDO%20ROSSI%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSWALDO%20URBANI%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAOLA%20MARCHI%20CABRAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20AUGUSTO%20DE%20TOLEDO%20PACHECO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20CARLOS%20GALIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20CESAR%20CERAGIOLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20CESAR%20DO%20NASCIMENTO%20PAVAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20DE%20TARSO%20GAETA%20PAIXAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20EDUARDO%20PASCON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20HENRIQUE%20CHAVES%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20ROBERTO%20CORTEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20SERGIO%20FABRIS%20DE%20MATOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20GUSTAVO%20CORDOBA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20LUIZ%20ROSALEN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20PAULO%20AUGUSTO%20FABIANO%20ARANTES%20PEREIRA%20BARRETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20RICARDO%20DRUMMOND',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20SIENA%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PERICLES%20ASSAD%20HASSUN%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILA%20PEREIRA%20FAVERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20ALVES%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20AUGUSTUS%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20GIULIANO%20PILEGGI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20NETTO%20MOREIRA%20GARCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20PILLON%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20TADEU%20BENEDITO%20DE%20OLIVEIRA%20ROMAN%20LUQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAIMUNDO%20JOSE%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAQUEL%20VALERIO%20DE%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=REGINA%20BERNARDINA%20JOHANNA%20HAKVOORT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=REJANE%20MARIA%20TOMMASINI%20GROTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENAN%20BARROS%20DOMINGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENAN%20FERRARI%20BANGOIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20DE%20SOUZA%20LEAO%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20MORELLI%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20RIBEIRO%20DO%20VAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATO%20JOSE%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATO%20MORANDIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20GERETTO%20KORTAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20GOMIDE%20WOISKY%20DO%20RIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20GUIMARAES%20MORRONE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20HARAKAVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20LEVRINI%20DE%20TOLEDO%20DIAS%20BAPTISTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20LUIS%20DUARTE%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20MURVILLE%20CAMPS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20REIS%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RITA%20DE%20CASSIA%20MENDONCA%20SALES%20CONTINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RITA%20DE%20CASSIA%20SAVIO%20FIGUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RITA%20MARIA%20BORGES%20DE%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTA%20AVERNA%20VALENTE%20B%20TOLINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTA%20CRISTINA%20RUEDAS%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20TOMASI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20ALVARENGA%20REZENDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20BASILE%20JUNQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20FRANCO%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROGERIO%20MARCIO%20RYKOVSKY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROSANGELA%20RODRIGUES%20LEME%20PELLEGRINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RUI%20BARBOSA%20DE%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMIR%20AUED',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20DUARTE%20PENNA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20HENRIQUE%20GARBE%20ORESTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SHAILA%20FABI%20MOREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SIDINEY%20PERUCHI%20DE%20GODOY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILVIA%20AMANDA%20MELO%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILVIA%20CRISTINA%20NUÑEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILVIA%20MAYUMI%20TAKEY%20DELLAMANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILVIO%20ANTONIO%20TONISSI%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILVIO%20CASSIO%20BISPO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SOELLY%20MAGALHAES%20DO%20VALLE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SORAYA%20EL%20KHATIB',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=STEFANO%20GUIMARAES%20VELLUDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TACITO%20MISTRORIGO%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TAKAHIRO%20OKITA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TAKAKO%20MATSUMURA-TUNDISI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TEREZA%20CRISTINA%20PEIXOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAYLA%20MORANDI%20RIDOLFI%20DE%20CARVALHO%20CURI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAYSE%20TIERI%20NONAKA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20BRUSCHI%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20GARA%20CAETANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TOMAS%20RICARDO%20CAMPOS%20PIMENTEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TORU%20MIYAGI%20KINJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VANESSA%20BARBOSA%20MALAQUIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VANIA%20REGINA%20NICOLETTI%20TELIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VILMAR%20RODRIGUES%20DE%20SOUSA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WAGNER%20PALMIERI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WALMIR%20GOMES%20LOURENCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILIAM%20DONISETE%20DE%20PAULA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAM%20CARNICELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAM%20JOSE%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILSON%20DE%20CARVALHO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WOLNEI%20GARRIDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ABNER%20DA%20SILVA%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ABNER%20EMANUEL%20DOS%20SANTOS%20CORREIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADELCI%20LOPES%20DE%20FRANCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADELSON%20DUARTE%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADILSON%20BERVEGLIERI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADIVANIA%20DE%20SOUZA%20BORGES%20NICOLETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIA%20MENEZES%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIAN%20COLANTONIO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIAN%20SOTERO%20DE%20WITT%20BATISTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANA%20CACERES%20BRAGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANA%20CRISTINA%20MOTTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANA%20MARQUES%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANA%20PINHEIRO%20DA%20FRANCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANA%20ROSOLIA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANA%20TALHARI%20MENDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANA%20VIABONI%20LONGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANO%20ANDRULIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANO%20CAMARGO%20RODRIGUES%20DA%20CUNHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANO%20CAPATTI%20CASSIANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANO%20DA%20SILVA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANO%20DA%20SILVA%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANO%20GOMES%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANO%20GUILGER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANO%20LUIS%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIANO%20MENDONCA%20MASSON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ADRIELE%20BEATRIZ%20KUCINSKAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AFONSO%20FELIPE%20BORGONOVI%20CHRISTIANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AGNALDO%20DIOGO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AIRTON%20ALMEIDA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AIRTON%20PASIANOT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AISSA%20HADJ%20MOHAMED',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALAM%20CLEBER%20FERREIRA%20CROCO%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALAN%20DEL%20ARCO%20PASCHOAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALAN%20GIOVANINI%20DE%20OLIVEIRA%20SARTORI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALAN%20LUCIAN%20MILANES%20TORMENTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALBERTO%20BARBOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALBERTO%20CACERES%20ALVAREZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALBERTO%20DE%20ANDRADE%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALBERTO%20KYOSHI%20IMAMURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALBERTO%20ROQUE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALBERTO%20STEINBERG',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALCIDES%20MIGNOSO%20E%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALCINO%20HIROYUKI%20FUJII%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALECIO%20JULIO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEJANDRA%20JIMENA%20INGA%20QUEZADA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSA%20BAPTISTA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRA%20APARECIDA%20TOYAMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRA%20ARAUJO%20TAVARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRA%20DOS%20SANTOS%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRA%20GOMES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRA%20LIA%20GASPARETTI%20GUARILHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRA%20STRANGUETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRA%20XAVIER%20DE%20PADUA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRO%20AUGUSTO%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRO%20OLIVEIRA%20DE%20MORAES%20NOGUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRO%20OLIVEIRA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRO%20PAULO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALESSANDRO%20RODOLPHO%20GONCALVES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEX%20APARECIDO%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEX%20CRISTIANO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEX%20DA%20SILVA%20NORONHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEX%20DEL%20ARCO%20PASCHOAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEX%20FELIX%20DOS%20SANTOS%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEX%20FERNANDO%20DE%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEX%20LOPES%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEX%20RODOLFO%20GALVAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEX%20SANDRO%20AGUIAR%20PESSOA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEX%20TOSHIO%20KAKIZAKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDER%20SIMOES%20DEKKER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20ANTONIO%20DE%20AFFINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20ANTUNES%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20BOA%20VENTURA%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20CANDIDO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20CANDIDO%20MOREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20CARBONE%20NOVAES%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20CARVALHO%20DIAS%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20DE%20ALMEIDA%20NAHAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20DE%20SA%20MACEDO%20FERREIRA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20DO%20PRADO%20MATIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20DONISETE%20BENSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20GARCIA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20OLIVEIRA%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20PANOSSO%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20PASSOS%20FREITAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20RICARDO%20DOS%20SANTOS%20FORNARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20SILVEIRA%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20SURKUS%20FORNI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20TAZONIERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXANDRE%20VASCONCELLOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXEY%20GORKI%20BIGASZ%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXIS%20MATHEUS%20LOPES%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXSANDER%20MATZNER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALEXSANDRO%20ALVES%20LUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALFONSO%20AREIZA%20GUERRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALFREDO%20COLENCI%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALICE%20MARIA%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALICE%20YUKI%20SHINTANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALINE%20ARAUJO%20POLITANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALINE%20BEZERRA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALINE%20CHITERO%20BUENO%20FAGLIARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALINE%20COLARES%20DO%20VALE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALINE%20CRISTINA%20DIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALINE%20FERNANDA%20DEJANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALINE%20MONTEIRO%20PIRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALINE%20NEVES%20EUGENIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALINE%20OLIVAS%20KAJI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALINE%20RISSON%20BELINOVSKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALINI%20SUILI%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALISSON%20ANTONIO%20GALLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALISSON%20FABIANO%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALISSON%20TURINI%20FIORINI%20BOLSONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALLAN%20AMARAL%20TORI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALLAN%20BERECZKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALLAN%20DOUGLAS%20DOS%20SANTOS%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALLAN%20DOUGLAS%20RODRIGUES%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALLAN%20MARIANO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALLAN%20NOZOMU%20FUKASAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALLAN%20PATRICK%20CORDEIRO%20DIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALLBERSON%20BRUNO%20DE%20OLIVEIRA%20DANTAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALLISON%20FAUAT%20SCHRAIER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALVARO%20JHOVALDO%20LOPEZ%20AYME',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALVARO%20LUIZ%20MERICI%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALVARO%20RODOLFO%20ALKSCHBIRS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALWEYD%20TESSER%20DE%20MORAIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ALYSSON%20CAMPOS%20MELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMADOR%20POCEIRO%20ORELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20DE%20ALMEIDA%20SALES%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20MARIA%20CLARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20MIGUEL%20COUTINHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20NEUMANN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20ROCHA%20CHAVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20RODRIGUES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20SALLES%20MINELI%20FLORINDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20SETTI%20RAIZE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20TAVARES%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20TEIXEIRA%20BADARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMANDA%20ZANCHET%20FERNANDEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMAURI%20FERREIRA%20DO%20PATROCINIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AMAURI%20FOGANHOLI%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20BEATRIZ%20CHERNICHENCO%20DE%20OLIVEIRA%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20BEATRIZ%20DELFORNO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20BEATRIZ%20PRAIA%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CAROLINA%20AGARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CAROLINA%20BARBOSA%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CAROLINA%20BORTOLOSSI%20REZENDE%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CAROLINA%20CHIOZI%20ZANETTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CAROLINA%20DALL%20ANTONIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CAROLINA%20DE%20ALMEIDA%20PINTO%20SCHWARZER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CAROLINA%20TAN%20MOREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CAROLINE%20ARAMAKI%20HITOMI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CAROLINE%20FIRMIANO%20DE%20JESUS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CLARA%20CUCO%20GIOCONDO%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20CLAUDIA%20DUARTE%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20ELISA%20DE%20OLIVEIRA%20E%20LONGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20ELISA%20FERNANDES%20TOBAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20FLAVIA%20DE%20BRITO%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20FLAVIA%20PATTARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20JULIA%20DE%20LIMA%20BOMFIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20KARINA%20FONTES%20PRIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20KASSIA%20SPAGNOLLO%20ROSSETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20LIDIA%20ARANA%20CAMIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20LUIZA%20BOMFIM%20LONGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20LUIZA%20OLIVEIRA%20LOMBA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20LUIZA%20RODRIGUES%20MANSUR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20LUIZA%20WASELCIAC%20MICHELETTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20MARIA%20ARLETTE%20PEDROSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20MARIA%20DE%20FREITAS%20PINHEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20PAULA%20DE%20JESUS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20PAULA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20PAULA%20JIORA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20PAULA%20LEITE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20PAULA%20MARQUES%20DE%20LIMA%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20PAULA%20POLETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20PAULA%20SILVINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20PAULA%20WOLF%20QUEIROZ%20SAMPAIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20RAQUEL%20RUIZ%20ABRAHAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANA%20TEREZINHA%20VICENTINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANALU%20VICENTIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANANDA%20BEATRIZ%20PASSETO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDERSON%20ANJOS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDERSON%20CARLOS%20BUENO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDERSON%20DA%20SILVA%20MARCOLINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDERSON%20DE%20RIENZO%20NORONHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDERSON%20GOMES%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDERSON%20HIDEAKI%20MATSUO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDERSON%20LUIS%20NAKANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDERSON%20NOGUEIRA%20COTRIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDERSON%20RODRIGO%20DE%20SOUZA%20CADEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDERSON%20SILVA%20CHAVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDERSON%20VICOSO%20DE%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDERSON%20VINICIUS%20DE%20MEDEIROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20ALBUQUERQUE%20ANICET%20LISBOA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20ANTONIO%20PELEGRINE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20AUGUSTO%20TORBITONI%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20CARDOSO%20CAPELLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20CORREA%20MARCILIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20CORREIA%20TOSSANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20DE%20SOUZA%20TARALLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20DEIENNO%20PANSANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20FRAGALLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20FROES%20DE%20BORJA%20REIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20GOMES%20LAMAS%20OTERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20GUSTAVO%20MALETZKE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20HENRIQUE%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LINHARES%20GIORGINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIS%20ANTONELI%20SENJU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIS%20DELORME',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIS%20MENESES%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIS%20ODA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIS%20PAVAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIS%20PERLOTI%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20COSTA%20DE%20ARRUDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20DELAI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20FERNANDES%20DO%20PRADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20NUNES%20MARINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20NUNES%20TARGINO%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20POLZER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20ROCHA%20D%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20ROQUE%20RUMAQUELLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20LUIZ%20VIEIRA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20MAURICIO%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20MOZETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20MUEZERIE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20NASCIMENTO%20DE%20PAULA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20OLIVEIRA%20DIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20PEREZ%20SEGATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20RAMOS%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20SERNAGLIA%20CERDEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20SILVA%20ROCHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20SILVA%20SCHMUTZLER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20SOUTO%20MAIOR%20PESSOA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20TRENO%20RICARTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20VINICIUS%20ALVARENGA%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20VINICIUS%20ROCHA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRE%20VITAL%20SAUDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREA%20ALEXANDRA%20LIAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREA%20GALLANI%20PINTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREA%20GARRIDO%20RONDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREA%20GOMES%20CAMPOS%20BIANCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREA%20NASTRI%20GRASSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREA%20STOCKL%20GEROSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREA%20VASQUEZ%20GARCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREI%20DALACHI%20ORLANDI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREI%20FELIPE%20MOREIRA%20BUSZINSKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREI%20LEON%20ARIEL%20DINIZ%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREIA%20AKEMI%20KONDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREIA%20CRISTIANE%20GEHRMANN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREIA%20CRISTINA%20DE%20ALMEIDA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREIA%20DA%20SILVA%20BORGES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREIA%20REGINA%20PEREIRA%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREIA%20VIEIRA%20DO%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREO%20DE%20FREITAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRESA%20APARECIDA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRESA%20LEDO%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRESSA%20GONCALVES%20CERQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRESSA%20KESLEY%20ARRIAIS%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDRESSA%20VIANNA%20BOTARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREY%20MONTAGNINI%20CASETTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDREZZA%20FURQUIM%20DA%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANDY%20GONZALEZ%20RIVERA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANGELICA%20ROMAO%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANGELO%20ALVES%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANGELO%20JOSE%20RONCALI%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANGY%20LISETH%20DAVALOS%20MACIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANIELI%20DALSIN%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANNA%20ALEKSEEVNA%20TIMOSHENKO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANNA%20CRISTHINA%20CARMINE%20DE%20MELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANNE%20KAROLINE%20DIMAS%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANNY%20CAROLINY%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20AUGUSTO%20ANDRADE%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20CARLOS%20FALCAO%20PETRI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20CARLOS%20SANTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20DAVI%20MACEDO%20DE%20CASTRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20DIOGO%20SILVA%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20FERNANDO%20DOS%20SANTOS%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20JOAO%20VIALLE%20CORDEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20JOSE%20DE%20LIMA%20BATISTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20MARCOS%20CANDIDO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20MARCOS%20ROCHA%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20MARCOS%20RODRIGUES%20FRANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20RAIMUNDO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20RENATO%20SANCHES%20COLUCCI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20SERGIO%20SILVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ANTONIO%20VITOR%20ELIAS%20SWAID%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARETHA%20BARBOSA%20ALENCAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARI%20MAGALHAES%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARIADNE%20OIDE%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARIAGNA%20RAMON%20CUETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARIAN%20PEREZ%20NARIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARIANE%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARIANY%20ROSSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARIEL%20EMANUEL%20SEBASTIAN%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARIEL%20HENRIQUE%20CANAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARIELE%20ROSSI%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARLENE%20KITA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARLINDO%20RECH%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARMANDO%20EDUARDO%20BARBIERI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARNALDO%20FRANCISCO%20VITALIANO%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTHUR%20ANTONIO%20RUIZ%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTHUR%20FELIZ%20DANTAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTHUR%20HENRIQUE%20PUCCETTI%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTHUR%20MARRETTO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTHUR%20PADIAL%20NOGUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTHUR%20TAVARES%20DOS%20ANJOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTUR%20ANDRE%20ALMEIDA%20DE%20MACEDO%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTUR%20CANTISANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTUR%20MARIANO%20DE%20SOUSA%20MALAFAIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTUR%20SOUZA%20POLIZEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTURO%20GONZALEZ%20QUIROGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARTURO%20JAVIER%20MIGUEL%20DE%20PRIEGO%20PAZ%20SOLDAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ARYADNE%20MIRANDA%20GOMES%20DE%20ASSIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ATHOS%20JACOMINI%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AUDREY%20MAIA%20EGEA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AUGUSTO%20CARBOL%20LORZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AUGUSTO%20CESAR%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AUGUSTO%20FERNANDES%20NALIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AUGUSTO%20HIRAO%20SHIGUEOKA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AUGUSTO%20RODRIGUES%20GUIMARAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=AULUS%20ROBERTO%20ROMAO%20BINELI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BARBARA%20CASTELANO%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BARBARA%20GIMENES%20DE%20CASTRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BARBARA%20PATRICIA%20NEVES%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BARBARA%20SILVA%20VIGNATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BARBARAH%20CALDERAN%20MARTIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIS%20GOMES%20SIQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20ALVES%20FONSECA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20CIRINO%20LUCCHETTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20CRISTINA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20CYRILLO%20AMORIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20DA%20CRUZ%20MENEZES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20DAMASIO%20DE%20FREITAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20DE%20OLIVEIRA%20TRISTAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20GARCIA%20ZILIOTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20GOMES%20RICARDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20LIE%20OKUBO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20MARTINS%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20MENEZES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20SCHMIDT%20MENEGALI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20VIDIGAL%20XAVIER%20DA%20SILVEIRA%20ROSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BEATRIZ%20VILAS%20BOAS%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BENEDITO%20NISHIDA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BENEDITO%20TADEU%20SARAIVA%20FITTIPALDI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BERNARDO%20TRINDADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BIANCA%20ANDRADE%20NICOLAU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BIANCA%20MACIEL%20BRASILINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BIANCA%20MARIA%20CUEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BIANCA%20MARTINS%20BENETOLE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRAYAN%20DA%20SILVA%20PAINO%20CALEFE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRENDA%20CRISTINA%20PINHEIRO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRENDA%20MARQUES%20DE%20PAULA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRENDA%20NOZELLA%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRENO%20ALVES%20GUALDANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRENO%20DA%20SILVA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRENO%20SPINELLI%20COELHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNA%20ALVES%20MALHEIROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNA%20CAROLINA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNA%20CHIERON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNA%20DA%20SILVA%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNA%20ELISA%20ZANCHETTA%20LEAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNA%20MARIELE%20DE%20ALMEIDA%20GUAZZELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNA%20RAFAELA%20VIDORETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNA%20SANCHES%20BISAIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNA%20VAZ%20NEGRAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20ALBERTI%20OHASHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20AMA%20STEPHAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20CARLOS%20DA%20COSTA%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20CARNEIRO%20JUNQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20DA%20SILVA%20MOREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20DANIEL%20CORDEIRO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20DE%20ALMEIDA%20SILVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20DOS%20SANTOS%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20FAVARETTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20FELIPE%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20FELIPE%20NARCIZO%20CARAVIERI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20FERES%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20FERREIRA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20FRANCISCO%20ZAPATA%20CANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20GARCIA%20ROCHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20HENRIQUE%20VEDOVATTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20JOSE%20LEME%20TRIVELLATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20JULIAO%20ROSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20KLAVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20LEMESZENSKI%20BRANDELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20LOLLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20LUIS%20HONIGMANN%20CERESER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20MARQUES%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20MOURAO%20SIQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20OTAVIO%20DE%20CAMARGO%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20PEREIRA%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20PINAFFI%20FRARE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20RIBEIRO%20MARIUTTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRUNO%20VALVERDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=BRYAN%20OLLIVIE%20CUNHA%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAEZILIA%20LOIBL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIKE%20MARCEL%20MENDONCA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAINA%20DE%20OLIVEIRA%20FIGARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAINAN%20LOYOLA%20SCHIAVOLIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20BUTAFAVA%20DIZERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20CASTELO%20FIGUEIRA%20ALCANTARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20CESAR%20ALVAREZ%20KISSAJIKIAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20CESAR%20SOARES%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20CIGAGNA%20DE%20GODOY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20ELIAS%20SAAD',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20FABRICIO%20CONTENTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20FASCINA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20FERNANDES%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20MAIA%20DOS%20REIS%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20SOUZA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20VINICIUS%20OSMAN%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAIO%20VINICIUS%20PEREIRA%20MARCELAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20ABRAM%20FAVERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20AKEMI%20DERRE%20MITOOKA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20APREIA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20ATELLI%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20CARDOSO%20DI%20SANTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20CESTARO%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20CUBAYACHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20DE%20SOUZA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20DIAS%20LOURENCO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20FERNANDES%20HERGERT%20GARCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20GUEDES%20FRANCISCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20KOZLOWSKI%20DELLA%20CORTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20LUNA%20DE%20CAMARGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20MAZINI%20RAMOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20PARIZZI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20RODRIGUES%20SIMOES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20SANTOS%20WOLOCHE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20STEFFANE%20FERNANDES%20TEIXEIRA%20DE%20MOURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILA%20VACCARI%20SUNDERMANN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILLA%20ALVES%20FRANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILLA%20FRAGA%20DO%20AMARAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAMILLO%20SEGRETO%20BARILLARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CANDIDA%20NUNES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARINA%20GEROSA%20POZO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARINA%20MARQUES%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARINA%20ROSSI%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLA%20CRISTINA%20BEDIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLA%20CRISTINA%20DOESCHER%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLA%20DEPIERI%20COLONNA%20HAMAM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLA%20REGINA%20LANZOTTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLA%20RENATA%20MOREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20ALBERTO%20CANDIDO%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20ALBERTO%20DE%20AZEVEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20ALBERTO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20ALBERTO%20HUAIRA%20CONTRERAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20ALEXANDRE%20FIORONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20ANDRE%20SANCHES%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20AUGUSTO%20FIGUEIREDO%20FREIRE%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20AURELIO%20ROSAN%20MENIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20CESAR%20FARIAS%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20DANIEL%20RIQUELME%20CUADROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20EDUARDO%20ALFARO%20MORALES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20EDUARDO%20ANTONIOLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20EDUARDO%20DE%20ARAUJO%20BATISTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20EDUARDO%20MARCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20ENRIQUE%20HERNANDEZ%20SIMOES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20GABRIEL%20GONCALVES%20DE%20MOURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20HENRIQUE%20CHILANTE%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20HENRIQUE%20MIGUEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20HIROJI%20HIROKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20ISSAO%20IGARASHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20RICARDO%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20ROBERTO%20DO%20NASCIMENTO%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20ROBERTO%20MINGOTO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20SOUTO%20ANDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20TAKEO%20TAKAHASHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARLOS%20TOSHIMITSU%20OSHIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARMEN%20BOSCHETTI%20NEIAS%20AYRES%20NETA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARMEN%20COELHO%20PITA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CARMEN%20PAMELA%20ROSALES%20SEDANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINA%20ALVES%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINA%20BEATRIZ%20FILIPIM%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINA%20BISPO%20SANTANA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINA%20BUGALHO%20KOHORI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINA%20HELENA%20LIBANORI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINA%20LOPEZ%20CASTRILLON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINA%20LUCIA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINA%20MENDONCA%20DE%20ALMEIDA%20MALZONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINA%20PEREIRA%20GUIMARAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINA%20TIEMI%20ODASHIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINE%20ARAUJO%20DOS%20SANTOS%20TADEU%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINE%20BORGES%20AZEVEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINE%20DE%20LIMA%20FRACHIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINE%20DE%20SOUZA%20MILANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINE%20FERNANDA%20BARBOSA%20LEAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINE%20ISAAC%20FERREIRA%20ZUIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINE%20KIE%20ISHIMOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINE%20LINDINALVA%20DE%20OLIVEIRA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINE%20LORRAINE%20NEVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINE%20NUNES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINE%20SANDANIELI%20DE%20AGUIAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAROLINNE%20DOS%20SANTOS%20PINHEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CASSIANO%20DA%20SILVA%20TAVARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CASSIANO%20RODRIGUES%20NEVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CASSIUS%20RESENDE%20DUARTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CATARINA%20FORTTI%20PANDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CATIA%20FREDERICCI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAUE%20LUIZ%20THENORIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAUE%20PINHO%20ORTIZ%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAUE%20SIERRA%20DE%20CAMARGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAUE%20STOCCHI%20SOMENSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAUE%20YOSHINAGA%20NOGUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CAUE%20YUICHI%20SHIMABUKURO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CECILIA%20HARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CECILIA%20LUMI%20KAKUDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CECILIO%20COSAC%20FRAGUAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CEDRICK%20BAMBA%20NSIMBA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CELIO%20FERNANDO%20DOS%20SANTOS%20CAMARGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CELSO%20ANDRE%20RODRIGUES%20DE%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CELSO%20HENRIQUE%20DE%20FREITAS%20PARRUCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CELSO%20SATOSHI%20KISHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CESAR%20AUGUSTO%20DE%20ARAUJO%20PINHAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CESAR%20AUGUSTO%20GOMES%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CESAR%20AUGUSTO%20LIPARINI%20ZUCCATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CESAR%20AUGUSTO%20MASCARENHAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CESAR%20CAETANO%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CESAR%20DE%20ALMEIDA%20DA%20FONSECA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CESAR%20GIACOMINI%20PENTEADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CESAR%20YUKISHIGUE%20KIYONO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CEZAR%20AUGUSTO%20CORDEIRO%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CEZAR%20HENRIQUE%20TEIXEIRA%20ZANIOLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CEZAR%20ROGERIO%20BORGES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHEN%20PING%20WANG',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTIAN%20BAUDET',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTIAN%20DANNIEL%20PAZ%20TRILLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTIAN%20DE%20OLIVEIRA%20BUENO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTIAN%20EDUARDO%20BARREIRO%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTIAN%20HIDEKI%20KOTSUBO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTIANE%20DE%20PAULA%20REIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTIANE%20REGINA%20SOARES%20BRASIL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTIANE%20YUMI%20OZAKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTIANO%20PEREIRA%20GUERRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CHRISTOPH%20PEREIRA%20DIAS%20BERGER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CIBELE%20APARECIDA%20ORTIZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CICERO%20RIBEIRO%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CILENE%20RONDOLFO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CINARA%20GUELLNER%20GHEDINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CINTIA%20BEATRIZ%20DE%20SOUZA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CINTIA%20PRADO%20DE%20REZENDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CINTIA%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLARA%20CECILIA%20REYES%20OCHOA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLARICE%20FERREIRA%20DE%20ABREU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLARISSA%20BECHUATE%20DE%20SOUZA%20AZEVEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDEMIR%20MONTEIRO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDETE%20VICTORINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIA%20AKEMI%20KODAIRA%20GOES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIA%20FERNANDA%20PEGHIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIA%20JOSIMAR%20ABRAO%20DE%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIA%20LARISSA%20VIANA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIA%20MENDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIANA%20LAMEU%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIO%20DONIZETI%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIO%20LUIZ%20GONCALVES%20PIRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIO%20RICARDO%20SANDRINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIO%20ROBERTO%20DA%20SILVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLAUDIO%20RODOLFO%20SOUSA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEBER%20BARBIERI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEBER%20DE%20MICO%20MURAMOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEBER%20FABIO%20MORETTI%20VOLPE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEBER%20GIMENEZ%20CORREA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEBER%20RICARDO%20PAIVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEBERSON%20HENRIQUE%20OLIVEIRA%20DA%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEBESON%20CANUTO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEDER%20ROGERIO%20BAGATINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEMENCIA%20NORIEGA%20SONDAHL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEODENIR%20DUARTE%20CARDOSO%20ALVES%20ZANONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEONICE%20DONIZETI%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLEVERSON%20MOREIRA%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLINTON%20AUTO%20DO%20ESPIRITO%20SANTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLODOALDO%20DE%20SOUZA%20FARIA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CLOVIS%20PERIN%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CONRADO%20DE%20CASTRO%20VALISE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CRISTIANE%20CASONATO%20MELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CRISTIANE%20HAYUMI%20TANIGUTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CRISTIANE%20ROBERTA%20DE%20SOUZA%20BARROS%20PINHEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CRISTIANE%20RUOTTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CRISTIANE%20SIQUEIRA%20TAXA%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CRISTIANO%20BORGES%20CARDOSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CRISTIANO%20CLAUDER%20PEREIRA%20FURQUIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CRISTIANO%20FERNANDES%20LAGATTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CRISTIANO%20IGLESIAS%20BENINCASA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CRISTINA%20CELIA%20BARROS%20CAVALCANTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CRISTINA%20VALVERDE%20SHUMAHER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=CRISTINA%20WADA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAIANA%20PEREIRA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAJARA%20MOANA%20BARBOSA%20MOREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DALILA%20DE%20LAS%20GLORIA%20THOMPSON%20RIOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DALTON%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAMARIS%20FORTUNATO%20DE%20ANDRADE%20ANTUNES%20GUERREIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAMIAN%20RODRIGUEZ%20SANCHEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20ARGENTIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20AUGUSTO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20BARBOSA%20GARCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20BASTOS%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20BORGES%20DE%20LAZARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20BROGINI%20DE%20ASSIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20CAMPOS%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20CARNEIRO%20AFONSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20CARNELOSSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20CARNIO%20JUNQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20CESAR%20BRAZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20DE%20ARAUJO%20IMAMURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20DE%20SANTI%20BARRANTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20ESPANHOL%20RAZERA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20FARIAS%20MARINHO%20DO%20MONTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20GAIESKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20GEIGER%20SMIDT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20GURGEL%20TERRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20GUSTAVO%20SANTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20KOJIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20LOMBARDI%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20LOPES%20BRANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20LUCREDIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20MACHADO%20DE%20FARIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20MASSAITI%20MAEDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20MASSAKI%20KUROKAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20OSAKU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20PENALVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20PIROLA%20ROSSELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20RODRIGUES%20BALBIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20RODRIGUES%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20SERGENT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20SILVA%20BARBOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20SOBRAL%20BARRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20SWATER%20DE%20CASTRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20TREVISAN%20BRAVO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIEL%20VILLANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIELA%20DE%20ARGOLLO%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIELA%20LUDVIGER%20INGUI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIELA%20PAES%20DE%20ALMEIDA%20BRAGA%20MATTAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIELA%20PALUMBO%20JORGE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIELA%20TIEPO%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIELE%20FILIPPETTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIELI%20BIANCA%20JUSTO%20DONA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIELLA%20DEBENEDETTI%20TAMBASCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANIELLE%20RITA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANILO%20ALVES%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANILO%20AUGUSTO%20CARDIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANILO%20BARBARO%20GARCIA%20MARTINEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANILO%20BORGES%20VILLARINO%20DE%20CASTRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANILO%20DE%20SIQUEIRA%20FORTUNATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANILO%20LACERDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANILO%20LAVIGNE%20HALLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANILO%20NOGUEIRA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANILO%20RAFAEL%20MOREIRA%20DE%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANILO%20SILVA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DANILO%20TOME%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAPHNE%20BATISTA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DARLES%20LEANDRO%20DA%20CUNHA%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DARLINNE%20HUBERT%20PALO%20SOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVI%20BORACINI%20PRATES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVI%20CLAUDIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVI%20DANDREA%20BACCAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVI%20JOSE%20FARIA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVI%20TASSINARI%20DE%20FIGUEIREDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVID%20ACIOLE%20BARBOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVID%20GUIMARAES%20MONTEIRO%20FRANCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVID%20KONDRASOVAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVID%20MADI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVID%20PEIXOTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVID%20SEBASTIAO%20CABRAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAVID%20SENA%20DE%20MELLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAYANE%20AISE%20MENINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAYANE%20CARVALHO%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DAYANE%20MALTA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DEBORA%20MARCHESAN%20CUNHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DEBORA%20MOREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DEBORA%20RAYSA%20AGUIAR%20PENHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DEBORA%20ZAMARO%20TOLEDO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DEBORA%20ZENAIDE%20GORRI%20MAZZALI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DEBORAH%20MENOCCI%20LARIOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DEBORAH%20THAIS%20DA%20CUNHA%20MENEZES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENI%20WERIK%20RIBEIRO%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENIR%20PEREIRA%20DAMASCENO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENIS%20JHUN%20KUSANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENIS%20MOREIRA%20DOS%20REIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENISE%20BOITO%20PEREIRA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENISE%20DE%20FATIMA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENISE%20DE%20OLIVEIRA%20LINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENISE%20FURIGO%20DE%20MELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENISE%20STRINGHINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENNIS%20NAKAMURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENNY%20ERIKSON%20PIMENTA%20MONTEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DENYSE%20ALVARENGA%20ZANIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DESYREE%20YUMIKO%20SADOYAMA%20RANGEL%20OZAKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20ALONSO%20FERNANDEZ%20MERJILDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20ALVES%20RODRIGUES%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20BALLADOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20CAMPACI%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20CARVALHO%20DO%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20DA%20SILVA%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20DAVID%20INACIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20DE%20ARAUJO%20FRAZILIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20DE%20SOUSA%20MADEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20DOMINGUES%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20DUTRA%20VIOT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20FERNANDO%20BARRERA%20PACHECO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20FERNANDO%20MOMESSO%20BALARDIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20FRANCIS%20GONCALVES%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20GENTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20HUDSON%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20KAROL%20GOUVEAL%20LANA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20MATOS%20DE%20MELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20MUNHOZ%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20MUSARRA%20DOIMO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20NOE%20RODRIGUEZ%20SANCHEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20PACHECO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20PAVAN%20SOLER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20RIBEIRO%20MIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20ROBLE%20VIOL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20RODRIGUES%20DE%20SOUZA%20SPINOLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20RONALDO%20THOMAZ%20SAMPAIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20SCAPIN%20RECALDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIEGO%20TERUO%20MENDES%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIMAS%20TADEU%20DE%20OLIVEIRA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DINIZ%20CARVALHO%20DE%20ARRUDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DINO%20SEIGO%20GUSHIKEN%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIOGO%20CAMARA%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIOGO%20DA%20SILVA%20DIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIOGO%20FAVERO%20STOLFO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIOGO%20FELICIANO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIOGO%20NISHIE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIOGO%20PASCHOALINI%20VOLANTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIOGO%20SATTOLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIOGO%20SILVA%20SANCHES%20JORQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIOGO%20SOUZA%20MAGDALENO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIOMAR%20CAVALCANTE%20DE%20MIRANDA%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIONATAN%20HENRYK%20MALLMANN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DIRCEU%20YOCHIHALU%20OHASHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DJANE%20DINIZ%20RIBEIRO%20LUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOMINIK%20JAKOB%20WEISS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DONIZETE%20LUCIO%20DE%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOUGLAS%20ALEXANDRE%20DE%20FREITAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOUGLAS%20ANTONIO%20ALVAREDO%20PAIXAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOUGLAS%20APARECIDO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOUGLAS%20ASSUNCAO%20MATEUS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOUGLAS%20BARROS%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOUGLAS%20CAVALCANTE%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOUGLAS%20DE%20OLIVEIRA%20FORCHEZATTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOUGLAS%20DINIZ%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOUGLAS%20MAZZARO%20BERTOLIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOUGLAS%20ROMAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOUGLAS%20TAKAO%20MIYATA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DOUGLAS%20ZANOTTA%20PORTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DRAYLSON%20MICAEL%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DRIELE%20BRETONES%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=DUCLERC%20FERNANDES%20PARRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EAMONN%20JOHN%20KEOGH',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDER%20CRISTIANO%20ROSSETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDER%20TIMOTEO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDGAR%20KAZUO%20KUBO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDGAR%20KENJI%20TANAKA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDGAR%20PHELIPE%20DE%20MATOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDGARD%20DA%20CUNHA%20PONTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDI%20SAMUEL%20DE%20BRITO%20MENDONCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDILSON%20DANTAS%20DIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDILSON%20TAMURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDIMILSON%20BATISTA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDIMILSON%20DO%20AMARAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDINEIA%20DALVANA%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDISON%20EDUARDO%20AGUIAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDISON%20JOSE%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDIVAN%20RENATO%20SAVI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDMAR%20BRAULINO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDNILSON%20CESAR%20RODELLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDSON%20DE%20SOUZA%20CHAVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDSON%20KOITI%20NAKAMURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDSON%20LUIZ%20DE%20AQUINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDSON%20MAGALHAES%20TAVARES%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDSON%20MIOSSO%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDSON%20PAULINO%20GARBI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20ALEXANDRE%20SIQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20BONIFACIO%20MOLERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20BRITO%20DE%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20CAPATTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20CORREA%20MATTEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20CUSTODIO%20ANTUNES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20DANTE%20PEREIRA%20PIZORNO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20DE%20BRITTO%20CASTRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20DE%20MATOS%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20DOMINGOS%20BORGES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20FERNANDO%20VELLUDO%20PRADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20FERREIRA%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20FRANCISCO%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20GHERGHI%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20GOMES%20HULSHOF',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20GOMES%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20HENRIQUE%20DE%20PONTES%20ELLERY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20KIOSHI%20TANISHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20LEAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20LUIS%20CARRARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20LUIS%20GARCIA%20ESCOVAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20LUIZ%20ROSSINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20MACHADO%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20MALATESTA%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20MANTOVANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20MARCOS%20DE%20JESUS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20MARTINS%20MARCONDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20MENDES%20AGOSTINHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20MIRANDA%20STEINER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20RAFAEL%20LLAPA%20RODRIGUEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20ROBERTO%20FELIX',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20ROCHA%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20TENORIO%20SIMOES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDUARDO%20UEMURA%20OKADA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EDWIN%20LUIS%20CHOQUEHUANCA%20MAMANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EGON%20HENRIQUE%20SALERNO%20GALEMBECK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EILEM%20DA%20CONCEICAO%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELAINE%20AYUMI%20CHIBA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELAINE%20DE%20SOUZA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELBANO%20DAVID%20BATISTA%20PEREZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELCIO%20RAMOS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELDER%20FIGUEIREDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELIAN%20DA%20SILVA%20MEDEIROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELIANA%20ROSA%20LIMA%20FILHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELIANE%20BRIGIDA%20DAS%20NEVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELIAS%20PEREIRA%20GARCIA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELIEDSON%20PANDORF',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELIESER%20GABRIEL%20SILVA%20CORREA%20SIQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELINA%20CASSIA%20TORRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELINALDO%20SOUSA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELIO%20ANTONIO%20PAULINO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELISA%20COSTA%20NADALINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELISA%20DE%20OLIVEIRA%20GIORNES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELISA%20MEDEIROS%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELISANGELA%20CRISTINA%20TREVISAN%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELISANGELA%20DAYANI%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELISEU%20EVANGELISTA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELISEU%20JUNIO%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELITON%20LUIZ%20SCARDIN%20PERIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELIVELTON%20LAURIANO%20MOREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELIZABETH%20CARVALHO%20LEITE%20CARDOSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELMER%20ROLANDO%20LLANOS%20VILLARREAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELOA%20DE%20LUCCA%20LEITE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELOISA%20JOANA%20DE%20ALMEIDA%20TONELLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELSON%20DE%20SOUZA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELTER%20APARECIDO%20FEITOZA%20DE%20MELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ELTON%20DE%20SOUZA%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EMANUEL%20THALES%20LARA%20PIZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EMERSON%20DAL%20SANTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EMERSON%20LUIS%20BELATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EMERSON%20RODRIGUES%20DE%20CAMARGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ENDI%20SAMBA%20LUAMBA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ENIO%20JOSE%20BOLOGNINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ENRIQUE%20DE%20PAULA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ENZO%20CAUE%20DOS%20SANTOS%20BARCENA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERIC%20HIDEKI%20NOZAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERIC%20RIBEIRO%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERICA%20HAYASHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERICA%20REGINA%20FILLETTI%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERICA%20REGINA%20RODRIGUES%20CINTRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERICK%20GUSTAVO%20DORLASS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERICK%20KEIJI%20HIGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERICK%20MASCAGNI%20FERDINANDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERICK%20RIBEIRO%20OTSUKA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERICK%20SOTO%20CALLEGARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERICO%20DE%20SOUZA%20PASTANA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERIKA%20DO%20CARMO%20OTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERIKA%20FRANKE%20TERRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERIKA%20RABELLO%20MORETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERIKA%20WANESSA%20OLIVEIRA%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ERNANI%20NEGREIROS%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ESTEVAN%20ELTINK%20NOGUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EUGENIO%20FERNANDES%20MONTALLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EUGENIO%20VERAS%20MARIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EURICO%20DE%20PAULA%20ARRUDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVANDRO%20GUELFI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVANDRO%20LUIS%20FERREIRA%20DUGNANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVANDRO%20LUIS%20NOHARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVANDRO%20LUIS%20PRIETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVANDRO%20VALE%20MIQUELITO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVELYN%20KETHERINE%20BRUN%20RAMOS%20MATOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVERLANDIO%20REBOUCAS%20QUEIROZ%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVERTON%20CORTEZ%20ROSADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVERTON%20FERNANDES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVERTON%20LUIS%20SANTOS%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVERTON%20MELCHIADES%20CARDOSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVERTON%20MOREIRA%20ELIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EVERTON%20SEGATO%20ZANON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EWERTON%20ALVES%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=EZEQUIEL%20SILVEIRA%20LUCAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANA%20ALVES%20LOUREIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANA%20DIUK%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANA%20MALUF%20BARROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANA%20NAOMI%20IEGAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANA%20RACOSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANA%20RODRIGUES%20DE%20GOES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANA%20RODRIGUES%20DE%20LARA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANE%20FANTINELLI%20FRANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANE%20MIYUKI%20WATANABE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANO%20ARRUDA%20FERREIRA%20DAS%20GRACAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANO%20DE%20OLIVEIRA%20LUCCHESE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANO%20FERNANDES%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANO%20FRANCA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIANO%20REZENDE%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20AKIO%20KISHIMOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20AKIO%20TAKEUCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20ALCARAZ%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20ANGELUCI%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20ARAUJO%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20ASNIS%20CAMPOS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20BARBOSA%20PINHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20BATAGIN%20ARMELIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20BEKER%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20CARVALHO%20MOTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20CESAR%20BENTO%20DE%20CAMARGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20DA%20SILVA%20RONCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20DALLA%20COSTA%20FIGUEIREDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20FEITAL%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20FUMITO%20SUZUKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20HAJIME%20TSUIJOKA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20LACRETA%20DE%20TOLEDO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20LUIS%20PEREIRA%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20LUIZ%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20MARTINS%20DE%20NOVAIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20MARTINS%20LEITE%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20MINGATTO%20DA%20COSTA%20AMORIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20OLIVEIRA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20PUCCA%20DE%20AVELLAR%20PIRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20RAFAEL%20DA%20ROSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20RODRIGUES%20ORSETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20TOSETTO%20REALE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIO%20VILLAR%20FLORINDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIOLA%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABIOLA%20RIBEIRO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABRICIO%20BELINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABRICIO%20BEZERRA%20GONDIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABRICIO%20DAVID%20COUTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABRICIO%20HENRIQUE%20SIMOZO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABRICIO%20NOGUEIRA%20BUZETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABRICIO%20RODRIGUES%20PASSOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FABRICIU%20ALARCAO%20VEIGA%20BENINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELICIO%20HISSAAKI%20SAKAMOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20ALONSO%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20AUGUSTO%20FLORENTINO%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20BENEDETTI%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20BIZZO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20BUENO%20DUTRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20CABRERA%20RIBEIRO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20CAPITELI%20BERTOCCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20CARAVAGGIO%20DAMASCENO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20CAUE%20LEGAL%20BENEDITO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20DE%20PAULA%20COLLYER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20DE%20SOUZA%20NOGUEIRA%20COELHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20DOS%20REIS%20RUIVO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20DOS%20SANTOS%20FOGACA%20DE%20AGUIAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20DRUDE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20EDUARDO%20MANOEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20FERREIRA%20BOCCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20GOMES%20DE%20MELO%20DELIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20GUSTAVO%20SILVA%20TEODORO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20JOIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20MARRESE%20BERSOTTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20MARTINS%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20MORILLO%20SANZ%20DIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20OLIVEIRA%20FERRAZ%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20OLIVEIRA%20MACIEL%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20PADULA%20SANCHES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20PALOMBI%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20PAULO%20GUAZZI%20BERGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20POSTAL%20RIOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20PROVENCANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20RAFAEL%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20SERTORIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20SOBRAL%20LAZER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20TETSUO%20YAMADA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20WERLE%20MELZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FELIPE%20XIMENES%20VIANA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20ASSUNCAO%20ALVARINHO%20SEPULBEDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20BARBOSA%20ALMENDRA%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20BEATRIZ%20JORDAN%20ROJAS%20DALLAQUA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20CARVALHO%20LEAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20COSTA%20JUBILATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20CRUZ%20FIGUEIREDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20DA%20CUNHA%20CORREIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20DE%20SOUZA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20GARCIA%20MENDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20JUNQUEIRA%20JULIANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20KALIL%20CORREA%20LIMA%20DIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20KAORI%20UCHIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20MAGALHAES%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20MARTINS%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20MIKI%20NAGAHAMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20NICOLELA%20SUSANNA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20PALLONE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20RIGLER%20PERUCCI%20WALTER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20SANTOS%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20SOUZA%20CANOAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDA%20YARA%20DOS%20SANTOS%20FOSCHIANI%20BERTOLINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20ABE%20OHARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20ANTONIOS%20MAMAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20AURELIO%20MARTINS%20MUNIZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20CAVIQUIOLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20CELSO%20LONGHIM%20QUENZER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20CESAR%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20CESAR%20DE%20TOLEDO%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20CLOSS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20DE%20ANDRADE%20CASTILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20DE%20ASSIS%20MELRO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20EDUARDO%20MATURI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20EOVIDIO%20DA%20ROSA%20FIGUEIREDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20GIROTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20HENRIQUE%20BENATTO%20PERINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20HENRIQUE%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20HENRIQUE%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20HENRIQUE%20GARCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20JOSE%20ARROYO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20JOSE%20SOUSA%20NECULQUEO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20NEGRI%20MORALLES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20ORTOLANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20PEREIRA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20PERIN%20MUNERATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20SALES%20PANONT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FERNANDO%20SAMBIANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FIDEL%20ERNESTO%20DIAZ%20ANDINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FILIPE%20GABARRA%20MARCATI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FILIPE%20JOSE%20DAL%20BO%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FILIPE%20LOBO%20DEL%20MONTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FILIPE%20NUNES%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FILIPE%20RODRIGUES%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIA%20BUENO%20MENDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIA%20CORRER%20STENICO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIA%20DOMINGOS%20PACHECO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIA%20FAVERO%20TANGERINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIA%20FERREIRA%20DE%20SA%20E%20BENEVIDES%20FOZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIA%20ORTIGOSA%20CALORI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIA%20PALAVANI%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIA%20YUMI%20TAKEUCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIANE%20CAROLINE%20PAGOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20ADALBERTO%20KUBOTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20APARECIDO%20DOS%20SANTOS%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20AUGUSTO%20ZAMOT%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20CAMARINHO%20MOREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20D%20ANGELO%20PEREIRA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20EDUARDO%20AOKI%20HORITA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20EDUARDO%20TAPPARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20FRANCO%20TRIVELLATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20HENRIQUE%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20HENRIQUE%20PULLITO%20REAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20LUIZ%20DOS%20SANTOS%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20MARIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20MIKIO%20KAWAOKU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20RENATO%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20SATOSHI%20HIGA%20KODAMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLAVIO%20SEIXAS%20LEAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLORA%20MORTARI%20RAMOS%20FONSECA%20MARIOTTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FLORENCE%20ALYSSA%20SAKUMA%20SHIBATA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCES%20ALBERT%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCIANE%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCIELLY%20BERTO%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCILEI%20CAMPOS%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCINE%20DA%20SILVA%20BRANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCIS%20SHINOHARA%20DE%20SANT%20ANNA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20AREAS%20GUIMARAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20DAS%20CHAGAS%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20DE%20ASSIS%20DE%20SOUZA%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20DE%20PAULA%20ASSIS%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20HENRIQUE%20RODRIGUES%20DEODATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20IGNACIO%20RABELLO%20JARDIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20JAVIER%20RAMIREZ%20FERNANDEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20JOSE%20PENA%20Y%20LILLO%20MADRID',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20JOSE%20TALLARICO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20ROBERTO%20CARNEIRO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCISCO%20SEGOVIA%20DE%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANCOIS%20XAVIER%20ALEXANDRE%20RIBAC',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANK%20MOSHE%20COTACALLAPA%20CHOQUE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FRANKLIN%20CESAR%20FLORES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=FREDERICO%20MARTINS%20BIBER%20SAMPAIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20ALMEIDA%20BUENO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20AUGUSTO%20DAVID',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20AUGUSTO%20DE%20ANDRADE%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20BELEM%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20BENITES%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20CALIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20CAPELLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20CHARLUI%20CORREA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20CONDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20COSTABEBER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20DAROZ%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20DE%20CAMPOS%20GUIMARAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20DE%20PAULA%20EDUARDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20DE%20SANTI%20PERNAMBUCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20DE%20SOUZA%20AMARAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20DE%20SOUZA%20MISSALI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20DE%20SOUZA%20PONTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20DERIGGI%20TORRESAM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20DO%20ROSARIO%20MOTTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20FROES%20FRANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20GALEMBECK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20HENRIQUE%20FAUSTINI%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20JOSE%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20KEITI%20KOIKE%20SANTANA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20KLABIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20LEIVA%20MAURILIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20LIMA%20MESQUITA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20LUIS%20ALVES%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20MACORIN%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20MARCELINO%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20MARQUES%20TAVARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20NARDI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20NASCIMENTO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20NEGRELLI%20GARCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20NOVAIS%20GUARNIERI%20SALVADOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20ONUMA%20BIANCONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20PAEZ%20DE%20CASTRO%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20PELLA%20NOGUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20PEREIRA%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20RIBEIRO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20RIBEIRO%20SIQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20RODRIGUES%20DUARTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20ROSA%20ARCANGELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20TACCOLINI%20PAPP',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20TADAO%20KUAE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20TAMASHIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20TETZNER%20MENEGUETI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20TOGNELLA%20POCCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20VALERIO%20PEREIRA%20MANFREDI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIEL%20VICENTE%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20ADRIANO%20SARILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20ANDRADE%20DIAS%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20AZEVEDO%20MOTTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20BORDINASSI%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20BRASIL%20ROMAO%20VELOSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20BRENTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20CARDOSO%20DE%20SOUZA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20DOMINGOS%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20MARINHO%20RIGHETTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20NEYRA%20BELDERRAIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20PATACA%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20ROQUE%20DO%20PINHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELA%20VIEIRA%20SEMPRINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELE%20DE%20FATIMA%20MARCHEZIN%20BACCARIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELE%20LUIZA%20CORDEIRO%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELLA%20CARNEIRO%20JUNQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELLA%20RIBEIRO%20BELINATTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELLE%20DORACENZI%20DA%20CUNHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GABRIELLE%20RESENDE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GARBAS%20ANACLETO%20DOS%20SANTOS%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GEAN%20DAVIS%20BREDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GEANNY%20BORTOLETTO%20CLARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GEBSON%20EDUARDO%20BUSCHER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GEISON%20DA%20SILVA%20LEITE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GEORGE%20AUGUSTO%20FURTADO%20MARTINS%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GEORGE%20ULGUIM%20PEDRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GEORGIO%20FREESZ%20VALADARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GEOVANA%20MACARINI%20FRANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GEOVANNA%20CAMPOS%20MATIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GEOVANY%20CANDIDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GERALDO%20CARLOS%20CARNEIRO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GERALDO%20FERREIRA%20MENDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GERALDO%20GUIESI%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GERMAN%20ANDRES%20ESTRADA%20BONILLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GERMAN%20DARIO%20BUITRAGO%20SALAZAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GERSON%20DE%20SOUZA%20FARIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GERSON%20LUIZ%20BRAND',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GERSON%20MIGUEL%20FETT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GERSON%20PIZZIRANI%20MURARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GESIEL%20GALVAO%20BERNARDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIAMPAOLO%20LUIZ%20LIBRALON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIANNANDREA%20DE%20MOURA%20QUINTALE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIBERTO%20MITSUYOSHI%20YUKI%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIL%20CAPOTE%20RODRIGUEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIL%20VIEIRA%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GILBERTO%20DE%20ALMEIDA%20MEIRELES%20PATROCINIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GILBERTO%20DE%20TADEU%20SANTOS%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GILBERTO%20DOMINGUES%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GILBERTO%20LUIS%20VALENTE%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GINO%20CAPOBIANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANA%20CONTARELLI%20LAMONICA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANA%20FURQUIM%20ANTUNES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANA%20GABRIEL%20PRADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANA%20MORELI%20AVANCINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANA%20TOLEDO%20ALONSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANA%20TREVISAN%20NOGUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANI%20BRUNO%20CICONE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANI%20GIANERI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANNA%20BRUNA%20BRAGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANNA%20COSTA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANNA%20MARQUES%20GRASSINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANNA%20PEREIRA%20CORREIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANNA%20ROCHA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANNA%20VERONEZZI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIOVANNI%20ROGERIO%20FURQUIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GISAH%20AMARAL%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GISELE%20BULHOES%20PORTAPILLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GISELE%20DE%20MORAES%20FERREIRA%20E%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GISELE%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GISELE%20MARIA%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GISELI%20DE%20ARAUJO%20RAMOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GISNANDO%20CARLOS%20DE%20ALMEIDA%20KFURI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIULIA%20BALLESTERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIULIANNA%20ELENA%20BOSCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIULIANO%20ROSSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIULIO%20AUGUSTO%20CERVELLIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIULLIANE%20APARECIDA%20GONCALVES%20FIORAVANTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIULLIANO%20PAES%20CARNIELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIUSEPPE%20VULCANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIUSI%20ASTA%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GIVANILDO%20DE%20SOUSA%20GRAMACHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLAUBER%20MICHELONI%20GALLEGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLAUBER%20NOVAES%20FRANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLAUCIA%20MARIA%20DOS%20SANTOS%20PAIVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLAUCIA%20SCHNOELLER%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLAUCO%20GUAITOLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLAUCO%20MEDEIROS%20VOLPE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLAUCO%20TOSCHIO%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLAUCYA%20CARREIRO%20BOECHAT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLEICA%20OLIVEIRA%20DA%20FONSECA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLEISON%20ELIAS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GLEYSON%20DOS%20SANTOS%20BUENO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GRACE%20M%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GRACIELE%20CRISTINA%20PADOIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GRACIELI%20DALACHI%20ORLANDI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GRACIELLE%20GESTEIRA%20ROCHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GRACIELY%20GOMIDES%20GOBO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GRACIETH%20CAVALCANTI%20BATISTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GRAZIELA%20MAROSTEGAN%20MARUCCI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GRAZIELE%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GREICY%20THAMIS%20MONTANHA%20CORDEIRO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GROVER%20ENRIQUE%20CASTRO%20GUZMAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUALTIERO%20VICTOR%20LOPES%20DOMINGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20ALVES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20AUGUSTO%20ALMEIDA%20LIMA%20DE%20FIGUEIREDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20AUGUSTO%20ONODY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20BATISTA%20LEITE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20BELISSIMO%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20BITENCOURT%20NUNES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20BOIX%20VERDUM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20BORGES%20RESENDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20BRANDAO%20BIBER%20SAMPAIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20CAMARGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20CAZZINI%20CARDOSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20CORREA%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20DE%20ANDRADE%20RISSATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20DE%20LIMA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20DOMINGOS%20CANEDO%20MORAIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20DOS%20SANTOS%20BENETI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20ERIC%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20HENRIQUE%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20HIROKI%20TINEM%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20KAIO%20SOUZA%20DE%20CAMARGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20LUIZ%20DA%20SILVA%20GERMANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20LUIZ%20DA%20TRINDADE%20PINTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20MACHADO%20GAGLIARDI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20MARIANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20MAURO%20ARANHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20MAZONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20MEDEIROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20NOVAES%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20PAVAO%20RIBAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20PIEDADE%20AGUIAR%20FERREIRA%20ROSARIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUILHERME%20SOATTO%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUISELLA%20CLARA%20ANGULO%20ARMIJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20ANSALDI%20OLIVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20BORGES%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20DE%20ABREU%20BRAGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20DE%20MELO%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20DE%20SOUZA%20MATOSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20GIANOTTI%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20GIORDAN%20SANTOS%20CAMARGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20HENRIQUE%20ROBER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20HENRIQUE%20RUBIN%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20JUNIOR%20ESCOBEDO%20TICONA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20LORENCINI%20MARTINS%20PEREIRA%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20LUIZ%20PASQUALINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20MENDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20MERSZI%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20MESCOKI%20SARTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20PEDROSO%20DE%20LIMA%20BRUSSE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20POLI%20LAMEIRAO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20PORTO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20REBECHI%20BRUNASSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20RIBEIRO%20PALMA%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20RODRIGUES%20LURIAL%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20RONCOLATO%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20SALUSTIANO%20CAGNANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20SALVADOR%20BAPTISTA%20DO%20CARMO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20SIQUEIRA%20CATOIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20SOBRAL%20NOVELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=GUSTAVO%20TEODORO%20LAUREANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HALAILTON%20ALEXANDRE%20DA%20MOTA%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HAMILTON%20ANGELO%20ORIENTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HAMILTON%20CARDOSO%20NOGUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HAMILTON%20MARQUES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HEITOR%20COELHO%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HEITOR%20DIAS%20MURBACH',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HEITOR%20ELISIO%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HEITOR%20GOMES%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HEITOR%20LUIS%20POLIDORO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELDER%20FERREIRA%20HENTZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELDER%20MAY%20NUNES%20DA%20SILVA%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELDER%20SCHLICKMANN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELDER%20SILVA%20LOPES%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELENO%20MURILO%20CAMPEAO%20VALE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELENO%20QUEVEDO%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELIO%20BELTRAME',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELIO%20CESAR%20ALVES%20SEABRA%20SALLES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELIO%20DINIZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELIO%20MASSAHARU%20MURATA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELIZANI%20COUTO%20BAZAME',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELLEN%20YUKA%20LEITE%20DA%20SILVA%20KOBAYASHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELMO%20KELIS%20MORALES%20PAREDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELOISA%20AUGUSTO%20ZEN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELOISA%20DINIZ%20DE%20REZENDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELOISA%20HELENA%20MULLER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELOISA%20NOGUEIRA%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELOSMAN%20VALENTE%20DE%20FIGUEIREDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HELOYSA%20HELENA%20DAINEZI%20SALES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HENDERSON%20IVAN%20QUINTERO%20PEREZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HENILSON%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HENRIQUE%20DOS%20SANTOS%20FIORELI%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HENRIQUE%20GOULART%20DE%20SOUSA%20WU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HENRIQUE%20PEDROSA%20CHAGAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HENRIQUE%20PERSICO%20ROSSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HENRIQUE%20PIGATTO%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HENRIQUE%20PRZIBISCZKI%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HENRIQUE%20RODRIGUES%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HENRIQUE%20TETSUO%20TAKAHASHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HENRIQUE%20ULISSES%20TEIXEIRA%20CALDAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HERALDO%20CARLOS%20DOS%20SANTOS%20FABIANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HERCILIO%20GOULART%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HERMANN%20ARTHUR%20FLOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HEVELINE%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HIAGO%20ARAUJO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HIGOR%20BARRETO%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HIGOR%20COELHO%20PHILLIPPI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HILTON%20HENRIQUE%20BERTAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HINGRYD%20APARECIDA%20OLMO%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HIRAN%20RODRIGUES%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HIRNAK%20BENONE%20ARAUJO%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HIROSI%20SUZUKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HITALO%20RODRIGO%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HOMERO%20BARROCAS%20SOARES%20ESMERALDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HUDSON%20RODRIGUES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HUGO%20LEONARDO%20OLIVEIRA%20DA%20CUNHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HUGO%20QUEIROZ%20ABONIZIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HUGO%20RAFAEL%20DE%20OLIVEIRA%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HUGO%20TSUTOMU%20TAKAHASHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HUMBERTO%20D%20MUNIZ%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=HUMBERTO%20RODRIGO%20SANDMANN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IAN%20ANDERSON%20GUIMARAES%20KONOPCZYK%20MALUF%20FARHAT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IASMIN%20ROSANNE%20SILVA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ICLEIA%20SIQUEIRA%20BARRETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20AUGUSTO%20BRANDAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20AUGUSTO%20NEGRI%20DONINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20BRUNO%20DE%20JESUS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20CORREA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20FABIO%20STEINMACHER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20FERNANDES%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20GABRIEL%20MARTINS%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20LIMA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20LUIS%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20RAPHAEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20RODOLFO%20BERALDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20SARTI%20MARIANO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IGOR%20TADEU%20SILVA%20BATISTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ILDOMAR%20COELHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ILORAN%20DO%20ROSARIO%20CORREA%20MOREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=INGRID%20DE%20MIRANDA%20ESTEVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IRIS%20TODESCHINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IRVING%20SOARES%20DE%20SOUZA%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISAAC%20GUILLERMO%20GONZALES%20VIZCARRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELA%20AMANDA%20POLSON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELA%20CARVALHO%20VELLOSO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELA%20DE%20OLIVEIRA%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELA%20LIBERATOSCIOLI%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELA%20LULA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELA%20SILVA%20FILIZOLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELA%20SOARES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELLA%20MAGNANTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELLA%20MALACHIAS%20GASPAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELLA%20SALGADO%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELLA%20SILVA%20FRAZAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELLE%20CARDOSO%20ALVES%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELLE%20DE%20LIMA%20JAHNZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISABELLY%20MALHEIRO%20PINHEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISADORA%20MARTINI%20COELHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISADORA%20SILVA%20FERNANDES%20CUSTODIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISAIAS%20PAULINO%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISAQUE%20VIEIRA%20DE%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISEU%20DA%20SILVA%20NUNES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISIS%20FERNANDA%20MASCARIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISMAEL%20TEODORO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISMAR%20FRANGO%20SILVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISRAEL%20AUGUSTO%20DA%20COSTA%20E%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ISRAEL%20DE%20MORAES%20PINTO%20CANECA%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ITALO%20ADRIANO%20MORAES%20DE%20FREITAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ITALO%20PELICAO%20CALIARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ITALO%20URIEL%20GERALDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ITAMAR%20APARECIDO%20LOURENCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IURY%20TERCIO%20SIMOES%20DE%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IVAIR%20ROBERTO%20BERTOLLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IVAN%20ALVES%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IVAN%20GALHARDONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IVAN%20NEVADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IVAN%20ROGERIO%20BIZARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IVAN%20SILVESTRIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IVANETE%20BELLUCCI%20PIRES%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IVO%20ALESSANDRO%20RECK%20CLAUDINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IVO%20KENJI%20KOGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IZABEL%20FERNANDEZ%20MONTEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IZABELA%20MENDES%20BITENCORT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IZABELA%20QUEIROZ%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IZABELLA%20RABELO%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IZABELY%20GONCALVES%20ARAUJO%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=IZIS%20CAVALCANTI%20ALBUQUERQUE%20DE%20SOUZA%20QUEIROZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JAILMA%20JANUARIO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JAIME%20ROBERTO%20SCHMIDT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JAIME%20VALERIO%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JAIR%20ALVES%20DOS%20SANTOS%20AGUILAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JAMES%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JAMILE%20DE%20SOUZA%20MARCONDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JAN%20HADRAVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JANAINA%20BAPTISTA%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JANAINA%20BARROS%20DE%20FARIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JANEANE%20MERRARI%20CARNEIRO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JARDEL%20EDSON%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JASMINE%20DE%20CARVALHO%20BIANCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JAVIER%20ALEXANDER%20MONTOYA%20ZEGARRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JAVIER%20DARIO%20PULIDO%20GOMEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JAYSON%20CAMPOS%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEAN%20ANDRADE%20CANESTRI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEAN%20CESAR%20SOUZA%20BATISTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEAN%20CLAUDIO%20KANNENBERG',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEAN%20GOMES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEAN%20JUNIOR%20DE%20MORAIS%20REZENDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEAN%20MICHEL%20ROCHA%20SAMPAIO%20LEITE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEAN%20MIMAR%20SANTA%20CRUZ%20YABARRENA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEAN%20VITOR%20DE%20PAULO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEAN-PHILIPPE%20PIERREL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEANE%20MENDES%20DE%20ALMEIDA%20BARBOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEFERSON%20ALVES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEFERSON%20ROBERTO%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEFERSON%20RODRIGUES%20BUENO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEFFERSON%20DE%20OLIVEIRA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEFFERSON%20MARY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEFFERSON%20OTONI%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEFFERSON%20RODRIGUES%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEFFERSON%20RODRIGUES%20LEME',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEFFERSON%20VIEIRA%20PASCON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEISON%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JENIFFER%20LENSK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JENNIFER%20WELLEN%20SILVA%20SIQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JENNY%20CAROLINA%20LOMBO%20CARRILLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEOVANE%20HONORIO%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JEREMIHAS%20SULZBACHER%20CARUSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JERONIMO%20NONATO%20TORRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JESSE%20OLIVEIRA%20DE%20FREITAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JESSICA%20CALDEIRA%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JESSICA%20CASAROTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JESSICA%20DOMINGUES%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JESSICA%20FERNANDA%20BARETTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JESSICA%20HELISA%20HAUTRIVE%20ROSSATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JESSICA%20NATSUMI%20YAMASHIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JESSICA%20TAYANE%20CLEMENTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JHEIMIS%20FERNANDES%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JHONANTHAN%20JEOVANI%20FERREIRA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JHONATAS%20PEDROSA%20MARIM%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JHONNE%20PEDRO%20PEDOTT%20SANTANA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOACY%20DE%20LIMA%20FREITAS%20JR.',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAHANNES%20BRUNO%20DIAS%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOALISSON%20GONCALVES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOANA%20D%20ARC%20FELIX%20DE%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOANA%20ESTHER%20GONZALES%20MALAVERRI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOANA%20RAMOS%20RIBEIRO%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOANA%20STEFANI%20VIANA%20COELHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOANNE%20HELOISE%20QUAGLIATO%20FORSTER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20ANDRADE%20GRILO%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20AUGUSTO%20ANCHESCHI%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20AUGUSTO%20BAGATTINI%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20CARLOS%20BALDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20CARLOS%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20CARLOS%20TEIXEIRA%20SOARES%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20CEZAR%20PILOTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20DURVAL%20ARANTES%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20FELIPE%20GROMBONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20FERREIRA%20MENDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20GABRIEL%20CAMACHO%20PRESOTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20GUILHERME%20DAREZZO%20MARTINS%20DE%20FRANCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20GUILHERME%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20HENRIQUE%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20HENRIQUE%20SASS%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20LUIZ%20CHELA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20MARCELO%20BONTURI%20VON%20ZUBEN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20MARCOS%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20MARCOS%20VILLELA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20MARCUS%20MARTINS%20REIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20MARTINS%20CORTEZ%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20MIGUEL%20MORENO%20FERREIRA%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20MORAIS%20DA%20SILVA%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20PAULO%20ALCANTARA%20E%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20PAULO%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20PAULO%20COELHO%20LEITE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20PAULO%20LUIZ%20CRISPIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20PAULO%20MARTINS%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20PAULO%20MULLER%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20PAULO%20PEREIRA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20PAULO%20PEREIRA%20ZANETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20PEDRO%20DA%20SILVA%20BRAVO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20PLAZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20RENATO%20GALLO%20VIEIRA%20DA%20ROCHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20SOARES%20DE%20OLIVEIRA%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20VITOR%20MENGATTO%20GIMENES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20VITOR%20MOREIRA%20NICOLETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20VITOR%20NAPOLITANO%20VIOTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAO%20VITOR%20SILVA%20BISPO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOAQUIM%20MANOEL%20JUSTINO%20NETTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOE%20TONOLLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOEL%20RODRIGUES%20RUIVO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOELSON%20DE%20CARVALHO%20ROCHA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOHN%20FREDDY%20GARAVITO%20SUAREZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOHN%20LENNON%20OLIVEIRA%20COUTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOICE%20CASSIMIRO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JONAS%20APARECIDO%20GALLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JONAS%20DORIGAN%20GIACOMINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JONATAS%20ALVARENGA%20MAXIMIANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JONATAS%20SILVA%20ROMANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JONATHAN%20DE%20ALMEIDA%20LAPA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JONES%20ERNI%20SCHMITZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JONES%20MAIKON%20MARCONSSONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JONNATHANN%20SILVA%20FINIZOLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JORGE%20ARTURO%20MARTIN%20POLAR%20SEMINARIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JORGE%20FRANCO%20MARINGOLI%20CARDOSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JORGE%20LUIS%20BIANCHETTI%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JORGE%20LUIZ%20NICOLAU%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JORGE%20MURILO%20SUGUISAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JORGE%20NICOLAU%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ALBERTO%20FERNANDES%20CANEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ANGEL%20MEDEL%20TIRADOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ANTONIO%20CASTILHO%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ANTONIO%20MOREIRA%20DE%20REZENDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20AUGUSTO%20COURA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20BRESIL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20CARLOS%20DOS%20SANTOS%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20CARLOS%20LAZZARI%20ALBERTIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20CARLOS%20PIZOLATO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20CARLOS%20TOLEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20D%20AMICO%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20EDUARDO%20BELO%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20EDUARDO%20CHIARELLI%20BUENO%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ERINALDO%20DA%20FONSECA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ERNESTO%20VALLESPIN%20CAMPOY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20FERNANDES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20FERNANDO%20BURDELIS%20DA%20COSTA%20NEVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20FERNANDO%20E%20TOLEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20GERALDO%20GODOI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20GUILHERME%20COSSETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20LUIS%20SEGATTO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20MAIA%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20MARCELO%20PACHECO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20MARCELO%20POPI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20MARIA%20PASSARELLI%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20MASSAYOSHI%20MIYAGUSKO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20PAES%20DA%20COSTA%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20PEDRO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20POTT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20RENATO%20PONTES%20CAMBRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20RICARDO%20SORIA%20PORRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20RICARDO%20TEIXEIRA%20BRITO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ROBERTO%20ANDRADE%20PEREIRA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ROBERTO%20BOFFINO%20DE%20ALMEIDA%20MONTEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ROBERTO%20CAON%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ROBERTO%20ESCODEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ROBERTO%20GODOY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ROBERTO%20OSSES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20ROMARY%20FREITAS%20BRANDAO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20TADEU%20ALDRIGUE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20VICENTE%20VALARELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20VICTOR%20ALCANTARA%20DA%20GRACA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSE%20VIDAL%20BELLINETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSEMARA%20CONCEICAO%20DE%20MENDONCA%20FLAUSINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSENIVALDO%20BENITO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSIANE%20CRISTINA%20VERZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSIAS%20DE%20LIMA%20BERNARDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSIMAR%20EDINSON%20CHIRE%20SAIRE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSIMARI%20MARTARELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSSEMAR%20AVILA%20DE%20MORAIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOSUE%20CAMELO%20DOS%20SANTOS%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOYCE%20MARA%20BARBOSA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOYCE%20RODRIGUES%20ABRANTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JOYCE%20STENICO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JUAN%20CARLOS%20MINANGO%20NEGRETE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JUAN%20HERBERT%20CHUCTAYA%20HUMARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JUAN%20PABLO%20CORREA%20SANTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIA%20ALUOTTO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIA%20BARBIERI%20DE%20OLIVEIRA%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIA%20CAPOZZI%20FRANKE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIA%20KIMURA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIA%20REINALDI%20FINASSI%20PINTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIA%20SGARIONI%20LIMA%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIAN%20ANDRES%20VARGAS%20GRAJALES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIAN%20AUGUSTO%20BORALLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIAN%20GERALDES%20MONTEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20ALVES%20FUSARO%20DA%20ROCHA%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20ALVES%20PICCOLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20ARCADEPANI%20CORREA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20DE%20PAULA%20VITAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20FERNANDA%20ALMEIDA%20CASTRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20GOUVEIA%20DENIPOTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20HELENA%20VERISSIMO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20LOURENCAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20MITIE%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20MORALES%20RONCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20NUNES%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20PATRICIA%20BRAGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20PEDROZO%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20PEREIRA%20SCHNETZLER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20SOUZA%20PIRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20SUYAMA%20HIGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20TERUMI%20TAKAHASHI%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANA%20VIANA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANNE%20LARENS%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANO%20ALVES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANO%20CESAR%20GIUSTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANO%20JUNIO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANO%20KAWAI%20SOJA%20FONSECA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIANO%20RODRIGUES%20SANGALLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIENE%20ROBERT%20DA%20ROCHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIO%20CESAR%20ANASTACIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIO%20CESAR%20CANDIDO%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIO%20CESAR%20CARRIEL%20SANDRONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIO%20CESAR%20DE%20OLIVEIRA%20TORRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIO%20HENRIQUE%20FILIPINI%20ROSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULIO%20MITSUGUI%20KAWAI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JULLIANA%20CARVALHO%20CAMPOS%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JUNIOR%20ANTONIO%20DECARLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JUNIOR%20CESAR%20EVANGELISTA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JURGEN%20SAND',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JUSLEY%20LIRA%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=JUSSARA%20MIRANDA%20DIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KAILA%20PETRONILA%20MEDINA%20ALARCON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KAIO%20CARVALHO%20DELAMICO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KARCIUS%20DAY%20ROSARIO%20ASSIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KAREN%20HELENA%20DE%20ANDRADE%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KAREN%20HONDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KAREN%20RAMOS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KARINA%20LUCAS%20DA%20SILVA%20BRANDAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KARINA%20RUBIO%20DE%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KAROLINY%20DELAVALE%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KARYN%20FERNANDA%20MANIERI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KATIA%20BARRETO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KATIA%20PAULA%20ALEIXO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KAUE%20FREITAS%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KEILA%20NAZARE%20DE%20OLIVEIRA%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KEISE%20NAYARA%20FERNANDES%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KELCY%20CAROLINA%20SANTOS%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KELLI%20REJIANE%20POKER%20DE%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KELLY%20CRISTINA%20CAMARGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KELLY%20MEDEIROS%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KELLY%20VITORIA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KELVIN%20SOUSA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KENDY%20SIERRAS%20NAMBU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KENNEDY%20CORREA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KERAN%20HANNIEL%20RAMASSAMY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KEVIN%20CARVALHO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KIM%20NEHRING',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KIMBERLI%20TAEMY%20ISOO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KLAAS%20MINNE%20VAN%20DER%20ZWAAG',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KLEICY%20CAVALCANTE%20AMARAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=KLEYTON%20NISHIMURA%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAECIO%20SOUSA%20BARBOZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAERCIO%20LIMA%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAIS%20CANEVA%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAIS%20JANAINA%20APARECIDA%20HENRIQUE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAIS%20TRAJANO%20MENDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LALESKA%20RODRIGUES%20SALOMAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAMECK%20SILVA%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LANDERCI%20NUNES%20RIBEIRO%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARA%20BACCAR%20FONSECA%20CASAROTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARA%20GIORGINA%20PARODI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARA%20MARIA%20SILVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARISSA%20BARBOSA%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARISSA%20COIMBRA%20MARTINEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARISSA%20CRESCENCIO%20VAZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARISSA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARISSA%20GONCALVES%20DE%20PASCHOAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARISSA%20MOREIRA%20DE%20BRITO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARISSA%20NOGUEIRA%20OLMO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARISSA%20PELOSO%20MOBILON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARISSA%20RUIZ%20PRESTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARISSA%20VIEIRA%20MUSETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARISSA%20ZANIN%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LARYSSA%20FLEURY%20FRANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAURA%20CONES%20BIGARAM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAURA%20DINIZ%20VAGNINI%20ALAVARSE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAURA%20FRENEDA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAURA%20GIOVANA%20GIACOMINI%20DE%20LACERDA%20MIRANDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAURA%20GONCALVES%20CARR%20TRAINA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAURA%20LETICIA%20DE%20FREITAS%20CHAMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAURA%20MACEDO%20ROCHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAURA%20OLIVEIRA%20REBOUCAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAURIE%20MARCHETI%20MANTOVANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LAZARO%20MORATELLI%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEA%20APARECIDA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20APARECIDO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20CESAR%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20CESAR%20VIOTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20D%20AGOSTINO%20AMARAL%20CONTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20DOS%20REIS%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20FERREIRA%20MAZZILLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20HENRIQUE%20RIBEIRO%20IVO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20JACINTHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20JOSE%20TEIXEIRA%20BRAGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20LOPES%20BORGES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20MONTEIRO%20POLISELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20PAGANI%20MASSAROTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20PAOLO%20RICCI%20SOSSAI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20RESENDE%20MUNDIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20RUBIM%20DE%20FREITAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEANDRO%20TEICHIMANN%20CABRAL%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LECIENI%20FRACOLA%20MEDEIROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEILA%20SPINOLA%20VITORINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEISSI%20MARGARITA%20CASTANEDA%20LEON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEIZZA%20FERREIRA%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LENON%20LIBERDADE%20ALVARES%20GUIMARAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONAM%20JOAO%20LEAL%20DE%20PAULA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20ADEMIR%20TONEZI%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20ANDRE%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20AUGUSTO%20GUIMARAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20BELCHIOR%20DA%20SILVA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20BERLIM%20SCHNEIDER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20BLANGER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20DE%20BRITO%20GALEGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20DE%20CAMPOS%20PEREIRA%20LOPES%20RUZANTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20DE%20OLIVEIRA%20SIBELA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20DE%20SOUZA%20DAMALIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20DOS%20SANTOS%20ALMELIN%20GUIMARAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20GABRIEL%20CARVALHO%20DA%20COSTA%20CONCEICAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20HENRIQUE%20FAZAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20HENRIQUE%20MOREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20HENRIQUE%20ROBERTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20HENRIQUE%20TOMASSETTI%20FERREIRA%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20JOSE%20RAMOS%20FREITAS%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20MATTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20MENDES%20RIBEIRO%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20MENEZES%20CAPETTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20PICCIONI%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20PIRES%20DE%20SANTANA%20CASTRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20SANCHES%20DO%20CARMO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20SOARES%20CARRIJO%20NAZARE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20SOI%20SATO%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20VIEIRA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20VIEIRA%20VON%20ZUBEN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONARDO%20VINICIUS%20PEREIRA%20PRETES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONEL%20ENRIQUE%20SANCHEZ%20CURRIHUINCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONEL%20MORENO%20MOLANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LEONICE%20DOS%20REIS%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETHICIA%20SUZIGAN%20CORNIANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20CANDANCAN%20CORREA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20CONEGLIAN%20DE%20CONTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20CRUZ%20HENZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20DA%20SILVA%20BOMFIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20DA%20SILVA%20SANTA%20BARBARA%20DE%20JESUS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20DE%20AGUIAR%20DIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20DE%20SOUZA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20FERNANDA%20LAVEZZO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20FERREIRA%20MAGNIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20HECK%20BELLAVER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20LI%20KOGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20MARIANO%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20MINERVINO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20MURBACK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20POLLO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20SANFILIPPO%20ROJAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LETICIA%20TIEMI%20TOMISAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LHAIS%20SILVERIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LIDIANE%20LAILA%20ALBRECHT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LIGIA%20AKEMI%20KIYUNA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LIGIA%20XIOL%20MORAIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LILIAN%20MIHO%20SAKUNO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LILIANE%20NAOMI%20SUZUKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LILIANE%20VENTURA%20SCHIABEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LILIANI%20CRISTINA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LITERCILIA%20SOARES%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LIVIA%20DE%20SOUSA%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LIVIA%20MARIA%20DE%20OLIVEIRA%20CIABATI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LIVIA%20MATEUS%20REGUENGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LIVIA%20PEDRINO%20SIMAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LORD%20FLAUBERT%20STEVE%20ATAUCURI%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LORENA%20CRISTINA%20RODRIGUES%20MANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LORENO%20MENEZES%20DA%20SILVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUAN%20DA%20SILVA%20DIAS%20RABELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUAN%20MARTINS%20DE%20ARAUJO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUAN%20YUKIO%20NEVES%20HOSHINA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUANA%20GONCALVES%20ZAMARRENHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUANA%20PASSOS%20REIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUANA%20PRISCILA%20RODRIGUES%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUANA%20ROSSI%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUANA%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUANA%20ZANCHETTA%20TESCHE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCA%20MARCELLO%20MANTOVANELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20AMARAL%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20ANTONIO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20APARECIDO%20REZENDE%20ERCOLINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20AUGUSTO%20SARDELARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20BIGOLIN%20LORENZON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20BOTELHO%20COELHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20CASSIM%20CAIRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20COELHO%20RIGHETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20CORASOLLA%20CARREGARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20D%20ELIA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20DE%20BIASI%20CABRAL%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20DE%20CAMARGO%20ZECHIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20DE%20OLIVEIRA%20FUJII',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20DE%20OLIVEIRA%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20DE%20PAIVA%20PIROLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20DE%20PAULA%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20DE%20SOUZA%20FRANCISCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20DO%20AMARAL%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20EDUARDO%20ACORSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20FERNANDO%20DOS%20SANTOS%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20FERREIRA%20DIB',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20FERREIRA%20PONTIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20FIORI%20ROSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20FLORENTINO%20VARELLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20FRIGO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20GONCALVES%20TESSARINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20GOZZI%20RODRIGUES%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20HENRIQUE%20DE%20JESUS%20SIGAKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20HENRIQUE%20SARRACINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20HUBACEK%20TSUCHIYA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20LEONARDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20LOUREIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20MACIEL%20DIANIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20MAGON%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20MARQUES%20DE%20SOUSA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20MENDONCA%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20NICOLLI%20TOSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20ORTEGA%20VENZEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20PFISTER%20DE%20MENEZES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20PINAZZA%20MARCONATTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20PIRES%20GOMES%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20PRUDENTE%20DA%20PAIVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20REZENDE%20TEDESCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20RODRIGUES%20DE%20GOES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20ROESLER%20TIRADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20ROMEIRO%20PELLOZO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20SANFELICI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20SIQUEIRA%20GERMANO%20PERECIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20TAKESHI%20CHIGAMI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20TEODORO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20TOMILHEIRO%20SANCASSANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20VICENTE%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20VINICIUS%20AVANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCAS%20ZANCO%20LADEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIA%20HELENA%20DE%20OLIVEIRA%20BORGES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIAN%20BERALDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANA%20FARINA%20ALMANSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANA%20LISI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANA%20RELLY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANA%20ROSA%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANE%20CRISTINA%20TRULHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANE%20MENDONCA%20GRANADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANO%20AUGUSTO%20FERNANDES%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANO%20CASSIO%20LUGLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANO%20KIYOSHI%20SOUZA%20NAGANAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANO%20MARCELO%20CHRISTOFOLETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANO%20MENDES%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANO%20TADEU%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANO%20TEIXEIRA%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIANO%20TITO%20DE%20MOURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCILAINE%20MAISA%20SAVASSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIMARA%20FERNANDA%20FERRARA%20GALBIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIO%20FELIPE%20GARCIA%20MUNHOZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUCIO%20JOSE%20HERCULANO%20CORREIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUDIMILA%20OTERO%20OLIVEIRA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUDYMILA%20RIBEIRO%20BORGES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20ALBERTO%20ROSERO%20ROSERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20ANTONIO%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20ARTURO%20PEREZ%20LOZADA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20BRUNO%20PEREIRA%20DO%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20CARLOS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20CLAUDIO%20LEITE%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20EDUARDO%20NERY%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20ENRIQUE%20BROSSARD%20GONZALEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20FELIPE%20LOPES%20MILARE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20FELIPE%20PIRES%20BARROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20FERNANDO%20BARBOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20FERNANDO%20CURCI%20CHAVIER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20FERNANDO%20MARTINS%20CARLOS%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20FILIPE%20DE%20MOURA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20GUILHERME%20BRUNETTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20GUSTAVO%20CHRISPIM%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20GUSTAVO%20DE%20SOUZA%20PENA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20GUSTAVO%20WESZ%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20H.%20R.%20CISTERNA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20HENRIQUE%20LOPES%20PIRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20HENRIQUE%20ROMANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20HENRIQUE%20VENTRILHO%20FRANCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20MAURO%20MOREIRA%20PENHOLATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20OTAVIO%20MENDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20ROBERTO%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20SERGIO%20WILKE%20MUHLEN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIS%20VALVERDE%20REBAZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUISA%20BRANDAO%20CAVALCANTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUISA%20NAOMI%20DE%20CASTRO%20SUDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20ALBERTO%20CANETTIERI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20ALFREDO%20PUGA%20CISNEROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20ANTONIO%20FRAMARTINO%20BEZERRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20AUGUSTO%20MOTA%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20CARLOS%20SALGUEIRO%20DONATO%20BACELAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20CLAUDIO%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20EDGAR%20TUSQUI%20MASCARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20EDUARDO%20ARAUJO%20ZUCCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20EDUARDO%20BONASSOLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20EDUARDO%20GUARDIA%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20EDUARDO%20NISHINO%20GOMES%20DO%20AMARAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20EUGENIO%20SANTOS%20ARAUJO%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20FELIPE%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20FELIPE%20DO%20NASCIMENTO%20BRITO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20FELIPE%20SETHA%20FOSSATTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20FERNANDO%20DE%20ALMEIDA%20BICALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20FERNANDO%20PINTO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20FERNANDO%20RODRIGUES%20VON%20ATZINGEN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20FERNANDO%20SATOLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20FERREIRA%20DE%20LIMA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20GUILHERME%20CERIGATTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20GUSTAVO%20DA%20ROCHA%20CHARAMBA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20GUSTAVO%20NASCIMENTO%20MARQUETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20GUSTAVO%20RAVETTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20GUSTAVO%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20GUSTAVO%20SEVERIANO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20HENRIQUE%20BORGES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20HENRIQUE%20DE%20FARIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20HENRIQUE%20DE%20MOURA%20TEIXEIRA%20BALLOTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20HENRIQUE%20DEL%20COL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20HENRIQUE%20GALEOTI%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20HENRIQUE%20KIEHN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20HENRIQUE%20YAMANAKA%20MELLUCCI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20JOSE%20TONUS%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20MARCELO%20LEITE%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20PAULO%20SEROA%20TAVARES%20DE%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20RAIMUNDO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZ%20ROBERTO%20DE%20OLIVEIRA%20BENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUIZA%20CORTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUMA%20GALLACIO%20GOMES%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LUZIANA%20SANT%20ANA%20SIMOES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=LYANI%20ROSSI%20KLINK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAGALI%20MERCEDES%20RONDON%20GONZALEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAGALI%20SANCHES%20DURAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAGDALENA%20SOFIA%20PEREZ%20FERRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAIARA%20CRISTINA%20TROPEIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAICKEL%20HUBNER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAICON%20VILABRUNA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAINARA%20CHICARELI%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAIRA%20DE%20PAULA%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAIRTON%20SANTOS%20DE%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAISA%20DE%20CARVALHO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAISE%20PASTORE%20GIMENEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAISE%20PERINI%20MAYA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MANOEL%20MESSIAS%20VIEIRA%20FARIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCEL%20AMBO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCEL%20BRUN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCEL%20CORREA%20DE%20MELLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCEL%20DA%20SILVA%20LUNGHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCEL%20FIGUEIREDO%20GARCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCEL%20PIETER%20HUIJSER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCEL%20POPOLIN%20DE%20ARAUJO%20CUNHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCEL%20ROBERTO%20BELINI%20TESSARINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCEL%20TAKESHI%20HOROIWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELA%20ANDREA%20DURAN%20HAUN%20SENATORE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELA%20DE%20OLIVEIRA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELA%20FERNANDES%20SILVA%20DE%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELIS%20ALESSANDRO%20MORAIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELLE%20BALDUINO%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20ALVES%20FAVARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20ANTONIO%20NERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20BALLESTIERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20BASTOS%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20CASARI%20CARLOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20CRISCUOLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20DA%20SILVA%20BARREIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20EDUARDO%20DE%20ANDRADE%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20GALDINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20GALVAO%20DE%20ALMEIDA%20PINTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20HENRIQUE%20FRITZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20HIROSHI%20TAKEDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20LERESCHE%20MICCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20MOURA%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20NOGUEIRA%20FURTADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20PEREIRA%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20PETRUCELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20QUINTAS%20PARRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20REBELLO%20PIMENTEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20SIQUEIRA%20CATOIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20VAZ%20DO%20AMARAL%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCELO%20VIANA%20DE%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIA%20BEATRIZ%20PEREIRA%20DOMINGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIA%20DINIZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIA%20MARIA%20RIPPEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIA%20MARIA%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIA%20MAYUMI%20HARADA%20HAGUIWARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIA%20VALERIA%20ROCHA%20DA%20CRUZ%20MONTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIO%20AMADO%20DUARTE%20SANTANA%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIO%20BARBOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIO%20CERQUEIRA%20DE%20FARIAS%20MACEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIO%20DO%20ROSARIO%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIO%20FERNANDO%20FLORES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCIO%20HADDAD%20DANTAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCO%20ANTONIO%20ALVES%20CAIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCO%20ANTONIO%20BERGAMASCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCO%20ANTONIO%20FERNANDES%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCO%20ANTONIO%20LIMA%20CARIBE%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCO%20ANTONIO%20TAMAI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20AKINORI%20MASSUDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20ANTONIO%20BOTELHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20ANTONIO%20ISAAC%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20AURELIO%20ALVES%20DE%20MACENA%20SA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20BARBOSA%20SALLES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20CESAR%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20DANIEL%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20EDGAR%20HERKENHOFF',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20FABIO%20JARDINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20FERREIRA%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20HIDEKI%20YAMANAKA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20JOSE%20CHAGAS%20CERIMARCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20JOSE%20DE%20MORAIS%20MENDONCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20MONTEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20OKAMURA%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20PAULO%20CALDERARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20PAULO%20MARCUZ%20VENIER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20PINTO%20MONTEIRO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20RADOICO%20DE%20FIGUEIREDO%20GUIMARAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20RAPHAEL%20BERNARDI%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20ROBERTO%20ALVES%20MEDEIROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20ROBERTO%20CATALFO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20RODRIGUES%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20SOBRAL%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20SOUZA%20ULIANA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20VENICIOS%20CORRER%20DANTAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCOS%20VINICIUS%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCUS%20PAULO%20VENDITO%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCUS%20VINICIUS%20MELEGARI%20CASTILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCUS%20VINICIUS%20VITORATTI%20DE%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARCUS%20VINICIUS%20WENDEL%20JORDAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARESSA%20FABIANO%20CUEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20APARECIDA%20TAKEKO%20TOMON%20AVELINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20CAROLINA%20ANDRADE%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20CECILIA%20MOTTA%20TORRES%20GIGLIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20CRISTINA%20LORENCINI%20DE%20BRITTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20ELISA%20DO%20CARMO%20VENEGAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20ERICA%20DO%20NASCIMENTO%20LINHARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20GABRIELA%20ORTIZ%20LADWIG',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20GABRIELA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20INES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20ISABEL%20DE%20CARVALHO%20SOLER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20JOSE%20ALVES%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20JULIA%20FESTA%20FRANZINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20LUCIA%20DE%20PAIVA%20JACOBINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20LUCIA%20DEL%20ROSARIO%20CASTRO%20JORGE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20LUIZA%20SILVA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20MADALENA%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20MERCEDES%20GAMBOA%20MEDINA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20NATALIA%20GUINDALINI%20MELLONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20NEIDE%20FERREIRA%20MASCARENHAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20PEREIRA%20MENDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20TASCIANA%20DAS%20NEVES%20RAMOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIA%20VICTORIA%20ALONSOPEREZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20ALVES%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20ANCELMO%20BONFIM%20PITA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20ANDOZIA%20MORINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20BARTILOTTI%20GARCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20BASTOS%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20BERNARDO%20DA%20ROCHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20COELHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20CORREIA%20DE%20OLIVEIRA%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20CRISTINA%20GALEANE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20CRUZ%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20DE%20MORAES%20LEME',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20DELGADO%20OLIVEIRA%20ZENERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20LEME%20DE%20CALAIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20MONTAG%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20PEREIRA%20CARRILES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20RIGOTTO%20CORDEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20SCROCHIO%20RUDGE%20FURTADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIANA%20TAISE%20ZERBINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIELA%20FERNANDA%20PAVAN%20AIRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARILENE%20LIMA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARILIA%20JANEIS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARILIA%20LOPES%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARILIA%20PARDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARINA%20ABICHABKI%20PIVATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARINA%20BALDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARINA%20OLIVEIRA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARINA%20SPATTI%20MARINELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARINILSON%20LEONARDO%20FREIRE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIO%20JOSE%20FELIX%20DE%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARIO%20SERGIO%20ALVAREZ%20FIORETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARK%20DRUMMOND%20ADDY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARLON%20DYO%20FUKUDA%20KOGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARLON%20FERNANDES%20DE%20ALCANTARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARLON%20HELDER%20COSTA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARLON%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MARYANNE%20TRAFANI%20DE%20MELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MASARO%20MAEHARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATEUS%20ALBERANI%20FARIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATEUS%20DE%20SOUSA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATEUS%20GERALDO%20SCHIAVETTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATEUS%20KAZUICHI%20YAMAMOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATEUS%20LEONEL%20SOUTO%20ALONSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATEUS%20RODER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATEUS%20SALMAZO%20TAKAKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATEUS%20TONELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20ANGELI%20CASTANHEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20BARCELOS%20BANHARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20BERTOLINO%20BARROS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20CALDERAN%20RUY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20EDUARDO%20JOANA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20FASSIS%20COROCHER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20FELIPE%20CELESTINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20FERRARONI%20SANCHES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20FERREIRA%20MENDONCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20GABARRAO%20SANTANA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20GASPARETTO%20FEITOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20GUILGER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20HENRIQUE%20TOZZI%20GUARITA%20BORGES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20LIMA%20DE%20MELLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20MAGALHAES%20CAMPOS%20DEVIDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20PERESSIM%20PARACAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20SARDELI%20MALHEIROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MATHEUS%20SIQUEIRA%20LOBO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURICIO%20BRAGA%20MEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURICIO%20DORNELLAS%20TABBAL%20CHAMATI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURICIO%20FERNANDO%20LIMA%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURICIO%20FONTANA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURICIO%20JOSE%20RODRIGUES%20PEDROSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURICIO%20LUIZ%20SOBRINHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURICIO%20NORIS%20FREIRE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURICIO%20SANTOS%20PUPO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURILIO%20SAKZENIAN%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURO%20ANTONIO%20GUARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAURO%20PEDROMONICO%20ARRYM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAX%20ROSAN%20DOS%20SANTOS%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAXIMILIANO%20SELMI%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAYARA%20DE%20SOUZA%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAYARA%20HERRMANN%20RUGGIERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAYARA%20MARTINS%20PERRONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAYARA%20RANI%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAYKON%20ROCHA%20SANTANA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MAYLON%20PIRES%20MACEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MEHDI%20KADIVAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MELANY%20ISABEL%20GARCIA%20NATALE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MELINA%20BAFUME',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MELISSA%20CRISTINA%20DO%20ESPIRITO%20SANTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MELISSA%20DA%20CRUZ%20BOTELHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MELISSA%20KELLY%20LIMA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MELISSA%20LIEKO%20SOUZA%20SUENAGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MICAELE%20APARECIDA%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MICHAEL%20PRIETO%20HERNANDEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MICHAEL%20SILVA%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MICHAEL%20WILLIAN%20ROACH',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MICHEL%20VINICIUS%20MATOS%20ROCHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MICHELLE%20CARVALHO%20METANIAS%20HALLACK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MICHELLE%20DE%20LIMA%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MICHELLE%20FERREIRA%20DA%20COSTA%20ABRAHAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MICHELLE%20THOMAZINE%20DO%20SACRAMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MICHELLI%20GOIS%20ROCCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MIGUEL%20ANGEL%20CARDENAS%20RUEDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MIGUEL%20CARLOS%20LEITE%20SIQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MIGUEL%20EDUARDO%20GUTIERREZ%20PAREDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MIGUEL%20GALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MIGUEL%20VICTOR%20CAMPANHA%20MANFREDI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MILENA%20MARTINS%20ROQUE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MILENA%20MISSE%20MIYOSHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MILENA%20RICCO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MILENA%20VEGA%20PEDROSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MILENE%20BIANCHINI%20ANTONIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MILENE%20SERRANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MILLENA%20HUANY%20ALVES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MILTON%20FELIPE%20SOUZA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MILTON%20FERREIRA%20DA%20SILVA%20DIAS%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MILTON%20MARCELINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MIRANI%20DA%20ROCHA%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MIRIAN%20OLIVEIRA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MOABE%20LIMA%20DE%20SOUZA%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MOACIR%20GONCALVES%20TESSARINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MONICA%20DOMINGUES%20DE%20ARRUDA%20CACHONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MONICA%20PALLONI%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MONICA%20ROCIO%20RAMIREZ%20HERNANDEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MONIQUE%20HELEN%20ARNEIRO%20DE%20CARVALHO%20LEITE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MONIQUE%20SAMAAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MONIQUE%20SIMPLICIO%20VIANA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MOTAHAREH%20ABBASI%20SHANBEHBAZARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MURILLO%20GUIMARAES%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MURILLO%20ROMANI%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MURILO%20GIACHINI%20FERRO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MURILO%20HOIAS%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MURILO%20MAGESTE%20DE%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MURILO%20NICOLAU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MURILO%20SANTOS%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MURILO%20TADEU%20RUSCONI%20FURLANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=MURYLLO%20LAGATTA%20DE%20AGUIAR%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NADIA%20MOREIRA%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NADIA%20VALERIO%20POSSIGNOLO%20VITTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NADINE%20APARECIDA%20VICENTINI%20STEFANUTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NAIADE%20CALANCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NAIARA%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NAIRA%20JOSELINA%20DO%20CARMO%20TERRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NAIRA%20VALLE%20DE%20CASTRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NANCI%20SOUTO%20DE%20ASSIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NARA%20LIGIA%20MARTINS%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20AKEMI%20KOHORI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20AYUMI%20GIL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20BROTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20CAMBIAGHI%20ATILIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20CASTRO%20RODRIGUES%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20CRISTINA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20CRISTINA%20MARTINS%20MARCONDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20CRISTINE%20DIAS%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20DE%20ALMEIDA%20BRUNO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20FERNANDA%20BUENO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20FRANCA%20BRANDAO%20DE%20MATOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20GUZELLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20NAIME',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20PELIZARI%20GARCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20PEREIRA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20RAMIRES%20PEDROSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATALIA%20VOLGARINE%20SCARABOTO%20BONFA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATASHA%20MARTINS%20RODRIGUES%20DE%20JESUS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALI%20CORDEIRO%20PINTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20CAMARA%20MUSTAFA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20FERREIRA%20FREGONEZI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20GARDIN%20PESSOA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20JACOMIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20LIMA%20LOCASPI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20LOPES%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20MACIEL%20LEONESSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20NOSSI%20DAVANZO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20PERUSSI%20CALCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20ROTHER%20DA%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20SOARES%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20SOUZA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20URSOLI%20FERREIRA%20BAPTISTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALIA%20YUKIE%20CREPALDI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALY%20ALCAZAR%20AMORIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHALY%20ARIELLA%20CHUHA%20OGATA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHAN%20GOMES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NATHAN%20MATHIAS%20E%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NAYARA%20REZENDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NEAL%20MOKRANE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NELKIS%20DE%20LA%20ORDEN%20MEDINA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NELSON%20KIYOSHI%20SASAKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NESTOR%20WALTER%20TREPODE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NEVIO%20SAVIETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NEYLOR%20WENDEL%20SILVA%20BAGAGI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NICHOLAS%20OZU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NICOLAS%20ALVARENGA%20ZANARDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NICOLAS%20DO%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NICOLLE%20LIBANIO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NIELS%20JØRGEN%20OLESEN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NILISA%20DOS%20SANTOS%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NILTON%20GOMES%20VALENTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NILVAN%20GARCIA%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NIRMAL%20TEJ%20KUMAR%20DIVAKU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NIVEA%20MARIA%20ROCHA%20MACEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NOELLEN%20CAROLINE%20CAVALCANTI%20DE%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NOEMI%20CANDIDO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NOEMIA%20WATANABE%20CHRISTINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NUNO%20ALEXANDRE%20CALDAS%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=NURY%20YULENY%20AROSQUIPA%20YANQUE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ODAILTON%20ALVES%20MACEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OLAVO%20AMORIM%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OLDAIR%20DONIZETI%20LEITE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ORACILDO%20BORDINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ORLANDO%20CARLOS%20CANOAS%20GUIMARAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ORLANDO%20SANCHES%20PADILHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ORLEM%20LIMA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ORLINDO%20WAGNER%20SOARES%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSCAR%20DE%20NUCCI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSCAR%20JAIME%20CICERI%20CORAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSMAR%20SIMOES%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSVALDO%20ADILSON%20DE%20CARVALHO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSVALDO%20CESAR%20PINHEIRO%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSVANDRE%20ALVES%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OSWALDO%20MARINELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OTAVIO%20HENRIQUE%20DOS%20SANTOS%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OTAVIO%20MOREIRA%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OTAVIO%20PEREZ%20PALAMONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=OZIEL%20DE%20QUEIROZ%20MATTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PABLO%20ALEJANDRO%20DE%20ABREU%20URBIZAGASTEGUI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PABLO%20FIGUEIREDO%20AGUILAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PABLO%20JENNER%20PAREDEZ%20ANGELES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PABLO%20TORRES%20CARREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PABLO%20XAVIER%20MELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PALOMA%20BARROS%20DIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PALOMA%20YURI%20DE%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAMELA%20APARECIDA%20MALDANER%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAMELLA%20MOURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAOLA%20TATIANA%20LLERENA%20VALDIVIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20ANDREA%20RIBEIRO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20BUSINARO%20AIELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20CASSIA%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20CRUZ%20BERGAMI%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20DE%20JESUS%20BORGES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20DE%20SOUZA%20SANTOS%20GRILLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20FERNANDES%20DE%20SOUZA%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20HELOISE%20ALVES%20BEZERRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20LIUS%20MELO%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20MIKA%20SAKUGAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20NICOLAU%20SALLES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20PAINS%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20TIEME%20MAEDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20VASCONCELOS%20DE%20PAULA%20DAINESI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20VIEIRA%20ZAMPAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PATRICIA%20YOSHIE%20KISHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAUL%20CHU%20NGUM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAUL%20JOSEPH%20HIDALGO%20FLORES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULA%20DA%20COSTA%20DE%20GOUVEIA%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULA%20DE%20ARAUJO%20SILVERIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULA%20LUMY%20TAKEUCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULA%20REBIERE%20TORTOLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULA%20SA%20FILIZZOLA%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULA%20VERZOLA%20OLIVIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULA%20VILLENA%20REDONDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULA%20ZANIN%20DE%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULINE%20TEIXEIRA%20STELLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20ANTONIO%20PEREIRA%20WENDHAUSEN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20DE%20TARSO%20CORDEIRO%20FRANCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20DRUMMOND%20MURGEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20FERNANDO%20RABELO%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20GUSTAVO%20OLIVEIRA%20DE%20MELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20HENRIQUE%20DA%20SILVA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20HENRIQUE%20DONEGA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20HENRIQUE%20RODRIGUES%20MOLCK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20JOSE%20DE%20CARLO%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20LUIZ%20ZANGRANDE%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20MARCOS%20SIQUEIRA%20BUENO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20PASSOS%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20RICARDO%20DA%20SILVA%20BARROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20RICARDO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20RICARDO%20ROCHA%20VIANNA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20ROBERTO%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20ROBERTO%20DE%20BARCELOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20ROBERTO%20DE%20OLIVEIRA%20BONIFACIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20ROBERTO%20MARCELINO%20DA%20CUNHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20ROGERIO%20NUNES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20ROGERIO%20PINTO%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20ROGERIO%20SCALASSARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20SALVADOR%20BRITTO%20NIGRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20SANTOS%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20SERGIO%20MONZANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20VICTOR%20BERTOLLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PAULO%20VINICIUS%20ALCANFOR%20XIMENES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20AFFONSO%20PIZELLI%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20AUGUSTO%20PALHARES%20BRAGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20COSTA%20FERREIRA%20PEREIRA%20LEITE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20DANTAS%20PALMEIRA%20GUIMARAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20FONSECA%20RAPOSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20GOYA%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20HAYASHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20HENRIQUE%20CORREA%20KIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20HENRIQUE%20CRISP%20MODESTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20HENRIQUE%20DE%20OLIVEIRA%20LOPES%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20HENRIQUE%20DOMICIANO%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20HENRIQUE%20GIRARDI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20HENRIQUE%20LOMNITZER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20HENRIQUE%20MINATEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20HENRIQUE%20OLIVEIRA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20HENRIQUE%20PAVAN%20DE%20MATTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20HENRIQUE%20ZORZANELLI%20DA%20VITORIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20JORGE%20SALDANHA%20RAMOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20LUIZ%20MASCARANHAS%20AMARAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20MESQUITA%20MOURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20MIGUEL%20BUENO%20LEITE%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20MOURA%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20NICOLELLA%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20PAULO%20JUSTINO%20DA%20SILVA%20ARANTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20PORTO%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20RAFAEL%20DOMOTOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20RIVIERE%20TORRADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20SOLA%20PIMENTEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20SOUSA%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20VITOR%20JUSTINO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEDRO%20VITOR%20QUINTA%20DE%20CASTRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PEI%20JEN%20SHIEH',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PERICLES%20SALES%20DURAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PERSING%20JUNIOR%20CARDENAS%20VIVANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PETER%20SIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PETER%20VAN%20HOOGEVEST',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PETERSON%20FRANCISCO%20DA%20COSTA%20SENHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PETERSON%20MORENO%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PHILIP%20AYRES%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PHILIPE%20LABOISSIERE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PIETRO%20GRAGNOLATI%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILA%20APARECIDA%20DE%20CASTRO%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILA%20APARECIDA%20DE%20MORAES%20IORIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILA%20COSTA%20GARRIDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILA%20DA%20SILVA%20DALVECHIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILA%20DA%20SILVA%20IDEYAMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILA%20DE%20QUEIROZ%20SOUZA%20PASSOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILA%20FERNANDA%20ARRUDA%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILA%20GONCALVES%20COUTO%20DE%20ALEXANDRIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILA%20LIRA%20DE%20MEDEIROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILA%20LOPES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILA%20MORETTI%20CAPELLATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=PRISCILLA%20DE%20ABREU%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RACHEL%20MIANA%20BEZERRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RACHEL%20TEMPERANI%20AMARAL%20MACHADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RADEMAKS%20BENTO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20ABILIO%20PUBLIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20ALBERTO%20MARAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20ALCANTARA%20DA%20ROCHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20ALVARES%20FRANCO%20NEVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20ALVES%20CAMILLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20APARECIDO%20GRANZOTTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20ARTHUR%20ROCHA%20MIRANDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20AVILA%20DE%20ESPINDOLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20BARROS%20NAVARRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20BESSI%20CONSTANTINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20CAMPOS%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20CASONATO%20ZOCCOLER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20DA%20SILVA%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20DE%20AQUINO%20CUNHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20DE%20CAMARGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20DE%20CAMARGO%20VAZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20DE%20MATOS%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20DE%20SOUZA%20TOLEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20DE%20TOLEDO%20CINTRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20DERRADI%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20DOS%20SANTOS%20ELIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20FARAONE%20RANDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20FARINACCIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20FELIPE%20SANDRONI%20DIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20FERNANDES%20GODINHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20FERNANDES%20PIGNOLI%20BENZI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20GARCIA%20CERQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20GIARETA%20FALCARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20GIUSTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20GUIMARAES%20PEQUENO%20FRANCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20IDALGO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20LAGE%20TAVARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20LOPES%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20MATEUS%20DA%20SILVA%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20MATHEUS%20NUNES%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20MITIO%20KUSHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20NANYA%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20PASCHOAL%20BRAGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20PRAVATTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20RAMOS%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20RIBEIRO%20LIRA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20ROQUE%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20SGOBBE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20SILVA%20BERTOLINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20SIMOES%20CELEGATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20SOARES%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20VIRGINIO%20MONTEIRO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAEL%20WILLWOHL%20SALGADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAELA%20ADLE%20GOMES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAELA%20BERTOLOTO%20VARGAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAELA%20CAROLINA%20CONSTANTINO%20ROMA%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAELA%20CATARINA%20DE%20OLIVEIRA%20RAMOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAELA%20CRISTINE%20ALVES%20MARCAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAELA%20MUNIZ%20TONETTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAELA%20REGINA%20FANTATTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAELA%20ZANCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAELLA%20LUPPI%20SEVERO%20E%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAELLA%20ZANETTI%20DIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAFAELY%20CAROLINA%20DA%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAIMUNDO%20ANCHIETA%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAINARA%20MORENO%20SANCHES%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAISSA%20YOLANDA%20DE%20OLIVEIRA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAMON%20CHIARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAMON%20GUSTAVO%20SANCHES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RANIERI%20DE%20CARVALHO%20PEREIRA%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAPHAEL%20ALEXANDER%20CINTRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAPHAEL%20ALEXANDRE%20MARIANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAPHAEL%20CHIUSO%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAPHAEL%20MONTALI%20DA%20ASSUMPCAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAPHAEL%20MORAL%20PIAZERA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAPHAEL%20SEDANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAPHAELA%20DE%20CARVALHO%20LOURENCO%20SIGNORINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAPHAELA%20DELL%20ACQUA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAQUEL%20ANDRADE%20LEITE%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAQUEL%20DALLA%20LANA%20CARDOSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAQUEL%20RIBEIRO%20SANTOLIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAQUEL%20RINKE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAQUEL%20SANTANA%20DA%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAQUEL%20SARMENTO%20MATEUS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAQUEL%20VETURIANO%20BARROSO%20ORBOLATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAUL%20BARBOSA%20MARTINS%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAUL%20DE%20FRANCA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAUL%20HABESCH',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAUL%20MURETE%20DE%20CASTRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAUL%20PRIMON%20DE%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RAUL%20SANTOS%20IGLESIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=REBEKA%20GOMES%20PINTO%20CUNHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=REGINA%20QUEIROZ%20MACHTURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=REGINALDO%20JOSE%20DA%20SILVA%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=REINALDO%20LIMA%20DE%20ABREU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=REMCO%20JONATHAN%20VAN%20DASSELAAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENAN%20CANDIDO%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENAN%20DO%20NASCIMENTO%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENAN%20GALEANE%20ALBOY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENAN%20OSCAR%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENAN%20PEREIRA%20BIAZINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENAN%20PRATES%20LOPES%20DE%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20BARBOSA%20SHIMOCOMAQUI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20CAROLINA%20BARREIRO%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20FATIMA%20PIVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20GENOVA%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20GIMENES%20LEITE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20MANZANO%20MARIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20MAYUMI%20GOUVEA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20RAMISCH',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20RIBEIRO%20SPITZER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20TIMM%20DA%20SILVA%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATA%20TOMAZ%20QUEVEDO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATO%20ALVES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATO%20ALVES%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATO%20BORELI%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATO%20CADECARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATO%20DE%20FREITAS%20BULCAO%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATO%20FERNANDES%20CANTAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATO%20KOSAKA%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATO%20MAFFEI%20BASTOS%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENATO%20TADAYOSHI%20HIRAKAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENE%20ADUAN%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENE%20LOPES%20DE%20BARROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RENI%20WILIYANTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=REYNALDO%20TRONCO%20GASPARINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=REYNIER%20HERNANDEZ%20TORRES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RHANI%20DUCATTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RHUAN%20DE%20CASTRO%20CORREIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20ANTONIO%20ZANETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20AUGUSTO%20OKIMOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20AUGUSTO%20REIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20BALDASSIN%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20BANHOS%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20BRIGATO%20SCHEICHER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20CAMILO%20GALAVOTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20CESAR%20CAMARA%20FERRARI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20CESAR%20CORREA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20CODINHOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20DE%20CAMPOS%20BULL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20DE%20CASSIO%20BARBOSA%20CARDOSO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20DE%20OLIVEIRA%20BICUDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20FRANCESCHINI%20OLIANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20FRANCISCO%20CABANAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20FROEHLICH',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20GALVAO%20GUIMARAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20GONCALVES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20HENRIQUE%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20JOHANSON%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20LUIZ%20LAZAREK%20VENTURINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20MARCOLINO%20DE%20SALES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20MARTINS%20FILGUEIRAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20MAURICIO%20CASPIRRO%20ARGEU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20MONDELLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20PIRES%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20PONTES%20BONFIGLIOLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20PORTELA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20ROBERTO%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20SCUCUGLIA%20RODRIGUES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20SENA%20GOMES%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20VICENTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICARDO%20WOISKY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICHARD%20PARCIASEPE%20MASCARIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICHARD%20RAFAEL%20SANT%20ANNA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RICHARLLA%20APARECIDA%20BUSCARIOL%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RINALDO%20MIRANDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RITA%20APARECIDA%20NICIOLI%20CERIONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RITA%20FABIANA%20ADAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RITA%20HELENA%20BUSO%20JACON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTA%20ALBINO%20DOS%20REIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTA%20APARECIDA%20REAL%20SUEROZ%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTA%20BENEDITA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTA%20HEHL%20DE%20SYLOS%20CINTRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTA%20MIWA%20CALDART%20SASAHARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTA%20MUCIACITO%20BOTTER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTA%20RODRIGUES%20CRUVINEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20ALIPIO%20BUSSOLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20ALVES%20GUELERI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20BARRETTO%20DIAS%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20GONCALVES%20DE%20MAGALHAES%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20KENJI%20HIRAMATSU',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20LAUDARI%20NEVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20LOPES%20PARRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20LUCAS%20FRANCA%20FALCAO%20CAMPOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20MARCHI%20GOULART',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20MARIA%20SCHELTINGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20MASCARENHAS%20BRAGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBERTO%20PORTO%20BENATTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBSON%20EISINGER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROBSON%20LUIZ%20FERNANDES%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODNEI%20IARTELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODOLFO%20DO%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODOLFO%20GABRIEL%20PIERRE%20MARCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODOLFO%20MARTINS%20DAVID%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODOLFO%20SOUZA%20SCOTOLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20ADLER%20DE%20LIMA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20ALVES%20MENDONCA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20AMARAL%20LAPA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20ANTONIO%20FACCIOLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20APARECIDO%20XAVIER%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20ASSAF',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20BOER%20FERRAREZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20CARMO%20TERIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20CARVALHO%20REZENDE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20CASSIARI%20MARTINHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20CESAR%20ANTONIALLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20CLEIR%20CASTELLON%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20COSTA%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20DE%20ARAUJO%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20DE%20CASTRO%20BORGES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20DIAS%20SAMOES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20DONIZETE%20SANTANA%20DE%20PADUA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20DUARTE%20PECHONERI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20EDUARDO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20ELIAS%20BIANCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20GARCIA%20LOPES%20INOCENTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20GOUVEA%20MOURAD',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20GRASSI%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20GUERATO%20SIQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20HERNANDEZ%20DE%20LUZIA%20GURDOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20LANNA%20FRANCO%20DA%20SILVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20LUIS%20COLLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20MARTINS%20ROMEIRA%20SAKAI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20MATTIAZO%20ROSOLINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20MEDEIROS%20TEODORO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20MOLOGNI%20GONCALVES%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20MURARO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20NAGAO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20NUNES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20OLIVEIRA%20CASTELLOES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20PASTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20PEREIRA%20BARRETTO%20DA%20COSTA-FELIX',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20PEREIRA%20TAVARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20QUERINO%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20SCHIMIDT%20ALVES%20PEIXOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20SENDIN%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20STOCKLER%20TOGNETTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20VIVIANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RODRIGO%20YOSHIAKI%20GOTO%20KATAHIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROGER%20FREDY%20LARICO%20CHAVEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROGER%20NOBUYUKI%20KAMOI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROGER%20TETSUO%20DE%20CASTRO%20TOYAMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROGER%20WILSON%20CANDIDO%20ROCHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROGERIO%20BARBOSA%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROGERIO%20COLLA%20HEROS%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROGERIO%20COSTA%20BARBOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROGERIO%20LEMBO%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROGERIO%20LUCAS%20DOS%20SANTOS%20ROSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROGERIO%20RODRIGUES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROGERIO%20SPURAS%20WERNECK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROMAN%20SPIRIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROMEO%20BULLA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROMEU%20GADOTTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROMINA%20PAULA%20YAMAMOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RONALDO%20ALESSANDER%20VIERA%20NUNEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RONALDO%20CARVALHO%20MOURA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RONALDO%20EPIFANIO%20DE%20OLIVEIRA%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RONALDO%20GONCALVES%20LEITE%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RONALDO%20LUIZ%20PEREIRA%20DE%20MACEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RONALDO%20MIRANDA%20PINTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RONALDO%20QUEIROZ%20MACEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RONEI%20DOS%20SANTOS%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RONEY%20BELHASSOF',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RONI%20RUIZ%20DURAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RONY%20FIGUEIREDO%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROQUE%20ENRIQUE%20LOPEZ%20CONDORI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROSANA%20VERONEZE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROSANGELA%20DE%20FATIMA%20PUGINA%20DE%20PONTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROSARIO%20EVA%20MONTES%20NIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROSE%20MARY%20DO%20PRADO%20DEMORI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROSELI%20APARECIDA%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROSEMEIRE%20NATSUKO%20SHOJI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ROSIANE%20CORREIA%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RUBENS%20DE%20ANDRADE%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RUBENS%20GONDEK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RUBENS%20MONTEIRO%20LUCIANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RUBIA%20MUNHOZ%20RAPELLI%20SENA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RUBIA%20PIMENTEL%20CERQUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RUDYARD%20ARIAN%20SARQUIS%20PINTO%20BARRETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RUSLAINE%20BALIZA%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=RUTH%20DEL%20RASO%20GARCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SABRINA%20AMANDA%20CORDEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SABRINA%20BERGOCH%20MONTEIRO%20SAMBATI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SABRINA%20CLETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SABRINA%20HELENA%20BERNARDI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SABRINA%20SANTOS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SABRINA%20SAYORI%20OKADA%20MATINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMANTA%20FERREIRA%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMANTA%20GISELI%20BODAS%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMANTA%20RAFAELA%20DE%20OMENA%20LOSKE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMANTHA%20FAIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMARA%20DE%20SOUSA%20MARIANO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMARA%20FEBRONIO%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMARA%20ROSANE%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMIRA%20POLEZI%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMUEL%20AGUIAR%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMUEL%20COVAS%20SALOMAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMUEL%20REGHIM%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMUEL%20RIMOLDI%20GUELLIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMUEL%20SIQUEIRA%20DO%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAMUEL%20ZANFERDINI%20OLIVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANDRA%20APARECIDA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANDRA%20SAYURI%20NAKAGAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANDRA%20SUMIE%20GONCALVES%20TAKEBAYASHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANDRO%20BROSCO%20SAKATA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANDRO%20CORDEIRO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANDRO%20COSTA%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANDRO%20DO%20NASCIMENTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANDRO%20GONCALES%20FUNK',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANDRO%20LUIS%20TEIXEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANDRO%20LUIS%20VATANABE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANDRO%20SILVA%20FERREIRA%20PINTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANTIAGO%20DAVID%20DAVILA%20BENAVIDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANTIAGO%20KRAISELBURD',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SANTIAGO%20VALDES%20RAVELO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SARA%20DA%20COSTA%20PEREIRA%20BUENO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SARA%20REDA%20MANSOUR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SARA%20RODRIGUEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SARA%20SUAREZ%20MARGARIDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SARAH%20CAROLINA%20FURUCHO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SARAH%20FRANK%20ROSSNER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SAULO%20DUARTE%20OZELIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SEBASTIANA%20MOREIRA%20ARAUJO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SEBASTIAO%20SANTIAGO%20BARRETTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20FERREIRA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20HENRIQUE%20LIMONE%20BROCCHETTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20HENRIQUE%20MORAES%20DURAND',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20HENRIQUE%20TROFINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20LEME%20DE%20GODOY%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20LUIS%20RABELO%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20LUIZ%20BRUNO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20MENOCHELLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20RICARDO%20GODINHO%20SALAZAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20SHIGUERO%20TUSTUMI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20SILVA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SERGIO%20TSUYOSHI%20GOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SHERMILA%20GUERRA%20SANTA%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SHIRLEY%20LIMA%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SIEGFRIED%20EUGEN%20RAYER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILMARA%20APARECIDA%20FAGGION',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILMARA%20RICHTER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILVANA%20FERREIRA%20DE%20QUEIROZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILVIA%20CRISTINA%20DIAS%20PINTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILVIA%20HELENA%20MOREIRA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILVIA%20SERRA%20NEGRA%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILVIA%20VASCONCELOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SILVIO%20DA%20COL%20DE%20BRITO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SIMEIA%20RAFAELA%20NUNES%20CASATI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SIMONE%20DOMINGUES%20FRANCISCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SIMONE%20MARISSOL%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SIMONE%20SANTOS%20CASOLLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SIMONE%20TATIANE%20DO%20CANTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SIMONY%20CAMARGO%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SINARA%20GABRIELA%20BUENO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SIOMEL%20SAVIO%20ODRIOZOLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SIYAMAK%20BRAKHASBUKANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SOLANGE%20MARIA%20LONGHITANO%20CARBONEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SONIA%20CASTELO%20QUISPE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SOPHIA%20BEATRIZ%20PEREIRA%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SOPHIA%20MARCAL%20SORREGOTTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=STEFANY%20MIRRELLE%20FAVERO%20ZUZE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=STELA%20DELAGRACIA%20ARRIGHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=STELVIO%20HENRIQUE%20IGNACIO%20BARBOZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=STEPHAN%20REIFF-MARGANIEC',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=STEPHANIE%20BELAZI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=STEPHANIE%20TOME%20LOBOZZO%20DOWER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=STEPHANNIE%20CAROLINE%20ESPER%20MAZIERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=STEVAN%20RODRIGUES%20MANZAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=STIVERSON%20STOPA%20ASSIS%20PALMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SUELINO%20GABRIEL%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SUENEY%20SANTOS%20ABBIATI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SUZAN%20ALINE%20CASARIN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SUZANA%20COIMBRA%20DE%20MOURA%20LUSTOSA%20E%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SUZANE%20ANDREA%20REOLON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SUZELAINE%20APARECIDA%20TOMAZI%20NIJENHUIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SWENEY%20TEIXEIRA%20MONTEVECHIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=SYLVIA%20PATRICIA%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TABATA%20SANTOS%20DELGADO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TABATHA%20NOVIKOV%20DO%20AMARAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TACIANA%20TONETTO%20CASTELO%20BRANCO%20TRIGO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TADEU%20PEREIRA%20PASSOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TAINA%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TAISA%20PAVANI%20GOMES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TALITA%20ARIELA%20SAMPAIO%20E%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TALITA%20BARBEDO%20AMORIM%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TALITA%20BERNARDINO%20LI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TALITA%20CRISTINA%20MENA%20SEGATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TALITA%20ROBERTA%20BRUNHARA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TALITHA%20BERTAZZO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TALLES%20VIANA%20VARGAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TALLITA%20EDUARDA%20DA%20VEIGA%20REIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TAMARA%20CARLI%20MOTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TAMARA%20GUINDO%20MESSIAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TAMIE%20GUIBU%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TAMIRES%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TAMIRES%20LEME%20LAMEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TAMIRIS%20MARTINS%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TANIA%20MARGARETH%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TARCIA%20DO%20VALE%20E%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TARSILA%20DE%20CONTI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TATIANA%20BENDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TATIANA%20COMPORTE%20STABELINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TATIANA%20LOPES%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TATIANE%20MARISIS%20GIOVANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TAYNA%20DE%20OLIVEIRA%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TAYNA%20NEGRI%20KUHN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TELMA%20CAROLINA%20CLIRISOSTOMO%20FRANCISCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TELMA%20CRISTINA%20SAITO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TERESINHA%20MARIA%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THADEU%20ANTONIO%20FERREIRA%20DE%20MELO%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIANE%20GAMBARRA%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAILLINE%20SILVA%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAINA%20FERNANDA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20ALMEIDA%20VALVASSOURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20BUENO%20DE%20TOLEDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20CANDIDO%20ROBERTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20DE%20SOUZA%20SOARES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20EMANUELE%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20FERREIRA%20ISABEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20HELENA%20DOS%20SANTOS%20ROCHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20HELENA%20MODERNO%20AUGUSTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20HELENA%20OLIVEIRA%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20LEITE%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20LINO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20LUGLI%20GONCALES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20MAYUMI%20KAIMOTI%20MANFRIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20MEGUMI%20SATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20ROCHA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20SILVA%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAIS%20SIQUEIRA%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THALITA%20CAETANO%20PAIVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THALITA%20DANIELA%20SCAPIM',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THALITA%20MARQUES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THALITA%20ROCHA%20SACILOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THALLES%20ROSSATO%20VAZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THALYTA%20POTENZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAMINE%20THAADA%20YAMADA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAMIRES%20ANDRESSA%20MARCELINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAYNA%20CHAVES%20E%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THAYNA%20DE%20SOUZA%20BRAZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THEO%20ARENA%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20ALVAREZ%20BERNARDEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20BERARDINELLI%20VIEIRA%20BRAZ%20GONCALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20BRITO%20GUERREIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20CALICHIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20CARLOS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20FERREIRA%20COVOES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20HENRIQUE%20CORAINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20HENRIQUE%20DE%20PAULO%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20HENRIQUE%20NETO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20INACIO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20LINS%20VANDERLEY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20MORAES%20STEFANI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20MOURA%20WITT',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20NICOLA%20CAJUELA%20GARCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20PEREIRA%20RICCIARDI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20PINHEIRO%20FELIX%20DA%20SILVA%20E%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20ROGERIO%20SCHMIDT%20PASTRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20ROSSENER%20NOGUEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20ROSSI%20SILVESTRINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20SERRA%20AZEVEDO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20SILVESTRE%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20TOGNOLI%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THIAGO%20VIEIRA%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THOMAS%20VON%20ZUBEN%20PACCHI%20CAPOVILLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=THOMAZ%20BOER%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20ALEXANDRE%20COCIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20ANDRADE%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20AUGUSTO%20DA%20CRUZ%20MAZZON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20AZEVEDO%20DUARTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20BARABASZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20CUNHA%20BUENO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20DE%20PAULA%20FERNANDES%20ESTOPA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20DE%20SOUZA%20MORAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20DOS%20SANTOS%20SOTERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20HENRIQUE%20DOS%20SANTOS%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20KESSLER%20FIGUEIREDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20KINUST%20BIAGE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20LARA%20MICHELIN%20SANCHES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20LUIZ%20DEBONZI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20PICON',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20ROGERIO%20DE%20CARVALHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20SANTOS%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20TAKASHI%20HONDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TIAGO%20VIGER%20ORTIZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TICIANE%20RIBEIRO%20RODRIGUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TOMAS%20POWELL%20VILLENA%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TOMAZ%20MARQUES%20GONCALVES%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TOMAZ%20WALTER',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=TONY%20SADAHITO%20CHINA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=UBALDO%20BANOS%20RODRIGUEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=UBIRAJARA%20PACHECO%20MALTEZ%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=UBIRATAN%20ALMEIDA%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=UDO%20TERSIANO%20SKIELKA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VAGNER%20DE%20SOUZA%20SERIKAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VAGNER%20VIRCHES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VAGNER%20ZEIZER%20CARVALHO%20PAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VALDETE%20DE%20LIMA%20ANK%20MORAIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VALERIA%20DE%20ALEM%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VALERIA%20LONGO%20PARSEKIAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VALERIO%20CITTADINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VALESCA%20VILELA%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VALIANA%20ALVES%20TEODORO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VALMIR%20CARLOS%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VALQUIRIA%20RISSATO%20GAZOLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VALTER%20ORICO%20DE%20MATOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VALTER%20SANTIAGO%20ROSA%20FILHO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VANESCA%20DE%20SOUZA%20LINO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VANESSA%20APARECIDA%20BATISTA%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VANESSA%20APARECIDA%20CRUZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VANESSA%20ARAUJO%20BORGES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VANESSA%20CAROLINE%20LOPES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VANESSA%20DE%20SOUZA%20TOTOLI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VANESSA%20LETICIA%20GALLO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VANESSA%20NASCIMENTO%20SOUSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VANESSA%20SANTOS%20CARAMORI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VASCO%20THOMAZ%20ZAMBON%20MAZIERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VERA%20LUCIA%20REIS%20DE%20GOUVEIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VERA%20NAGAMUTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VERENA%20EDUARDA%20ZANIBONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VERUSCA%20SEMMLER%20ROSSI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VERUSKA%20RODRIGUES%20MOREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20ANDRIETTA%20RAZOLI%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20AUGUSTO%20BALDI%20DE%20ALMEIDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20DE%20OLIVEIRA%20VENITES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20DE%20SOUZA%20DIAS%20CARVALHO%20GOULART',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20GASPAROTTO%20CAPONE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20HUGO%20CAMPOS%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20HUGO%20GARCIA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20HUGO%20SANTIAGO%20COSTA%20PINTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20HUGO%20SILVA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20MIRANDA%20GONCALVES%20JATOBA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20OLIVEIRA%20COLACELLI%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20REZENDE%20GERALDINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20SIQUEIRA%20FORTES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VICTOR%20TROTTMANN%20CORREA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIDAL%20BARRON%20LOPEZ%20DE%20TORRE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VILMARIA%20APARECIDA%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20ANCHESCHI%20STRINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20BATISTA%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20DA%20SILVA%20PUENTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20DE%20ARAUJO%20MAEDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20DE%20OLIVEIRA%20CASTRO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20DE%20OLIVEIRA%20CIVALI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20DOS%20SANTOS%20AGUIAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20DUARTE%20SEGISMUNDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20FERNANDES%20MOREIRA%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20FERNANDO%20DE%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20FERRUCCI%20TAVEIROS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20GALLUCCIO%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20HEITOR%20FRASSE%20DE%20PADUA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20LIMA%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20LUIZ%20RIBEIRO%20TEODORO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20MATOSO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20MOREIRA%20VIDOTTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20PAULO%20LOPES%20DE%20OLIVEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20PIZZO%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20RIBEIRO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20SANDRINI%20VECCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINICIUS%20TOHORU%20YOSHIURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VINY%20CESAR%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIRGINIA%20CLAUDIA%20PINELI%20ALVES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITAL%20CRUVINEL%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITHOR%20ANTONIO%20BARSALOBRE%20DE%20FREITAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITHORIA%20CAROLYNA%20TRINDADE%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20DOS%20SANTOS%20OLIVEIRA%20NOVO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20EDUARDO%20NARCISO%20DOS%20REIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20GARCIA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20GRIPA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20HISSAYOSHI%20KATO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20HUGO%20ALMEIDA%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20HUGO%20DA%20MOTTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20LEAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20LUIZ%20INNOCENTINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20LUIZ%20ZANDONADI%20VIEIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20MARCHINI%20ROLISOLA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20MATHIAS%20MUNERATTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20PUCCIARELLI%20ANTLOGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20QUEIROZ%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20RODRIGUES%20SOUTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITOR%20TARNOSCHI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITORIA%20DUARTE%20BORGONOVI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VITORIA%20MARIA%20DO%20PRADO%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIVIAN%20CRISTINA%20PIETROBON%20ORSOLINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIVIAN%20FREITAS%20AGUIAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIVIAN%20MARIA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIVIAN%20MAURA%20GREGORACCI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIVIANA%20STEPHANIE%20COSTA%20GAGOSIAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIVIANA%20VANESA%20URBINA%20GUERRERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIVIANE%20DA%20CUNHA%20CALIXTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIVIANE%20FERNANDES%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIVIANE%20PENITENTE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIVIANE%20RODRIGUES%20LEAL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VIVIANE%20YOSHIMI%20EGAWA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=VOSMARLINE%20GRAZIELA%20ROCHA%20LIMA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WAGNER%20AGUENA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WAGNER%20ALVES%20RIBEIRO%20NOVAES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WAGNER%20CORREA%20RAMOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WAGNER%20COSTA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WAGNER%20GUNTHER%20MONTERO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WALDIR%20BELINAZZI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WALDIR%20EDUARDO%20SIMIONI%20PEREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WALDO%20GONZALO%20CANCINO%20TICONA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WALKER%20SOARES%20DRUMOND',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WALTER%20ALEXANDRE%20BLOIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WALTER%20AUGUSTO%20PEREZ%20CASAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WALTER%20MALDONADO%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WALTER%20SERGIO%20SHIMADA%20OROSCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WALTER%20VINICIUS%20CALLIPO%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WAMBERTO%20ALEXANDRE%20VANZO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WANDER%20WAGNER%20MENDES%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WASHINGTON%20LUIS%20RIBEIRO%20DE%20CARVALHO%20SEGUNDO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WASHINGTON%20LUIZ%20OLIVATO%20ASSAGRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WELLINGTON%20ALTOMAR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WELLINGTON%20MIRANDA%20BARBOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WELSON%20ROBERTO%20COSTA%20JUNIOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WENDY%20PAULA%20DE%20LIRA%20FREITAS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WERNER%20ALFINITO%20FEIO%20DE%20MOURA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WESLEY%20JUSTINO%20MAGNABOSCO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WESLEY%20SOUZA%20DOS%20SANTOS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WEVISON%20RAMALHO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILIAM%20PEREIRA%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAM%20ALECSANDER%20FERNANDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAM%20BARBOSA%20CORSINI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAM%20DE%20ALMEIDA%20RIBEIRO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAM%20FERREIRA%20BARBOSA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAM%20KOJI%20YONAMINE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAM%20LARA%20DE%20OLIVEIRA%20REIS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAM%20SANCHEZ%20FARFAN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAN%20DE%20SOUZA%20BERNARDES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAN%20FELICIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAN%20SADAITI%20MURAMOTO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAN%20SILVA%20MARIANO%20DE%20SOUZA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAN%20TADEU%20BELTRAO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILLIAN%20TAKESHI%20KOMADA%20NOBREGA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILSON%20AUGUSTO%20LIMA%20VENANCIO',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILSON%20JOSE%20DE%20SA%20MARQUES',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WILSON%20RODRIGO%20PATRICIO%20FERREIRA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WINDERSON%20WOLF%20BENICIO%20DA%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=WOLODYMIR%20BORUSZEWSKI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YANISLEIDYS%20HERNANDEZ%20BERMUDEZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YARA%20LEMOS%20DE%20PAULA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YARA%20SBROLIN%20ROLDAO%20SBORDONI',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YOAV%20STEINMETZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YONA%20BENEDITO%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YOUKOU%20RICARDO%20OHY',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YUBIS%20PEREIRA%20MARTINS',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YUMI%20DAIARA%20CAVALCANTI%20SILVA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YUPANQUI%20JULHO%20MUNOZ',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YURANY%20CAMACHO%20ARDILA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YURI%20ADRIEL%20FACTOR',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YURI%20AJALA%20DA%20COSTA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YURI%20ALEXANDRE%20PEPE%20DE%20ANDRADE',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YURI%20DEL%20VIGNA%20YASUDA',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YURI%20VINICIUS%20DE%20JESUS%20PIMENTEL',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=YVES%20EMMANUEL%20DERRIEN',
#         'https://bv.fapesp.br/pt/metapesquisa/?q=ZULEMA%20NETTO%20FIGUEIREDO'
#     ]

# # Código para o link do lattes dos pesquisadores:

#     def parse(self, response):
#         for projeto in response.xpath('*//div[@class="w66"]'):
#             yield{
#                 'Nome': projeto.xpath('*//h1/text()').get(),
#                 'CV Lattes': projeto.xpath('*//h2/a/@onclick').get()
#             }


### 2.1.4. Download dos curriculos em XML.
Os links dos curriculos Lattes obtidos através do método de web scraping foram acessados e cada curriculo foi baixado em um arquivo ZIP que continha um arquivo XML. Para realizar a extração em massa dos arquivos ZIP foi utilizado o código abaixo:

In [ ]:
# @title
# # @title
# import os
# from zipfile import ZipFile

# outpath = 'C:/Users/rapha/OneDrive/Desktop/Lattes/curriculos_xml'

# pasta = 'C:/Users/rapha/OneDrive/Desktop/Lattes/curriculos'
# for diretorio, subpastas, arquivos in os.walk(pasta):
#     for arquivo in arquivos:
#         ziparq = os.path.join(diretorio, arquivo)
#         z = ZipFile(ziparq, 'r')
#         z.extract('curriculo.xml', outpath)
#         os.rename('C:/Users/rapha/OneDrive/Desktop/Lattes/curriculos_xml/curriculo.xml','C:/Users/rapha/OneDrive/Desktop/Lattes/curriculos_xml/' + arquivo +".xml")
#         z.close()

### 2.1.5. Extração dos dados do XML para um CSV.
O curriculo Lattes possui um quantidade significativa de dados, e por isso foi utilizada a biblioteca [ElementTree](https://docs.python.org/3/library/xml.etree.elementtree.html). Os dados de todos os currículos foram armazenados em um único arquivo CSV. \
[DADOS_LATTES_PESQUISADORES.csv](https://drive.google.com/file/d/1B_k8JshmZhBK0p9jgGjlQYd6ZvXEmhSI/view?usp=sharing)

In [ ]:
# @title
# # @title
# import pandas as pd
# from glob import glob
# import xml.etree.ElementTree as ET
# import os


# data = {'NOME-COMPLETO': ['Teste'],
#         'CIDADE-NASCIMENTO': ['Teste'],
#         'UF-NASCIMENTO': ['Teste'],
#         'PAIS-DE-NASCIMENTO': ['Teste'],
#         'ENDERECO-PROFISSIONAL-CIDADE': ['Teste'],
#         'ENDERECO-PROFISSIONAL-BAIRRO': ['Teste'],
#         'ENDERECO-PROFISSIONAL-UF': ['Teste'],
#         'GRADUACAO-NOME-INSTITUICAO': ['Teste'],
#         'GRADUACAO-CODIGO-INSTITUICAO': ['Teste'],
#         'MESTRADO-NOME-INSTITUICAO': ['Teste'],
#         'MESTRADO-CODIGO-INSTITUICAO': ['Teste'],
#         'DOUTORADO-NOME-INSTITUICAO': ['Teste'],
#         'DOUTORADO-CODIGO-INSTITUICAO': ['Teste'],
#         'POS-DOUTORADO-NOME-INSTITUICAO': ['Teste'],
#         'POS-DOUTORADO-CODIGO-INSTITUICAO': ['Teste'],
#         }

# data_df = pd.DataFrame(data)


# pasta = 'C:/Users/rapha/OneDrive/Desktop/Lattes/curriculos_xml'

# for diretorio, subpastas, arquivos in os.walk(pasta):
#     for arquivo in arquivos:
#         ziparq = os.path.join(diretorio, arquivo)
#         print(ziparq)
#         tree = ET.parse(ziparq)
#         root = tree.getroot()

#         for child in root.iter('DADOS-GERAIS'):
#             NOME_COMPLETO = child.attrib['NOME-COMPLETO']
#             CIDADE_NASCIMENTO = child.attrib['CIDADE-NASCIMENTO']
#             UF_NASCIMENTO = child.attrib['UF-NASCIMENTO']
#             PAIS_DE_NASCIMENTO = child.attrib['PAIS-DE-NASCIMENTO']

#         if NOME_COMPLETO == False:
#             NOME_COMPLETO = ""

#         if CIDADE_NASCIMENTO == False:
#             CIDADE_NASCIMENTO = ""

#         if UF_NASCIMENTO == False:
#             UF_NASCIMENTO = ""

#         if PAIS_DE_NASCIMENTO == False:
#             PAIS_DE_NASCIMENTO = ""

#         for child in root.iter('ENDERECO-PROFISSIONAL'):
#             ENDERECO_PROFISSIONAL_CIDADE = child.attrib['CIDADE']
#             ENDERECO_PROFISSIONAL_BAIRRO = child.attrib['BAIRRO']
#             ENDERECO_PROFISSIONAL_UF = child.attrib['UF']

#         if ENDERECO_PROFISSIONAL_CIDADE == False:
#             ENDERECO_PROFISSIONAL_CIDADE = ""

#         if ENDERECO_PROFISSIONAL_BAIRRO == False:
#             ENDERECO_PROFISSIONAL_BAIRRO = ""

#         if ENDERECO_PROFISSIONAL_UF == False:
#             ENDERECO_PROFISSIONAL_UF = ""

#         for child in root.iter('GRADUACAO'):
#             GRADUACAO_NOME_INSTITUICAO = child.attrib['NOME-INSTITUICAO']
#             GRADUACAO_CODIGO_INSTITUICAO = child.attrib['CODIGO-INSTITUICAO']
#             #GRADUACAO_NOME_CURSO = child.attrib['NOME-CURSO']


#         if GRADUACAO_NOME_INSTITUICAO == False:
#             GRADUACAO_NOME_INSTITUICAO = ""

#         if GRADUACAO_CODIGO_INSTITUICAO == False:
#             GRADUACAO_CODIGO_INSTITUICAO = ""

#         #if GRADUACAO_NOME_CURSO == False:
#         #    GRADUACAO_NOME_CURSO = ""

#         for child in root.iter('MESTRADO'):
#             MESTRADO_NOME_INSTITUICAO = child.attrib['NOME-INSTITUICAO']
#             MESTRADO_CODIGO_INSTITUICAO = child.attrib['CODIGO-INSTITUICAO']

#         if MESTRADO_NOME_INSTITUICAO == False:
#             MESTRADO_NOME_INSTITUICAO = ""

#         if MESTRADO_CODIGO_INSTITUICAO == False:
#             MESTRADO_CODIGO_INSTITUICAO = ""

#         for child in root.iter('DOUTORADO'):
#             DOUTORADO_NOME_INSTITUICAO = child.attrib['NOME-INSTITUICAO']
#             DOUTORADO_CODIGO_INSTITUICAO = child.attrib['CODIGO-INSTITUICAO']

#         if DOUTORADO_NOME_INSTITUICAO == False:
#             DOUTORADO_NOME_INSTITUICAO = ""

#         if DOUTORADO_CODIGO_INSTITUICAO == False:
#             DOUTORADO_CODIGO_INSTITUICAO = ""

#         for child in root.iter('POS-DOUTORADO'):
#             POS_DOUTORADO_NOME_INSTITUICAO = child.attrib['NOME-INSTITUICAO']
#             POS_DOUTORADO_CODIGO_INSTITUICAO = child.attrib['CODIGO-INSTITUICAO']

#         if POS_DOUTORADO_NOME_INSTITUICAO == False:
#             POS_DOUTORADO_NOME_INSTITUICAO = ""

#         if POS_DOUTORADO_CODIGO_INSTITUICAO == False:
#             POS_DOUTORADO_CODIGO_INSTITUICAO = ""

#         pesquisador_df = pd.DataFrame({'NOME-COMPLETO': [NOME_COMPLETO],
#                 'CIDADE-NASCIMENTO': [CIDADE_NASCIMENTO],
#                 'UF-NASCIMENTO': [UF_NASCIMENTO],
#                 'PAIS-DE-NASCIMENTO': [PAIS_DE_NASCIMENTO],
#                 'ENDERECO-PROFISSIONAL-CIDADE': [ENDERECO_PROFISSIONAL_CIDADE],
#                 'GRADUACAO-NOME-INSTITUICAO': [GRADUACAO_NOME_INSTITUICAO],
#                 'GRADUACAO-CODIGO-INSTITUICAO': [GRADUACAO_CODIGO_INSTITUICAO],
#                 'MESTRADO-NOME-INSTITUICAO': [MESTRADO_NOME_INSTITUICAO],
#                 'MESTRADO-CODIGO-INSTITUICAO': [MESTRADO_CODIGO_INSTITUICAO],
#                 'DOUTORADO-NOME-INSTITUICAO': [DOUTORADO_NOME_INSTITUICAO],
#                 'DOUTORADO-CODIGO-INSTITUICAO': [DOUTORADO_CODIGO_INSTITUICAO],
#                 'POS-DOUTORADO-NOME-INSTITUICAO': [POS_DOUTORADO_NOME_INSTITUICAO],
#                 'POS-DOUTORADO-CODIGO-INSTITUICAO': [POS_DOUTORADO_CODIGO_INSTITUICAO],
#                 })

#         data_df.loc[arquivo] = [NOME_COMPLETO, CIDADE_NASCIMENTO, UF_NASCIMENTO, PAIS_DE_NASCIMENTO, ENDERECO_PROFISSIONAL_CIDADE, ENDERECO_PROFISSIONAL_BAIRRO, ENDERECO_PROFISSIONAL_UF, GRADUACAO_NOME_INSTITUICAO, GRADUACAO_CODIGO_INSTITUICAO, MESTRADO_NOME_INSTITUICAO, MESTRADO_CODIGO_INSTITUICAO, DOUTORADO_NOME_INSTITUICAO, DOUTORADO_CODIGO_INSTITUICAO, POS_DOUTORADO_NOME_INSTITUICAO, POS_DOUTORADO_CODIGO_INSTITUICAO]
#         data_df.tail()

# data_df.to_csv('pesquisadores.csv')

# 3. Limpeza e Padronização

## 3.1. `df_projetos_pipe`
A tabela `df_projetos_pipe` foi atualizada para listar uma única entrada de cada projeto, através do filtro da coluna `Linha de Fomento` para valores com a palavra-chave "Auxílio".

In [ ]:
# @title
# ==============================================================================
# 1. ATUALIZAR TABELA DF_PROJETOS_PIPE
# ==============================================================================

# Filtrar df_projetos_pipe pela coluna 'Linha de Fomento' contendo 'Auxílio'
df_projetos_pipe = df_projetos_pipe[df_projetos_pipe['Linha de Fomento'].str.contains('Auxílio', na=False, case=False)]
print(f"Total de projetos com linha de fomento de 'Auxílio' em 'df_projetos_pipe': {len(df_projetos_pipe)}")
#display(df_projetos_pipe.head())

Total de projetos com linha de fomento de 'Auxílio' em 'df_projetos_pipe': 2938


## 3.2. `df_pesquisadores_projetos`
A tabela `df_pesquisadores_projetos` foi criada para listar, em formato longo, todas as entradas de nome dos pesquisadores que participaram de um ou mais projetos, relacionando o nome do pesquisador com a atuação profissional de pesquisa e o número do processo do projeto. Os nomes dos pesquisadores foram coletados na tabela `df_projetos_pipe`, nas colunas `Pesquisador Responsável`, `Pesquisadores Principais`, `Pesquisadores Associados`, e na tabela `df_bolsas_concedidas`, na coluna `Beneficiário` para Pesquisadores em Treinamento Técnico.

In [ ]:
# @title
# ==============================================================================
# 1. CRIAR TABELA PESQUISADORES PROJETOS
# ==============================================================================

# Lista para armazenar todos os pesquisadores, tipos e processos
pesquisadores_data = []

# Mapear todos os pesquisadores

# 1. Pesquisador Responsável (Tipo: PR)
if 'Pesquisador Responsável' in df_projetos_pipe.columns and 'N. Processo' in df_projetos_pipe.columns:
    for index, row in df_projetos_pipe.dropna(subset=['Pesquisador Responsável', 'N. Processo']).iterrows():
        nome = row['Pesquisador Responsável'].strip()
        processo = row['N. Processo'].strip()
        pesquisadores_data.append({'nome': nome, 'tipo': 'PR', 'processo_proj': processo})

# 2. Pesquisadores Principais (Tipo: PP)
if 'Pesquisadores Principais' in df_projetos_pipe.columns and 'N. Processo' in df_projetos_pipe.columns:
    for index, row in df_projetos_pipe.dropna(subset=['Pesquisadores Principais', 'N. Processo']).iterrows():
        celula_nomes = row['Pesquisadores Principais']
        processo = row['N. Processo'].strip()
        # Dividir por vírgula e limpar espaços em branco
        nomes = [nome.strip() for nome in celula_nomes.split(',') if nome.strip()]
        for nome in nomes:
            pesquisadores_data.append({'nome': nome, 'tipo': 'PP', 'processo_proj': processo})

# 3. Pesquisadores Associados (Tipo: PA)
if 'Pesquisadores Associados' in df_projetos_pipe.columns and 'N. Processo' in df_projetos_pipe.columns:
    for index, row in df_projetos_pipe.dropna(subset=['Pesquisadores Associados', 'N. Processo']).iterrows():
        celula_nomes = row['Pesquisadores Associados']
        processo = row['N. Processo'].strip()
        # Dividir por hífen e limpar espaços em branco
        nomes = [nome.strip() for nome in celula_nomes.split('-') if nome.strip()]
        for nome in nomes:
            pesquisadores_data.append({'nome': nome, 'tipo': 'PA', 'processo_proj': processo})


# 4. Beneficiário (Tipo: TT) - Filtrar por 'Bolsa no País - Programa Capacitação - Treinamento Técnico'
if 'Beneficiário' in df_bolsas_concedidas.columns and 'Tipo de Financiamento' in df_bolsas_concedidas.columns and 'Processo Vinculado' in df_bolsas_concedidas.columns:
    df_bolsas_tt = df_bolsas_concedidas[
        df_bolsas_concedidas['Tipo de Financiamento'] == 'Bolsa no País - Programa Capacitação - Treinamento Técnico'
    ].dropna(subset=['Beneficiário', 'Processo-Pai'])
    for index, row in df_bolsas_tt.iterrows():
        nome = row['Beneficiário'].strip()
        processo_original = row['Processo-Pai'].strip()
        # Remover os dois primeiros caracteres do 'Processo Vinculado'
        processo = processo_original[2:] if len(processo_original) >= 2 else processo_original
        pesquisadores_data.append({'nome': nome, 'tipo': 'TT', 'processo_proj': processo})

# Criar o DataFrame final
df_pesquisadores_projetos = pd.DataFrame(pesquisadores_data)

print(f"Tabela 'df_pesquisadores_projetos' - Total de linhas: {len(df_pesquisadores_projetos)}")
print(df_pesquisadores_projetos.dtypes)
print("-" * 60)
#print(df_pesquisadores_projetos.head())

Tabela 'df_pesquisadores_projetos' - Total de linhas: 9060
nome             object
tipo             object
processo_proj    object
dtype: object
------------------------------------------------------------


## 3.3. `df_pesquisadores_unicos`
A tabela `df_pesquisadores_unicos` foi criada a partir da tabela `df_pesquisadores_projetos`, com a finalidade de listar uma única entrada de nome dos pesquisadores que participaram de um ou mais projetos, atribuíndo-lhes uma identidade numérica única.
<br>
```
❗Existe um problema de confiabilidade na geração de ID dos pesquisadores através do nome. De maneira que seria mais adequado fazer a linkagem com o Lattes através de um intermediador, como o scrap realizado no site da Fapesp.
```

In [ ]:
# @title
def padronizar_nome(nome):
    if pd.isna(nome):
        return nome
    nome_upper = str(nome).upper()
    nome_sem_acento = unicodedata.normalize('NFKD', nome_upper).encode('ascii', 'ignore').decode('utf-8')
    return nome_sem_acento.strip()

# ==============================================================================
# 1. CRIAR TABELA PESQUISADORES ÚNICOS
# ==============================================================================

# Aproveitar a tabela df_pesquisadores_projetos.
df_pesquisadores_unicos = pd.DataFrame(df_pesquisadores_projetos)
# Remover colunas 'processo_proj' e 'tipo'.
df_pesquisadores_unicos.drop(columns=['processo_proj'], inplace=True)
df_pesquisadores_unicos.drop(columns=['tipo'], inplace=True)
# Cria uma coluna para o nome padronizado
df_pesquisadores_unicos['nome_padronizado'] = df_pesquisadores_unicos['nome'].apply(padronizar_nome)
# Remover duplicatas com base na coluna 'nome_padronizado'.
df_pesquisadores_unicos.drop_duplicates(subset=['nome_padronizado'], inplace=True)

# Criar um ID único para os pesquisadores
df_pesquisadores_unicos['id'] = range(1, len(df_pesquisadores_unicos) + 1)

#print(f"Total de pesquisadores únicos em 'df_pesquisadores_unicos': {len(df_pesquisadores_unicos)}")

# ==============================================================================
# 2. ATUALIZAR TABELA PESQUISADORES PROJETOS COM COLUNA DE ID
# ==============================================================================

# Verificar a contagem de linhas de df_pesquisadores_projetos antes do merge
original_pesquisadores_data_len = len(df_pesquisadores_projetos)
print(f"\nTotal de entradas em df_pesquisadores_projetos ANTES do merge: {original_pesquisadores_data_len}")

# Aplicar a padronização à coluna 'nome' de df_pesquisadores_projetos
df_pesquisadores_projetos['nome_padronizado'] = df_pesquisadores_projetos['nome'].apply(padronizar_nome)

# Realizar a junção (merge) para atribuir o 'id' de df_pesquisadores_unicos a df_pesquisadores_projetos
# Usamos um left merge para manter todas as linhas de df_pesquisadores_projetos
df_pesquisadores_projetos = df_pesquisadores_projetos.merge(
    df_pesquisadores_unicos, # Usar o DataFrame com nomes padronizados únicos
    left_on='nome_padronizado',
    right_on='nome_padronizado',
    how='left',
    suffixes=('_original', '') # O sufixo '' garante que a nova coluna 'id' não tenha sufixo se não houver conflito
)

#print(f"Total de entradas em df_pesquisadores_projetos APÓS o merge: {len(df_pesquisadores_projetos)}")

print(f"Tabela 'df_pesquisadores_unicos' - Total de linhas: {len(df_pesquisadores_unicos)}")
print(df_pesquisadores_unicos.dtypes)
print("-" * 60)
#display(df_pesquisadores_unicos.head())


Total de entradas em df_pesquisadores_projetos ANTES do merge: 9060
Tabela 'df_pesquisadores_unicos' - Total de linhas: 7004
nome                object
nome_padronizado    object
id                   int64
dtype: object
------------------------------------------------------------


In [ ]:
# @title
# ==============================================================================
# CÓDIGO PARA CHECAGEM DO IMPACTO DA PADRONIZAÇÃO NA CONTAGEM DE PESQUISADORES
# DEVE SER UTILIZADO EM ANTES DE CRIAR TABELA PESQUISADORES ÚNICOS
# ==============================================================================

# --- FUNÇÃO PADRONIZAÇÃO ---

def padronizar_nome(nome):
    if pd.isna(nome):
        return nome
    nome_upper = str(nome).upper()
    nome_sem_acento = unicodedata.normalize('NFKD', nome_upper).encode('ascii', 'ignore').decode('utf-8')
    return nome_sem_acento.strip()

# ==============================================================================
# 1. CRIAR TABELA PESQUISADORES ÚNICOS
# ==============================================================================

# Aproveitar a tabela df_pesquisadores_projetos.
df_pesquisadores_unicos_VF = pd.DataFrame(df_pesquisadores_projetos)
df_pesquisadores_unicos_VF['nome_padronizado'] = df_pesquisadores_unicos_VF['nome'].apply(padronizar_nome)
df_pesquisadores_unicos_1 = pd.DataFrame(df_pesquisadores_unicos_VF)
df_pesquisadores_unicos_2 = pd.DataFrame(df_pesquisadores_unicos_VF)
# Remover duplicatas com base na coluna 'nome'.
df_pesquisadores_unicos_1.drop_duplicates(subset=['nome'], inplace=True)
# Remover duplicatas com base na coluna 'nome'.
df_pesquisadores_unicos_2.drop_duplicates(subset=['nome_padronizado'], inplace=True)

print(f"Total de nomes em pesquisadores únicos 'nome': {len(df_pesquisadores_unicos_1)}")
print(f"Total de nomes em pesquisadores únicos 'nome_padronizado': {len(df_pesquisadores_unicos_2)}")

# ==============================================================================
# 2. IDENTIFICAR AS SOBREPOSIÇÕES (QUEM SÃO OS 204?)
# ==============================================================================

# 1. Encontrar todas as linhas onde o 'nome_padronizado' tem duplicatas
# O keep=False é a mágica: ele traz todas as variações envolvidas na duplicata
filtro_duplicados = df_pesquisadores_unicos_1.duplicated(subset=['nome_padronizado'], keep=False)

# 2. Aplicar o filtro na tabela para criar um novo DataFrame só com os "problemáticos"
df_conflitos = df_pesquisadores_unicos_1[filtro_duplicados].copy()

# 3. Ordenar a tabela pelo nome padronizado para que os nomes parecidos fiquem juntos
df_conflitos.sort_values(by='nome_padronizado', inplace=True)

# --- EXIBIÇÃO ---
print(f"Total de linhas envolvidas em conflitos de padronização: {len(df_conflitos)}")
print("-" * 50)
print("Amostra dos pesquisadores com variações de grafia:")

# Mostra apenas as colunas de nome original e nome padronizado para facilitar a leitura
print(df_conflitos[['nome', 'nome_padronizado']].head(20))

# Se quiser salvar para investigar no Excel:
df_conflitos.to_excel('/content/nomes_conflitantes.xlsx', index=False)


Total de nomes em pesquisadores únicos 'nome': 7004
Total de nomes em pesquisadores únicos 'nome_padronizado': 7004
Total de linhas envolvidas em conflitos de padronização: 0
--------------------------------------------------
Amostra dos pesquisadores com variações de grafia:
Empty DataFrame
Columns: [nome, nome_padronizado]
Index: []


## 3.4. `df_lattes`

A tabela df_lattes foi atualizada com a inserção da coluna `id` do pesquisador, a partir da verificação se cada valor da coluna `NOME-COMPLETO` é exatamente igual a algum dos valores da coluna `nome` da tabela `df_pesquisadores_unicos`. Para aumentar as chances de sucesso durante a comparação dos valores, foi necessário padronizar os nomes dos pesquisadores, deixando tudo em caixa alta e sem caracteres de acentuação.

In [ ]:
# @title
# ==============================================================================
# 1. INSERIR COLUNA ID PARA PESQUISADORES DA TABELA LATTES
# ==============================================================================

def padronizar_nome(nome):
    if pd.isna(nome):
        return nome
    nome_upper = str(nome).upper()
    nome_sem_acento = unicodedata.normalize('NFKD', nome_upper).encode('ascii', 'ignore').decode('utf-8')
    return nome_sem_acento.strip()

# Aplicar a padronização à coluna 'NOME-COMPLETO' de df_lattes
df_lattes['NOME_COMPLETO_PADRONIZADO'] = df_lattes['NOME-COMPLETO'].apply(padronizar_nome)

# Verificar a contagem de linhas de df_lattes antes do merge
original_lattes_len = len(df_lattes)
#print(f"\nTotal de entradas em df_lattes ANTES do merge: {original_lattes_len}")

# Realizar a junção (merge) para atribuir o 'id' de df_pesquisadores_unicos a df_lattes
# Usamos um left merge para manter todas as linhas de df_lattes
df_lattes = df_lattes.merge(
    df_pesquisadores_unicos, # Usar o DataFrame com nomes padronizados únicos
    left_on='NOME_COMPLETO_PADRONIZADO',
    right_on='nome_padronizado',
    how='left',
    suffixes=('_original', '') # O sufixo '' garante que a nova coluna 'id' não tenha sufixo se não houver conflito
)

# O merge pode ter adicionado 'nome_padronizado' como uma coluna extra (do 'right_on'). Vamos removê-la.
if 'nome_padronizado' in df_lattes.columns:
    df_lattes.drop(columns=['nome_padronizado'], inplace=True)

#print(f"Total de entradas em df_lattes APÓS o merge: {len(df_lattes)}")

matched_ids_count = df_lattes['id'].count()
print(f"Total de entradas em df_lattes com 'id' atribuído: {matched_ids_count}")
print("-" * 60)

print(f"Tabela 'df_lattes' - Total de linhas: {len(df_lattes)}")
print(df_lattes.dtypes)
print("-" * 60)
#display(df_lattes.head())

Total de entradas em df_lattes com 'id' atribuído: 5915
------------------------------------------------------------
Tabela 'df_lattes' - Total de linhas: 6615
NÚMERO LATTES                           object
NOME-COMPLETO                           object
CIDADE-NASCIMENTO                       object
UF-NASCIMENTO                           object
PAIS-DE-NASCIMENTO                      object
                                        ...   
NOME-DA-SUB-AREA-DO-CONHECIMENTO-11     object
NOME-DA-ESPECIALIDADE-11                object
NOME_COMPLETO_PADRONIZADO               object
nome                                    object
id                                     float64
Length: 64, dtype: object
------------------------------------------------------------


### 3.4.1. `df_potential_matches`
A tabela `df_potential_matches` foi criada para listar as potenciais correspondências (similaridade >= 85%) entre os nomes dos pesquisadores na tabela `df_lattes` sem `id`, e os nomes da tabela `df_pesquisadores_unicos`.

Dos 6615 curriculos lattes baixados foi possível fazer a correspondência exata pelo nome de 5915 pesquisadores. Ou seja, 700 curriculos estão sem correspondência, mas existe uma estimativa de 124 curriculos que podem ser aproximados com segurança (alteração de nome de casada, etc).
<br><br>
`df_potential_matches`<br>
--`Lattes_Nome_Original`<br>
--`Lattes_Nome_Padronizado`<br>
--`Pesquisadores_Nome_Padronizado`<br>
--`Pesquisadores_ID`<br>
--`Similaridade`<br>

```
❗POTENCIAIS MATCHES NÃO FORAM INCLUÍDOS NO PROCESSAMENTO
```

In [ ]:
# @title
# ==============================================================================
# 1. MATCHES NOMES PESQUISADORES
# ==============================================================================

# Identificar os nomes em df_lattes que não possuem id
unmatched_lattes_names = df_lattes[df_lattes['id'].isna()]['NOME-COMPLETO'].drop_duplicates().tolist()
print(f"Total de nomes em df_lattes sem ID: {len(unmatched_lattes_names)}")

# Obter a lista de nomes padronizados de df_pesquisadores_unicos
# Usaremos df_pesquisadores_unicos que já tem nomes padronizados únicos e IDs.
pesquisadores_com_id = df_pesquisadores_unicos[['id', 'nome_padronizado']].copy()

# Converter para dicionário para fácil busca do ID
pesquisadores_dict = pesquisadores_com_id.set_index('nome_padronizado')['id'].to_dict()

# Lista para armazenar as possíveis correspondências
potential_matches = []

# Threshold de similaridade (ajustável)
SIMILARITY_THRESHOLD = 85

# Iterar sobre cada nome não correspondido de df_lattes (agora a lista completa)
print(f"Iniciando o processamento de {len(unmatched_lattes_names)} nomes sem ID para fuzzy matching...")
for lattes_name in unmatched_lattes_names:
    # Padronizar o nome de lattes para comparação
    lattes_name_standardized = padronizar_nome(lattes_name)

    # Encontrar as 3 melhores correspondências no df_pesquisadores_unicos
    # usando o `process.extractOne` para obter o melhor match e sua pontuação.
    # Estamos buscando no `nome_padronizado` de `pesquisadores_com_id`.

    # Certifique-se de que a lista de choices não está vazia para evitar erro
    if not pesquisadores_com_id['nome_padronizado'].empty:
        best_match_info = process.extractOne(lattes_name_standardized,
                                             pesquisadores_com_id['nome_padronizado'],
                                             scorer=fuzz.token_sort_ratio)

        if best_match_info:
            matched_name_standardized, score = best_match_info[0], best_match_info[1]

            if score >= SIMILARITY_THRESHOLD:
                # Encontrar o ID correspondente ao nome padronizado encontrado
                matched_id = pesquisadores_dict.get(matched_name_standardized)
                potential_matches.append({
                    'Lattes_Nome_Original': lattes_name,
                    'Lattes_Nome_Padronizado': lattes_name_standardized,
                    'Pesquisadores_Nome_Padronizado': matched_name_standardized,
                    'Pesquisadores_ID': matched_id,
                    'Similaridade': score
                })

# Criar o DataFrame com os resultados
df_potential_matches = pd.DataFrame(potential_matches)

print(f"\nProcessamento concluído.\nTotal de potenciais correspondências encontradas (similaridade >= {SIMILARITY_THRESHOLD}%): {len(df_potential_matches)}")
#display(df_potential_matches.head(124))

Total de nomes em df_lattes sem ID: 700
Iniciando o processamento de 700 nomes sem ID para fuzzy matching...

Processamento concluído.
Total de potenciais correspondências encontradas (similaridade >= 85%): 124


# 4. Engenharia de Variáveis

## 4.1. `df_contagem_conhecimentos_pesquisadores`

A tabela `df_contagem_conhecimentos_pesquisadores` foi criada com o objetivo de armazenar a contagem dos conhecimentos dos pesquisadores únicos disponível na tabela `df_lattes`, com base na classificação dos conhecimentos disponível na tabela `df_conhecimentos_cnpq`.

A tabela `df_contagem_conhecimentos_pesquisadores` foi atualizada com a remoção de áreas de conhecimento cadastradas pelos pesquisadores no Lattes que estão fora da classificação (padronização) do CNPq.



In [ ]:
# @title
# --- FUNÇÃO PADRONIZAÇÃO ---

def padronizar_texto(texto):
    if not isinstance(texto, str):
        return texto
    nfkd_form = unicodedata.normalize('NFKD', texto)
    texto_sem_acento = "".join([c for c in nfkd_form if not unicodedata.combining(c)])
    return texto_sem_acento.upper().replace('_', ' ').strip()

# ==============================================================================
# 1. CONTAGEM DOS CONHECIMENTOS
# ==============================================================================

# --- CRIAÇÃO DA TABELA LONGA ID E CONHECIMENTOS DO LATTES ---

# Selecionar colunas de conhecimento + id
colunas_conhecimento = df_lattes.filter(regex='CONHECIMENTO').columns.tolist()
df_contagem_conhecimentos = df_lattes[['id'] + colunas_conhecimento].copy()

# Transformar de formato "largo" para "longo"
df_longo = df_contagem_conhecimentos.melt(
    id_vars=['id'],
    var_name='origem',
    value_name='area_conhecimento'
)

# Remover valores nulos ou vazios logo após o melt
df_longo = df_longo.dropna(subset=['area_conhecimento'])
df_longo = df_longo[df_longo['area_conhecimento'].astype(str).str.strip() != '']

# --- PADRONIZAÇÃO DOS DADOS ---

# Aplica a padronização na coluna de conhecimento
df_longo['area_conhecimento'] = df_longo['area_conhecimento'].apply(padronizar_texto)

# --- AGRUPAMENTO E RESULTADOS ---

# Realiza a contagem final por ID e Área de Conecimento Padronizada
df_resultado = df_longo.groupby(['id', 'area_conhecimento']).size().reset_index(name='contagem')

# Cálculo do total de registros processados
total_registros = df_resultado['contagem'].sum()

# --- EXIBIÇÃO ---

#print(df_resultado.head())
output_excel_filename = '/content/df_contagem_conhecimentos_total.xlsx'
df_contagem_conhecimentos.to_excel(output_excel_filename, index=False)

# ==============================================================================
# 2. REMOVER CONHECIMENTOS NÃO CLASSIFICADOS PELO CNPQ
# ==============================================================================

# --- PROCESSAMENTO E VALIDAÇÃO ---

# Criar a tabela de referência a partir do df_conhecimentos_cnpq
conhecimentos_referencia = df_conhecimentos_cnpq['TEMAS'].apply(padronizar_texto).unique()
df_referencia = pd.DataFrame(conhecimentos_referencia, columns=['area_conhecimento'])

# Preparar o df_resultado (Normalizando a coluna de entrada)
df_resultado['area_conhecimento_norm'] = df_resultado['area_conhecimento'].apply(padronizar_texto)

# Identificar conhecimentos que NÃO estão na lista oficial
conhecimentos_nao_listados = df_resultado[
    ~df_resultado['area_conhecimento_norm'].isin(df_referencia['area_conhecimento'])
]

print(f"Conhecimentos inválidos: {len(conhecimentos_nao_listados)}")

# Gerar tabela APENAS com resultados validados
df_validados = df_resultado[
    df_resultado['area_conhecimento_norm'].isin(df_referencia['area_conhecimento'])
].copy()

print(f"Conhecimentos válidos: {len(df_validados)}")

# Listar as discrepâncias únicas
lista_discrepancias = conhecimentos_nao_listados['area_conhecimento_norm'].unique()

print(f"Categorias não reconhecidas pelo CNPq na lista de conhecimentos inválidos: {len(lista_discrepancias)}")

# --- EXPORTAÇÃO 1: LISTA DE DISCREPÂNCIAS ---

# Transformamos as discrepâncias em DataFrame e adicionamos a frequência
df_export_discrepancias = pd.DataFrame(lista_discrepancias, columns=['Conhecimento_Nao_Encontrado'])

# Mapeia a soma da coluna 'contagem' para cada discrepância encontrada
contagens = conhecimentos_nao_listados.groupby('area_conhecimento_norm')['contagem'].sum()
df_export_discrepancias['Frequencia_Total'] = df_export_discrepancias['Conhecimento_Nao_Encontrado'].map(contagens)

# Ordenar pelas mais frequentes
df_export_discrepancias = df_export_discrepancias.sort_values(by='Frequencia_Total', ascending=False)

# --- EXIBIÇÃO ---

output_excel_filename = '/content/df_export_discrepancias.xlsx'
df_export_discrepancias.to_excel(output_excel_filename, index=False)

# ==============================================================================
# 3. PIVOTAR TABELA
# ==============================================================================

# Realizando o pivot da tabela
df_contagem_conhecimentos_pesquisadores = df_validados.pivot(
    index='id',
    columns='area_conhecimento',
    values='contagem'
)

# Opcional: Preencher com 0 as áreas que um determinado ID não possui (NaN)
df_contagem_conhecimentos_pesquisadores = df_contagem_conhecimentos_pesquisadores.fillna(0).astype(int)

# Resetar o índice para que 'id' volte a ser uma coluna comum
df_contagem_conhecimentos_pesquisadores = df_contagem_conhecimentos_pesquisadores.reset_index()

# --- EXIBIÇÃO ---

#print(df_contagem_conhecimentos_pesquisadores.head())
output_excel_filename = '/content/df_contagem_conhecimentos_pesquisadores.xlsx'
df_contagem_conhecimentos_pesquisadores.to_excel(output_excel_filename, index=False)

print("-" * 60)
print(f"Tabela 'df_contagem_conhecimentos_pesquisadores' - Total de linhas: {len(df_contagem_conhecimentos_pesquisadores)}")
print(df_contagem_conhecimentos_pesquisadores.dtypes)
print("-" * 60)


Conhecimentos inválidos: 5916
Conhecimentos válidos: 22822
Categorias não reconhecidas pelo CNPq na lista de conhecimentos inválidos: 2810
------------------------------------------------------------
Tabela 'df_contagem_conhecimentos_pesquisadores' - Total de linhas: 5054
area_conhecimento
id                                      float64
ADMINISTRACAO                             int64
ADMINISTRACAO DE EMPRESAS                 int64
ADMINISTRACAO DE SETORES ESPECIFICOS      int64
ADMINISTRACAO EDUCACIONAL                 int64
                                         ...   
TURISMO                                   int64
VEICULOS E EQUIPAMENTOS DE CONTROLE       int64
ZOOLOGIA                                  int64
ZOOLOGIA APLICADA                         int64
ZOOTECNIA                                 int64
Length: 360, dtype: object
------------------------------------------------------------


## 4.2. `df_contagem_conhecimentos_projetos`

A tabela `df_contagem_conhecimentos_projetos` foi criada para armazenar a soma dos conhecimentos de todos os pesquisadores relacionado ao mesmo projeto na tabela `df_contagem_conhecimentos_pesquisadores`.

In [ ]:
# @title
# ==============================================================================
# 1. CRIAR TABELA DA CONTAGEM DOS CONHECIMENTOS DOS PESQUISADORES NOS PROJETOS
# ==============================================================================

# Criar df_diversidade_pesquisadores a partir de df_pesquisadores_projetos
df_contagem_conhecimentos_projetos_prep = df_pesquisadores_projetos.copy()

# Realizar o merge com df_contagem_conhecimentos_pesquisadores usando a coluna 'id'
df_contagem_conhecimentos_projetos_prep = df_contagem_conhecimentos_projetos_prep.merge(
    df_contagem_conhecimentos_pesquisadores,
    on='id',
    how='left'
)

# Preencher valores NaN (que surgiram de IDs em df_pesquisadores_projetos que não estavam em df_contagem_conhecimentos)
# com 0, pois representam ausência de contagem de conhecimento para esses pesquisadores.
df_contagem_conhecimentos_projetos_prep = df_contagem_conhecimentos_projetos_prep.fillna(0)

print(f"Total de linhas em df_contagem_conhecimentos_projetos: {len(df_contagem_conhecimentos_projetos_prep)}")
#display(df_contagem_conhecimentos_projetos_prep.head())
output_excel_filename = '/content/df_contagem_conhecimentos_projetos_prep.xlsx'
df_contagem_conhecimentos_projetos_prep.to_excel(output_excel_filename, index=False)

# ==============================================================================
# 2. SOMAR OS CONECIMENTOS DOS PESQUISADORES
# ==============================================================================

df_contagem_conhecimentos_projetos = df_contagem_conhecimentos_projetos_prep.copy()
df_contagem_conhecimentos_projetos.drop(columns=['nome', 'tipo', 'id', 'nome_original', 'nome_padronizado'], inplace=True)
df_contagem_conhecimentos_projetos = df_contagem_conhecimentos_projetos.groupby('processo_proj').sum().reset_index()

print(f"Total de projetos únicos: {len(df_contagem_conhecimentos_projetos)}")
#display(df_contagem_conhecimentos_projetos.head())
output_excel_filename = '/content/df_contagem_conhecimentos_projetos.xlsx'
df_contagem_conhecimentos_projetos.to_excel(output_excel_filename, index=False)


print("-" * 60)
print(f"Tabela 'df_contagem_conhecimentos_projetos' - Total de linhas: {len(df_contagem_conhecimentos_projetos)}")
print(df_contagem_conhecimentos_projetos.dtypes)
print("-" * 60)

## 4.3. `df_diversidade_projetos`

A tabela `df_diversidade_projetos`foi criada para armazenar os indicadores de diversidade dos projetos.

Trechos retirados da dissertação, nas páginas: 41 e 42:

> Para o processamento dos dados foi utilizada a biblioteca Python Scikit-bio com o módulo skbio.diversity, especializado em cálculos sobre diversidade biológica. Dessa forma, foi possível calcular a diversidade alfa (Equações 8 e 9) das áreas de conhecimento dos projetos PIPE-FAPESP.
>
> <br>
>
>(8) $$\alpha_{Si} = \sum_{i} p_i^2$$
>
><br>
>
>(9) $$\alpha_{Sh} = - \sum_{i} p_i \cdot \ln p_i$$
>
><br>
>
>Sendo: $α_{Si}$ = diversidade alfa baseada em Simpson (1949); $α_{Sh}$ = diversidade alfa baseada em Shannon & Weaver (1964); $p_i$ = proporções das áreas de conhecimento da amostra.
>
>A diversidade em cada um dos três níveis hierárquicos do conhecimento das áreas de conhecimento do PIPE-FAPESP foi gerada separadamente (Figura 4), pois cada nível hierárquico possui uma variedade diferente de conhecimentos. Dessa forma, os indicadores de diversidade foram agregados para composição de um único indicador, possibilitando uma comparação geral (Equação 10):
>
><br>
>
>(10) $$d_g = d_a + d_b + d_c$$
>
><br>
>
>Sendo: $d_g$ = diversidade geral; $d_a$ = diversidade da grande área de conhecimentos dos projetos; $d_b$ = diversidade da área de conhecimentos dos projetos; $d_c$ = diversidade da subárea de conhecimentos dos projetos.




In [ ]:
# @title
# ==============================================================================
# GRERAR INDICADORES DE DIVERSIDADE DOS PROJETOS
# ==============================================================================

# FAZER A PADRONIZAÇÃO DO TEXTO.

def remover_acentos(texto):
    if not isinstance(texto, str):
        return ""
    nfkd_form = unicodedata.normalize('NFKD', texto)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

df_grandes_areas = df_conhecimentos_cnpq.loc[
    df_conhecimentos_cnpq['ÁREAS'] == 'Grande Área de Conhecimento',
    ['TEMAS']
].copy()

df_grandes_areas['TEMAS'] = (
    df_grandes_areas['TEMAS']
    .apply(remover_acentos)
    .str.upper()
    .str.strip()
)
df_areas = df_conhecimentos_cnpq.loc[
    df_conhecimentos_cnpq['ÁREAS'] == 'Área de Conhecimento',
    ['TEMAS']
].copy()

df_areas['TEMAS'] = (
    df_areas['TEMAS']
    .apply(remover_acentos)
    .str.upper()
    .str.strip()
)
df_sub_areas = df_conhecimentos_cnpq.loc[
    df_conhecimentos_cnpq['ÁREAS'] == 'Sub Área de Conhecimento',
    ['TEMAS']
].copy()

df_sub_areas['TEMAS'] = (
    df_sub_areas['TEMAS']
    .apply(remover_acentos)
    .str.upper()
    .str.strip()
)

df_grandes_areas = df_grandes_areas.drop_duplicates()
conhecimentos_ga = df_grandes_areas['TEMAS'].tolist()

df_areas = df_areas.drop_duplicates()
conhecimentos_a = df_areas['TEMAS'].tolist()

df_sub_areas = df_sub_areas.drop_duplicates()
conhecimentos_sa = df_sub_areas['TEMAS'].tolist()


# AREAS QUE NAO POSSUEM NENHUMA CONTAGEM E PRECISARAM TER AS COLUNAS REENSERIDAS COM VALOR 0.

df_contagem_conhecimentos_projetos[['ANTROPOLOGIA','PASTAGENS E FORRAGICULTURA', 'ENGENHARIA DE PESCA',
                         'PALEOBOTANICA', 'BIOQUIMICA DE MICROORGANISMOS', 'FARMACOLOGIA AUTONOMICA',
                         'ENFERMAGEM OBSTETRICA', 'ENFERMAGEM PEDIATRICA', 'ENFERMAGEM PSIQUIATRICA',
                         'ENFERMAGEM DE DOENCAS CONTAGIOSAS', 'ENFERMAGEM DE SAUDE PUBLICA',
                         'DESNUTRICAO E DESENVOLVIMENTO FISIOLOGICO', 'ASTROFISICA DO MEIO INTERESTELAR',
                         'ASTRONOMIA DO SISTEMA SOLAR', 'METAFISICA', 'FILOSOFIA BRASILEIRA',
                         'FUNDAMENTOS DA SOCIOLOGIA', 'SOCIOLOGIA DO CONHECIMENTO',
                         'SOCIOLOGIA DO DESENVOLVIMENTO', 'SOCIOLOGIA RURAL', 'SOCIOLOGIA DA SAUDE',
                         'TEORIA ANTROPOLOGICA', 'ETNOLOGIA INDIGENA', 'ANTROPOLOGIA URBANA',
                         'ANTROPOLOGIA RURAL', 'ANTROPOLOGIA DAS POPULACOES AFRO-BRASILEIRAS',
                         'TEORIA E METODO EM ARQUEOLOGIA', 'ARQUEOLOGIA PRE-HISTORICA',
                         'TEORIA E FILOSOFIA DA HISTORIA', 'HISTORIA DA AMERICA',
                         'PSICOLOGIA COMPARATIVA', 'ORIENTACAO E ACONSELHAMENTO', 'TEORIA POLITICA',
                         'ESTADO E GOVERNO', 'COMPORTAMENTO POLITICO', 'HISTORIA DA TEOLOGIA',
                         'TEOLOGIA MORAL', 'TEOLOGIA SISTEMATICA', 'TEOLOGIA PASTORAL',
                         'TEORIA DO DIREITO', 'DIREITOS ESPECIAIS', 'TENDENCIA POPULACIONAL',
                         'COMPONENTES DA DINAMICA DEMOGRAFICA', 'NUPCIALIDADE E FAMILIA',
                         'DEMOGRAFIA HISTORICA', 'FONTES DE DADOS DEMOGRAFICOS',
                         'FUNDAMENTOS DO SERVICO SOCIAL', 'LAVRA',
                         'MEDIDAS ELETRICAS, MAGNETICAS E ELETRONICAS, INSTRUMENTACAO',
                         'FENOMENOS DE TRANSPORTES', 'TRATAMENTOS DE AGUAS DE ABASTECIMENTO E RESIDUARIAS',
                         'TECNOLOGIA DE CONSTRUCAO NAVAL E DE SISTEMAS OCEANICOS',
                         'MATERIAIS E PROCESSOS PARA ENGENHARIA AERONAUTICA, AEROESPACIAL',
                         'FILOSOFIA DA LINGUAGEM', 'LINGUISTICA HISTORICA',
                         'SOCIOLINGUISTICA E DIALETOLOGIA', 'PSICOLINGUISTICA', 'LINGUAS CLASSICAS',
                         'LINGUAS INDIGENAS', 'TEORIA LITERARIA', 'OUTRAS LITERATURAS VERNACULAS',
                         'LITERATURAS ESTRANGEIRAS MODERNAS', 'LITERATURAS CLASSICAS',
                         'LITERATURA COMPARADA', 'FUNDAMENTOS E CRITICA DAS ARTES', 'OPERA']] = 0


# ==============================================================================
# USO DA BIBLIOTECA SCIKIT-BIO
# ==============================================================================

# GRANDE ÁREA DE CONHECIMENTO

data_ga = df_contagem_conhecimentos_projetos[conhecimentos_ga].copy()
ids = df_contagem_conhecimentos_projetos['processo_proj'].values.tolist()

alfa_sha_ga = alpha_diversity('shannon', data_ga, ids)
tb_alfa_sha_ga = pd.DataFrame(alfa_sha_ga)
tb_alfa_sha_ga = tb_alfa_sha_ga[0].fillna('')

alfa_sim_ga = alpha_diversity('simpson', data_ga, ids)
tb_alfa_sim_ga = pd.DataFrame(alfa_sim_ga)
tb_alfa_sim_ga = tb_alfa_sim_ga[0].fillna('')

# ÁREA DE CONHECIMENTO

data_a = df_contagem_conhecimentos_projetos[conhecimentos_a].copy()

alfa_sha_a = alpha_diversity('shannon', data_a, ids)
tb_alfa_sha_a = pd.DataFrame(alfa_sha_a)
tb_alfa_sha_a = tb_alfa_sha_a[0].fillna('')

alfa_sim_a = alpha_diversity('simpson', data_a, ids)
tb_alfa_sim_a = pd.DataFrame(alfa_sim_a)
tb_alfa_sim_a = tb_alfa_sim_a[0].fillna('')

# SUB ÁREA DE CONHECIMENTO

data_sa = df_contagem_conhecimentos_projetos[conhecimentos_sa].copy()

alfa_sha_sa = alpha_diversity('shannon', data_sa, ids)
tb_alfa_sha_sa = pd.DataFrame(alfa_sha_sa)
tb_alfa_sha_sa = tb_alfa_sha_sa[0].fillna('')

alfa_sim_sa = alpha_diversity('simpson', data_sa, ids)
tb_alfa_sim_sa = pd.DataFrame(alfa_sim_sa)
tb_alfa_sim_sa = tb_alfa_sim_sa[0].fillna('')

# ==============================================================================
# DIVERSIDADE GERAL
# ==============================================================================

df_sha_a = tb_alfa_sha_a.reset_index()
df_sha_ga = tb_alfa_sha_ga.reset_index()
df_sha_sa = tb_alfa_sha_sa.reset_index()

df_sim_a = tb_alfa_sim_a.reset_index()
df_sim_ga = tb_alfa_sim_ga.reset_index()
df_sim_sa = tb_alfa_sim_sa.reset_index()

shannon = [df_sha_a, df_sha_ga, df_sha_sa]
simpson = [df_sim_a, df_sim_ga, df_sim_sa]

# Concatenação
df_shannon_consolidado = pd.concat(shannon, ignore_index=True)
df_simpson_consolidado = pd.concat(simpson, ignore_index=True)

# primeira e a segunda colunas
col_chave_shannon = df_shannon_consolidado.columns[0]
col_soma_shannon = df_shannon_consolidado.columns[1]
df_shannon_consolidado[col_soma_shannon] = pd.to_numeric(df_shannon_consolidado[col_soma_shannon], errors='coerce').fillna(0)

col_chave_simpson = df_simpson_consolidado.columns[0]
col_soma_simpson = df_simpson_consolidado.columns[1]
df_simpson_consolidado[col_soma_simpson] = pd.to_numeric(df_simpson_consolidado[col_soma_simpson], errors='coerce').fillna(0)

# Agrupamento e Soma
df_diversidade_shannon = df_shannon_consolidado.groupby(col_chave_shannon)[col_soma_shannon].sum().reset_index()
df_diversidade_simpson = df_simpson_consolidado.groupby(col_chave_simpson)[col_soma_simpson].sum().reset_index()

# Renomeamos as colunas padronizando a chave (processo_proj)
df_diversidade_shannon.columns = ['processo_proj', 'Shannon']
df_diversidade_simpson.columns = ['processo_proj', 'Simpson']

# juntamos as duas tabelas agrupadas
df_diversidade_projetos = df_diversidade_shannon.merge(df_diversidade_simpson, on='processo_proj', how='outer')

# EXIBIÇÃO
print("-" * 60)
print(f"Tabela 'df_diversidade_projetos' - Total de linhas: {len(df_diversidade_projetos)}")
print(df_diversidade_projetos.dtypes)
print("-" * 60)
#display(df_diversidade_projetos.head())

## 4.4. `df_coesao_projetos`

A tabela `df_coesao_projetos` foi criada para armazenar o indicador de coesão de cada projeto.

Trechos retirados da dissertação, nas páginas: 46 e 47:

> A densidade de rede é uma medida que avalia a quantidade de conexões presentes em uma rede em relação ao número total de possíveis conexões (BARABÁSI, 2015). E pode ser obtida multiplicando o número de arestas por 2 para contar todas as conexões bidirecionais e, em seguida, divide esse valor pelo número total de possíveis conexões entre todos os pares de nós no grafo (Equação 11):
>
><br>
>
>(11) $$D = \frac{2E}{N \cdot (N - 1)}$$
>
><br>
>
>Sendo: $D$ = densidade de rede; $E$ = totalidade de arestas possíveis representando conhecimentos inter-pesquisadores; $N$ = número de nós representando a quantidade dos mesmos conhecimentos dos pesquisadores
>
>Para estimar a quantidade de conhecimentos combinados dos pesquisadores, é possível encontrar o total de arestas em um determinado número de nós (Equação 12):
>
><br>
>
>(12) $$E = \frac{(N - 1) \cdot N}{2}$$
>
><br>
>
>No entanto, ainda é necessário excluir as arestas que conectam os conhecimentos da trajetória de cada pesquisador (Equação 13):
>
><br>
>
>(13) $$e = \{ (c_p - 1) + (c_p - 1) \dots \}$$
>
><br>
>
>Sendo: e = soma das arestas das trajetórias dos pesquisadores; $c_p$ = quantidade de um único conhecimento de um único pesquisador.
Com o total de arestas inter-conhecimento dos pesquisadores (Equação 14) é possível calcular a coesão dos conhecimentos nos projetos através da densidade da rede:
>
><br>
>
>(14) $$C = \ln(E - e)$$
>
><br>
>
>Sendo: $C$ = logaritmo neperiano do total de arestas exclusivas que relacionam os conhecimentos inter-pesquisadores.


In [ ]:
# @title
# ==============================================================================
# 1. CRIAR TABELA DE COESÃO DOS PROJETOS
# ==============================================================================

# 1. Criar tabela do conhecimento dos pesquisadores
df_coesao_pesquisadores = df_contagem_conhecimentos_projetos_prep.copy()

# 2. Criar tabela do conhecimento dos pesquisadores -1
# Se o conhecimento do pesquisador é >0 então -1, senão 0.
df_coesao_pesquisadores_1 = df_contagem_conhecimentos_projetos_prep.copy()

colunas_texto = df_coesao_pesquisadores_1.select_dtypes(include=['object', 'string']).columns
#print(colunas_texto)
colunas_para_preservar = ['nome_original', 'tipo', 'processo_proj', 'nome_padronizado', 'nome']

# Criamos a lista inicial de ajuste (todas menos as preservadas)
colunas_candidatas = [col for col in df_coesao_pesquisadores_1.columns if col not in colunas_para_preservar]

# FILTRO DE SEGURANÇA: Manter apenas as colunas que são realmente números (int ou float)
# Isso evita que colunas de texto que esquecemos de listar causem o erro
colunas_para_ajuste = df_coesao_pesquisadores_1[colunas_candidatas].select_dtypes(include=['number']).columns

# Aplicamos a lógica (Corrigindo também o nome da tabela no final)
df_coesao_pesquisadores_1[colunas_para_ajuste] = df_coesao_pesquisadores_1[colunas_para_ajuste].where(
    df_coesao_pesquisadores_1[colunas_para_ajuste] <= 0,
    df_coesao_pesquisadores_1[colunas_para_ajuste] - 1
)

# 3. Total de Conhecimento
# Preparação (Removendo colunas indesejadas)
df_base = df_coesao_pesquisadores.drop(columns=['nome_original', 'tipo', 'nome_padronizado', 'nome', 'id'], errors='ignore')
colunas_calc = df_base.columns.drop('processo_proj')

# --- AVALIANDO AS PRÉ-CONDIÇÕES (Linha a linha) ---

# Regra A: O 'processo_proj' aparece mais de 1 vez na tabela? (Retorna True/False para cada linha)
regra_a = df_base.groupby('processo_proj')['processo_proj'].transform('size') > 1

# Regra B1: Existem MAIS DE 2 valores maiores que zero para esta coluna neste processo?
regra_b = (df_base[colunas_calc] > 0).groupby(df_base['processo_proj']).transform('sum') > 1

# Regra B2 (A principal): O valor da célula atual é maior que zero?
# (Isso impede que negativos entrem na soma indevidamente)
valor_positivo = df_base[colunas_calc] > 0

# --- APLICANDO AS PRÉ-CONDIÇÕES ANTES DE SOMAR ---

# Criamos uma tabela apenas com os valores que passarão para a soma final
df_valores_validos = df_base.copy()

# 1º Filtro: Zeramos as células que não cumprem a Regra B (tem que ter >2 positivos E a própria célula ser >0)
df_valores_validos[colunas_calc] = df_valores_validos[colunas_calc].where(regra_b & valor_positivo, 0)

# 2º Filtro: Zeramos todos os processos que não cumprem a Regra A (aparecer >1 vez)
# O símbolo ~ (til) significa "NÃO". Ou seja, onde NÃO atender a regra A, coloque 0.
df_valores_validos.loc[~regra_a, colunas_calc] = 0

# --- A SOMA ---
# Agora sim, fazemos a soma tranquilamente, pois os números que não cumpriam as regras já viraram 0.
df_coesao_total = df_valores_validos.groupby('processo_proj').sum().reset_index()


# print(f"Total de projetos únicos em df_coesao_total: {len(df_coesao_total)}")
# output_excel_filename = '/content/df_coesao_total.xlsx'
# df_coesao_total.to_excel(output_excel_filename, index=False)

# 4. Total de Conhecimentos -1

df_coesao_total_1 = df_coesao_pesquisadores_1.copy()
df_coesao_total_1.drop(columns=['nome', 'tipo', 'id'], inplace=True)
df_coesao_total_1 = df_coesao_total_1.groupby('processo_proj').sum().reset_index()
#print(f"Total de projetos únicos em df_coesao_total_1: {len(df_coesao_total_1)}")
#output_excel_filename = '/content/df_coesao_total_1.xlsx'
#df_coesao_total_1.to_excel(output_excel_filename, index=False)

# 5. Arestas de Conhecimento

df_coesao_projetos = df_coesao_total.copy()
colunas_calc = df_coesao_total.columns.drop('processo_proj')
calculo = (((df_coesao_total[colunas_calc]-1) * df_coesao_total[colunas_calc]) / 2) - df_coesao_total_1[colunas_calc]
df_coesao_projetos[colunas_calc] = calculo.where(df_coesao_total[colunas_calc] != 0, 0)
#print(df_coesao.head())

#output_excel_filename = '/content/df_coesao.xlsx'
#df_coesao.to_excel(output_excel_filename, index=False)

# 6. Soma das arestas das áreas e log

# Identificamos as colunas que serão somadas (todas, exceto o processo)
colunas_para_somar = df_coesao_projetos.columns.drop('processo_proj')

# Somamos os valores linha por linha (axis=1 indica soma horizontal)
soma_das_linhas = df_coesao_projetos[colunas_para_somar].sum(axis=1)

# Calculamos o logaritmo neperiano (ln) da soma e criamos a nova coluna
# Atenção: O logaritmo neperiano de 0 é -infinito (-inf).
# Para evitar erros no seu Excel depois, podemos colocar uma regra: se a soma for > 0 calcula o log, senão retorna 0.
df_coesao_projetos['coesao'] = np.where(soma_das_linhas > 0, np.log(soma_das_linhas), 0)

# Criar uma tabela LIMPA contendo APENAS o processo e o resultado da coesão:
df_coesao_projetos = df_coesao_projetos[['processo_proj', 'coesao']].copy()

# EXIBIÇÃO
print("-" * 60)
print(f"Tabela 'df_coesao_projetos' - Total de linhas: {len(df_coesao)}")
print(df_coesao_projetos.dtypes)
print("-" * 60)
#print(df_coesao.head())


## 4.5. `df_rede`



# 5. Unificação da Base de Dados

A tabela `df_projetos_pipe`foi atualida para conter os indicadores de diversidade e coesão.

In [ ]:
# @title

# 1. PADRONIZAÇÃO: Renomeia 'N. Processo' para 'processo_proj' para que todas as tabelas falem a mesma língua
df_projetos_pipe.rename(columns={'N. Processo': 'processo_proj'}, inplace=True)

# 2. Adiciona as colunas de df_diversidade_projetos
df_projetos_pipe = pd.merge(df_projetos_pipe, df_diversidade_projetos, on='processo_proj', how='left')

# 3. Adiciona as colunas de df_coesao
# Opcional: Se quiser puxar APENAS a coluna de resultado da coesão e ignorar o resto:
# df_graficos = pd.merge(df_graficos, df_coesao[['processo_proj', 'coesao']], on='processo_proj', how='left')
# Se quiser puxar tudo, use a linha abaixo:
df_projetos_pipe = pd.merge(df_projetos_pipe, df_coesao_projetos, on='processo_proj', how='left')

# Criar coluna com apenas o Ano de inicio do projeto.
df_projetos_pipe['Ano'] = df_projetos_pipe['Data de Início'].astype(str).str[:4].astype(int)
# Eperimento
df_projetos_pipe['coesao2'] = df_projetos_pipe['Shannon'] * df_projetos_pipe['coesao']

print(f"Total de projetos únicos: {len(df_projetos_pipe)}")
output_excel_filename = '/content/df_projetos_pipe.xlsx'
df_projetos_pipe.to_excel(output_excel_filename, index=False)

# 6. Análise Exploratória e Visualização

In [ ]:
# @title
import matplotlib.pyplot as plt
import seaborn as sns

# Preparação dos dados (Filtrando o ano como no seu exemplo)
#graf3 = df_projetos_pipe.loc[df_projetos_pipe["Ano"] >= 2013]

plt.figure(figsize=(10, 6))
sns.set_theme(style="white")

# Criando o histograma para a variável "coesao"
sns.histplot(
    data=graf3,
    x="coesao",
    hue="Grande Área do Conhecimento",
    kde=True,          # Adiciona a linha de densidade
    element="step",    # Estilo que evita que as barras se escondam atrás das outras
    palette="tab10",
    alpha=0.5
)

plt.title("Distribuição de Coesão por Área do Conhecimento")
plt.xlabel("Coesão")
plt.ylabel("Frequência")

# Mover a legenda para fora
sns.move_legend(plt.gca(), "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()

plt.show()

In [ ]:
# @title
# Criando uma grade de histogramas
g = sns.displot(
    data=graf3,
    x="coesao",
    col="Grande Área do Conhecimento",
    col_wrap=3,        # 3 colunas por linha
    kde=True,
    color="steelblue",
    height=4,
    aspect=1.2
)

g.set_axis_labels("Coesão", "Contagem")
g.set_titles("{col_name}") # Define o título de cada subgráfico como o nome da área

plt.show()

In [ ]:
# @title
plt.figure(figsize=(8, 6))

sns.histplot(
    data=graf3,
    x="coesao",
    y="Shannon",
    bins=30,
    pthresh=.1,
    cmap="mako"
)

plt.title("Densidade: Coesão vs Shannon")
plt.show()

In [ ]:
# @title
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Filtro de dados
#graf3 = df_projetos_pipe.loc[df_projetos_pipe["Ano"] >= 2013]

sns.set_theme(style="whitegrid")

# 2. Criando histogramas verticais para a variável Shannon
# O 'displot' é a melhor função para criar múltiplos subgráficos (facets)
g = sns.displot(
    data=graf3,
    x="Shannon",
    col="Grande Área do Conhecimento", # Cria um gráfico para cada área
    col_wrap=3,                       # Organiza em 3 colunas
    kde=True,                         # Adiciona a curva de densidade
    color="darkcyan",                 # Cor sóbria e profissional
    height=4,
    aspect=1.2,
    bins=20                           # Ajuste o número de barras conforme necessário
)

# 3. Ajustes de layout e títulos
g.set_axis_labels("Índice de Shannon", "Frequência (Contagem)")
g.set_titles("{col_name}") # Usa o nome da área como título de cada quadrante

# Título principal para a figura inteira
g.fig.suptitle("Distribuição do Índice de Shannon por Grande Área do Conhecimento (Pós-2013)",
               fontsize=16, y=1.05)

plt.show()

In [ ]:
# @title
graf3 = df_projetos_pipe
#graf3 = graf3.loc[(graf3["Ano"]>=2013)]
#graf3 = graf3.loc[(graf3["Ano"]>=2013) & (graf3["Grande Área do Conhecimento"]=="Ciências Agrárias")]
#graf3 = graf3.loc[(graf3["Ano"]>=2013) & (graf3["Grande Área"]=="Ciências Biológicas")]
#graf3 = graf3.loc[(graf3["Ano"]>=2013) & (graf3["Grande Área"]=="Ciências da Saúde")]
#graf3 = graf3.loc[(graf3["Ano"]>=2013) & (graf3["Grande Área do Conhecimento"]=="Ciências Exatas e da Terra")]
#graf3 = graf3.loc[(graf3["Ano"]>=2013) & (graf3["Grande Área"]=="Ciências Humanas")]
#graf3 = graf3.loc[(graf3["Ano"]>=2013) & (graf3["Grande Área"]=="Ciências Sociais Aplicadas")]
#graf3 = graf3.loc[(graf3["Ano"]>=2013) & (graf3["Grande Área"]=="Engenharias")]
#graf3 = graf3.loc[(graf3["Ano"]>=2013) & (graf3["Grande Área"]=="Interdisciplinar")]
#graf3 = graf3.loc[(graf3["Ano"]>=2013) & (graf3["Grande Área"]=="Linguística, Letras e Artes")]

sns.set_theme(style="white")

# Código ajustado: removido o truncate=False
g = sns.jointplot(
    data=graf3,
    x="coesao",
    y="Shannon",
    hue="Grande Área do Conhecimento",
    kind="scatter",
    height=7,
    xlim=(0, 9),
    ylim=(0, 7),
    palette="tab10",
    alpha=0.7
)

# --- FORÇANDO A LARGURA (O TRUQUE) ---
# Aqui você define (Largura, Altura). Vamos esticar a largura para 10.
g.fig.set_size_inches(10, 7)

# --- APLICANDO A ESCALA LOG ---
#g.ax_joint.set_xscale('log')
#g.ax_joint.set_yscale('log')

# --- MOVENDO A LEGENDA ---
# Remove o título padrão da legenda (opcional, para ficar mais limpo)
g.ax_joint.legend_.set_title(None)

# Move a legenda do eixo central para fora.
# bbox_to_anchor=(X, Y): 1.25 empurra para a direita, 0.5 centraliza na vertical
sns.move_legend(g.ax_joint, "center left", bbox_to_anchor=(1.25, 0.5))

# Garante que a legenda não seja "cortada" se você for salvar a imagem
plt.subplots_adjust(right=0.75)

plt.show()


In [ ]:
# @title
# 1. Preparação dos dados (Filtrando o ano se necessário)
df_filtrado = df_projetos_pipe.loc[df_projetos_pipe["Ano"] >= 2013]

# 2. Obter a lista de áreas únicas
areas = df_filtrado["Grande Área do Conhecimento"].unique()

sns.set_theme(style="white")

# 3. Loop para criar um gráfico para cada área
for area in areas:
    # Filtrando os dados da área específica
    dados_area = df_filtrado[df_filtrado["Grande Área do Conhecimento"] == area]

    # Criando o Jointplot
    g = sns.jointplot(
        data=dados_area,
        x="coesao",
        y="Shannon",
        kind="scatter",
        height=2,
        xlim=(0, 9),
        ylim=(0, 7),
        color="steelblue", # Cor fixa ou baseada em um mapa de cores
        alpha=0.7
    )

    # Ajustando o título para identificar a área
    g.fig.suptitle(f"Área: {area}", fontsize=14, y=1.02)

    # Ajustando o tamanho e margens
    g.fig.set_size_inches(8, 6)
    plt.subplots_adjust(top=0.9, right=0.9)

    plt.show()

In [ ]:
# @title
# Criando uma grade de comparação (Scatter plots lado a lado)
g = sns.relplot(
    data=df_filtrado,
    x="coesao",
    y="Shannon",
    col="Grande Área do Conhecimento", # Cria uma coluna para cada área
    col_wrap=3,                       # Define quantos gráficos por linha
    hue="Grande Área do Conhecimento",
    kind="scatter",
    palette="tab10",
    alpha=0.7,
    height=4,
    aspect=1
)

# Ajustando limites dos eixos para todos os gráficos
g.set(xlim=(0, 9), ylim=(0, 7))

# Título geral
g.fig.suptitle("Comparação por Grande Área do Conhecimento", fontsize=16, y=1.05)

plt.show()

In [ ]:
# @title


sns.set_theme(style="white")
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=graf3,
    x="Ano",
    y="Shannon",
    palette="Set2"
)

# Adiciona os pontos reais por cima da caixa
sns.stripplot(
    data=graf3,
    x="Ano",
    y="Shannon",
    color="black",
    alpha=0.3,
    size=2
)

plt.title("Distribuição do Índice de Shannon ao Longo dos Anos", fontsize=14)
plt.show()

graf3.groupby(["Grande Área do Conhecimento"])["Shannon"].describe()

In [ ]:
# @title


sns.set_theme(style="white")
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=graf3,
    x="Ano",
    y="coesao",
    palette="Set2"
)

# Adiciona os pontos reais por cima da caixa (Opcional, mas fica lindo!)
sns.stripplot(
    data=graf3,
    x="Ano",
    y="Shannon",
    color="black",
    alpha=0.3,
    size=2
)

plt.title("Distribuição da Coesão ao Longo dos Anos", fontsize=14)
plt.show()

graf3.groupby(["Grande Área do Conhecimento"])["coesao"].describe()

In [ ]:
# @title
# Supondo que seu DataFrame se chama 'df'
# Agrupando por município e calculando a média do indicador
df_agrupado = df_projetos_pipe
#df_agrupado = df_projetos_pipe.groupby('Município')['Shannon'].mean().reset_index()
#df_agrupado.loc[(df_agrupado["Grande Área do Conhecimento"]=="Engenharias")]
#print(df_agrupado.head())

import plotly.express as px
import json
import urllib.request

# 1. Carregar o GeoJSON (Certifique-se de que este arquivo tem TODOS os municípios de SP)
url = "https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-35-mun.json" # Exemplo: SP (35)
with urllib.request.urlopen(url) as response:
    municipios_geojson = json.load(response)

# 2. Criar o mapa
fig = px.choropleth(
    df_agrupado,
    geojson=municipios_geojson,
    locations='Município',
    featureidkey="properties.description",
    color='Shannon',
    color_continuous_scale="Viridis",
    # O segredo está aqui:
    basemap_visible=False, # Remove o mapa mundial de fundo
)

# 3. Configurar a cor neutra para quem NÃO tem dados no DataFrame
fig.update_traces(
    marker_line_width=0.5,
    marker_line_color="white",
    # Esta linha define a cor dos polígonos do GeoJSON que não encontram correspondência no DF
    unselected=dict(marker=dict(opacity=1)),
)

fig.update_geos(
    fitbounds="locations",
    visible=False,
    showcountries=False,
    # Define a cor de fundo para áreas do GeoJSON sem dados
    landcolor="lightgrey",
    showland=True,
)

fig.show()

# 7. Modelagem e Clusterização

In [ ]:
# @title

# 1. Preparação dos Dados
# Criamos uma cópia contendo apenas as colunas de interesse e removemos linhas vazias
df_kmeans = graf3[['coesao', 'Shannon']].dropna().copy()

# 2. Treinando o Modelo K-Means
# Definimos 4 clusters (n_clusters=4) e um random_state para que o resultado seja sempre o mesmo
modelo_kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

# O modelo analisa os dados e já cria uma nova coluna com o número do grupo (0, 1, 2 ou 3)
df_kmeans['Cluster'] = modelo_kmeans.fit_predict(df_kmeans[['coesao', 'Shannon']])

# Transformamos os números dos clusters em texto para o gráfico tratar como categorias fixas
df_kmeans['Cluster'] = df_kmeans['Cluster'].astype(str)

# 3. Desenhando o Gráfico
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 8))

# Criamos o gráfico de dispersão colorindo pelos clusters encontrados
sns.scatterplot(
    data=df_kmeans,
    x="coesao",
    y="Shannon",
    hue="Cluster",
    palette="viridis",  # Uma paleta excelente para visualizar diferentes grupos
    s=60,               # Tamanho dos pontos
    alpha=0.8
)

# Opcional (mas muito recomendado): Desenhar o "Centroide" (o coração de cada cluster)
centroides = modelo_kmeans.cluster_centers_
plt.scatter(
    centroides[:, 0], centroides[:, 1],
    c='red', s=200, marker='X', label='Centroides'
)

plt.title("Agrupamento K-Means (4 Clusters): Coesão vs Shannon", fontsize=15)
plt.xlabel("Coesão")
plt.ylabel("Índice de Shannon")
plt.legend(title="Grupos")

# Se quiser aplicar aquela mesma escala logarítmica de antes, descomente as linhas abaixo:
# plt.xscale('log')
# plt.yscale('log')

plt.show()

In [ ]:
!pip install networkx
!pip install nxviz==0.7.4
!pip install matplotlib
!pip install pyplot
!pip install pandas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors
import networkx as nx
import nxviz as nv
from nxviz import annotate
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
spreadsheet = gc.open('SCXDATA')
page_n = spreadsheet.worksheet("REDE_1_N")
page_e = spreadsheet.worksheet("REDE_1_A")

import pandas as pd
nodes = pd.DataFrame(page_n.get_all_records())
edges = pd.DataFrame(page_e.get_all_records())

#nodes["ANO"] = pd.to_numeric(nodes["ANO"], downcast="float")
#edges["ANO"] = pd.to_numeric(edges["ANO"], downcast="float")

edges.head()

In [ ]:
#Filtrar o ano
#edges = edges.loc[(edges["ANO"]<=2012) & (edges["TIPO"]=="TT")]
#edges = edges.loc[(edges["TIPO"]=="TT")]
edges.head()

In [ ]:
nodes = nodes.iloc[:,[0,1,2]]
edges = edges.iloc[:,[2,5,6,7]]
edges.rename(columns={"AREA": "source", "PESQ_ID": "target", "TIPO": "label", "COR": "color"}, inplace=True)

In [ ]:
edges.head()

In [ ]:
nodes.head()

In [ ]:
edges.columns = ["source", "target", "lable", "color"]
nodes.columns = ["nos", "type", "color"]

In [ ]:
data = nodes.set_index('nos').to_dict().items()
G = nx.from_pandas_edgelist(edges, edge_attr=True)
G.add_nodes_from(data)
#G = nx.relabel_nodes(G, {i: "#" + str(i) for i in range(len(G))})
#G.nodes
print(G)
print('Number of nodes', len(G.nodes))
print('Number of edges', len(G.edges))
print('Average degree', sum(dict(G.degree).values()) / len(G.nodes))

In [ ]:
remove = [node for node,degree in dict(G.degree()).items() if degree < 2]
G.remove_nodes_from(remove)
print(G)
print('Number of nodes', len(G.nodes))
print('Number of edges', len(G.edges))
print('Average degree', sum(dict(G.degree).values()) / len(G.nodes))

In [ ]:
plt.figure(figsize=(60,60))
plt.axis("off")

np.random.seed(0)
position = nx.spring_layout(G)

d = dict(G.degree)
arestas = G.edges()
colors = [G[u][v]['color'] for u,v in arestas]

#[v * 100 for v in d.values()]

nx.draw_networkx(G, pos=position, node_color="gray", edge_color=colors, font_size=15, node_size=20, with_labels=False);

In [ ]:
#nx.density(G)
#cen = nx.degree_centrality(G)
#centralidade = pd.DataFrame(cen)
#centralidade.head()

MD = pd.DataFrame(dict(
    DEGREE = dict(G.degree),
    DENSITY = nx.density(G),
    DEGREE_CENTRALITY = nx.degree_centrality(G),
    BETWEENNESS_CENTRALITY = nx.betweenness_centrality(G),
    CLOSENESS_CENTRALITY = nx.closeness_centrality(G),
    EIGENVECTOR = nx.eigenvector_centrality(G, max_iter=600),
    CLUSTCOEF = nx.clustering(G)
))

In [ ]:
#MD.head()
MD.to_excel('MD2.xlsx')

In [ ]:
nx.degree_centrality(G)
#print(dict(G.degree))

In [ ]:
deg_cen = nx.degree_centrality(G)
cen_btw = nx.betweenness_centrality(G)
cen_ei = nx.eigenvector_centrality(G)
cen_clos = nx.closeness_centrality(G)
cc = nx.clustering(G)

In [ ]:
for v in G.nodes():
    node = G.nodes[v]
    node['degree'] = deg_cen[v]
    node['betweeness'] = cen_btw[v]
    node['eigenvector'] = cen_ei[v]
    node['closeness'] = cen_clos[v]
    node['clustering'] = float(cc[v])

    print(v, node['degree'])

#G.nodes[3]['betweeness']

In [ ]:
indexes = ['degree', 'betweeness', 'eigenvector', 'closeness', 'clustering']

for i in indexes:
    nv.circos(G, node_color_by=i, sort_by=i)
    annotate.circos_labels(G, layout="rotate")
    plt.title(i.capitalize())
    cmap = cm.get_cmap("Spectral")
    norm = colors.Normalize(0, 1)
    plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap))
    #plt.suptitle(i.capitalize())
    #plt.savefig(f"{i}.png")
    plt.show()

# 8. Exportação dos Resultados